# Musanga V2 — Notebook 04 | Illumination-robust U-Net pilot

**Pilot then full four-site LOSO within this SAME notebook:** Train the original ResNet-18 U-Net on three forests and evaluate on the fourth, comparing:

- **M0:** native RGB + historical geometric/mild brightness augmentation
- **M1:** native RGB + realistic stronger illumination/gamma/contrast/smooth-shadow augmentation
- **M2:** M1 + per-site RGB gain referenced to **three training forests**

Sections 00–08 run the small pilot; Sections 09–11 run full four-site LOSO with all eligible crowns, M0/M1 and resumable fold checkpoints.

Default pilot LOSO holdout: **Baego**. This is a fast **single-fold pilot**, not full four-fold LOSO. Individual crown-centred native-resolution patches are used for cheap iteration, and sample sizes are capped. Only annotated crown pixels contribute to loss. Heldout test crowns never select epochs or hyperparameters. Checkpoints are resumable and per-batch/per-epoch progress is printed.

**Important M2 caveat:** Training reference median is fit on the three training forests, but M2 also reads the heldout site's **unlabelled** RGB median to compute its gain; therefore it represents *unsupervised test-image adaptation*, not strict fixed preprocessing. The distinction must be retained in later LOSO reporting.

**Luki:** completely excluded from Notebook 04 fitting and test selection. Its current merged 364-crown reference and forthcoming dark-area additions belong to versioned external evaluation later. No need to touch earlier source imagery or rerun earlier notebook stages to train this pilot.

**Artifacts:** exclusively `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training/`, with a fingerprinted pilot run directory. No full corrected mosaics or legacy checkpoints are overwritten.

**Execution:** Select a Colab GPU before Section 06. The first run generates and compresses a patch cache once, reusable across all three variants. For an entirely fresh GPU runtime, **run Sections 12–14 only** to train the final four-site M1 model and evaluate Luki. Sections 00–11 are the historical pilot/LOSO experiments and do not need to be rerun. The final model is trained independently from LOSO and is checked against the expanded frozen Luki reference.

## 00 — Setup, pilot budget and source locks

Configure a small **single held-out-forest pilot** first. Default held-out Baego; the other three forests provide spatial training/validation. All outputs are versioned under `KVO_work/V2/04_illumination_robust_training/`. Luki is not loaded or used for fitting.

In [ ]:
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=True
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("PILOT heldout:",HELDOUT,"epochs:",MAX_EPOCHS,"output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)


## 01 — Load immutable V2 crown splits

Reconstruct exactly the crowns from V2 Notebook 01 and join the saved spatial split IDs. Invalid geometries stay excluded. This stage does not merge, edit or read any Luki labels; forthcoming dark-area annotations will be handled by a separately versioned external-evaluation update.

In [ ]:
ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("LOSO TEST:",HELDOUT,"uses all annotation classes but only a bounded pilot subsample.")


## 02 — Honest training-only global normalization reference

For M2, calculate the shared reference using only the **three training forests**. Each site's own image brightness index is estimated without species labels. Applying an image-specific gain to the unseen held-out forest is **unsupervised target-image adaptation**, and must be reported as such rather than called strict fixed-preprocessing transfer. M0 and M1 are pure native-RGB transfer.

In [ ]:
stats=pd.read_csv(HARM/"03_site_luminance_and_fixed_gains.csv")
assert set(SITES).issubset(set(stats.site))
training_sites=[s for s in SITES if s!=HELDOUT]
LUMINANCE_MEDIAN={row.site:float(row.site_luminance_median) for _,row in stats.iterrows()}
reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
GLOBAL_GAIN_BOUNDS=(.85,1.15)
GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),*GLOBAL_GAIN_BOUNDS)) for s in SITES}
print("Per-fold reference from training sites ONLY:",training_sites,round(reference,4))
print("Per-site gains (test site's own unlabelled RGB statistics imply unsupervised adaptation):",GAIN)
assert HELDOUT not in training_sites


## 03 — Deterministic pilot samples and native-resolution patch cache

Build one shared patch cache per fold from identical source crowns for **all model variants**, with 256 px imagery at each forest's native GSD. Labels are rasterized only from the same training/validation split (all other pixels ignored). Sampling is class-balanced. No full-mosaic masks are constructed, and caches are fingerprinted and reusable.

In [ ]:
def choose_crowns(g,split,n,seed):
    selected=g[g.split==split].copy()
    if n is None or len(selected)<=n:return selected
    rng=np.random.default_rng(seed)
    # Sample both classes; avoid extreme imbalance and keep full indices.
    half=n//2
    p=selected[selected.label==1].index.to_numpy()
    q=selected[selected.label==0].index.to_numpy()
    rng.shuffle(p);rng.shuffle(q)
    p=p[:min(len(p),half)]
    q=q[:min(len(q),n-len(p))]
    if len(p)+len(q)<n:
        remaining=selected.index.difference(np.concatenate([p,q])).to_numpy()
        rng.shuffle(remaining)
        ids=np.concatenate([p,q,remaining[:n-len(p)-len(q)]])
    else:ids=np.concatenate([p,q])
    return selected.loc[ids].copy()

pilot_indices={}
for j,site in enumerate(training_sites):
    g=ANN[site]
    pilot_indices[(site,"training")]=choose_crowns(g,"training",PILOT_TRAIN_CROWNS_PER_SITE,SEED+10*j)
    pilot_indices[(site,"validation")]=choose_crowns(g,"validation",PILOT_VAL_CROWNS_PER_SITE,SEED+100+10*j)
TEST_CROWNS=choose_crowns(ANN[HELDOUT].assign(split="external"),"external",
                          PILOT_TEST_CROWNS,SEED+555)
assert TEST_CROWNS.label.nunique()==2
print("Pilot train",sum(len(pilot_indices[s,"training"]) for s in training_sites),
      "val",sum(len(pilot_indices[s,"validation"]) for s in training_sites),
      "heldout test",len(TEST_CROWNS))
configuration=dict(heldout=HELDOUT,seed=SEED,patch=PATCH,train_cap=PILOT_TRAIN_CROWNS_PER_SITE,
    val_cap=PILOT_VAL_CROWNS_PER_SITE,test_cap=PILOT_TEST_CROWNS,
    source=SOURCE_FINGERPRINT,ref=sha_file(HARM/"03_site_luminance_and_fixed_gains.csv"),
    experiment="v2_pilot_centered_crown_patch_v1")
RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
RUN_OUT=OUT/f"{HELDOUT}_{RUN_ID}";RUN_OUT.mkdir(exist_ok=True)
NPZ=CACHE/f"pilot_{HELDOUT}_{RUN_ID}.npz"
def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI
if NPZ.exists() and not FORCE_REBUILD_CACHE:
    obj=np.load(NPZ,allow_pickle=False)
    X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
    print("Reused fingerprinted pilot cache",NPZ)
else:
    X,Y,S,SI,CI=generate_cache03()
assert len(X)==len(Y)==len(S)==len(SI)==len(CI)
assert (S==0).any() and (S==1).any()
print("Patch cache:",len(X),"train:",int((S==0).sum()),"val:",int((S==1).sum()))
pd.DataFrame(dict(site=SI,split=np.where(S==0,"training","validation"),
                  crown_id=CI)).to_csv(RUN_OUT/"04_pilot_patch_registry.csv",index=False)


## 04 — Model, preprocessing and photometric augmentation

Keep the original ResNet-18 U-Net and ImageNet RGB normalization. Compare **M0** historical modest brightness jitter, **M1** randomized brightness/gamma/contrast/channel shifts and smooth shadows, **M2** exactly M1 plus a constant training-site gain. All share architecture, seed, sampled training crowns and optimization settings. Validation never uses random augmentation.

In [ ]:
MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("Each variant has identical architecture, patch inventory and RNG seed.")
print("M2 fold reference learned only from:",training_sites)


## 05 — Fast smoke test (before GPU training)

Validate that each treatment creates finite tensors and that masked targets contain both species classes. Detect configuration errors **before** allocating long GPU runs.

In [ ]:
assert len(set(SI))==len(training_sites)
for variant in RUN_VARIANTS:
    tr=PilotDataset04(0,variant,epoch=1)
    va=PilotDataset04(1,variant,epoch=0)
    a,b=tr[0];c,d=va[0]
    assert a.shape==(3,PATCH,PATCH) and c.shape==(3,PATCH,PATCH)
    assert torch.isfinite(a).all() and torch.isfinite(c).all()
    assert set(torch.unique(b).tolist()).issubset({0,1,IGNORE})
    print(variant,"train batches",math.ceil(len(tr)/BATCH_SIZE),
          "val batches",math.ceil(len(va)/BATCH_SIZE),
          "x range",round(float(a.min()),2),round(float(a.max()),2))
print("Smoke test passed.")


## 06 — Resumable pilot training with per-epoch progress

Run M0/M1/M2 sequentially with an identical pilot budget. Checkpoints are saved **every epoch**, including optimizer, scheduler-free training state, AMP scaler, early-stopping count and best validation AP; reruns resume instead of starting over. Console output is flushed each epoch. No held-out-forest labels are used for checkpoint selection.

In [ ]:
def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on three training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary
TRAIN_RESULTS04=[]
if RUN_TRAINING:
    assert DEVICE=="cuda","Change runtime to GPU; refusing long CPU training."
    for variant in RUN_VARIANTS:
        print("\nSTART VARIANT:",variant,flush=True)
        TRAIN_RESULTS04.append(train_variant04(variant))
    display(pd.DataFrame(TRAIN_RESULTS04))
else:print("RUN_TRAINING=False. Smoke test only.")


## 07 — Small independent Baego test; no selection on test

Evaluate each trained variant on an identical, class-balanced sample from the **held-out fourth forest**. Each crown is scored using mean predicted probability within its original labelled polygon on a native-resolution patch. Results are diagnostic (pilot sample), **not full wall-to-wall canopy accuracy or complete LOSO**. Do not choose hyperparameters based on Luki.

In [ ]:
def test_crowns04(model,variant,test_g):
    model.eval();rows=[]
    with rasterio.open(GEO/SITES[HELDOUT][0]) as src:
        for _,rec in tqdm(test_g.iterrows(),total=len(test_g),desc=f"{variant} heldout crowns"):
            w=centered_window(rec.geometry,PATCH,src)
            raw=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1)
            valid=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            interior=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                                    transform=src.window_transform(w),invert=True)&valid
            if interior.sum()<12:continue
            img=raw.astype(np.float32)/255.
            if variant=="M2_sitegain_photometric":
                img=np.clip(img*GAIN[HELDOUT],0,1)
            normalized=np.moveaxis((img-MEAN)/STD,-1,0).astype(np.float32).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(normalized[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            score=float(np.mean(p[interior]))
            rows.append(dict(site=HELDOUT,crown_id=str(rec.crown_id),label=int(rec.label),
                predicted_probability=score,valid_pixels=int(interior.sum()),
                variant=variant,experiment="balanced_heldout_crown_pilot"))
    return pd.DataFrame(rows)
TEST_PRED04=[]
if RUN_TRAINING:
    for variant in RUN_VARIANTS:
        folder=RUN_OUT/variant
        bestfile=folder/"best_model.pt"
        assert bestfile.exists(),f"Missing {bestfile}"
        seed_all(SEED);model=new_model04()
        ckpt=torch.load(bestfile,map_location=DEVICE,weights_only=False)
        assert ckpt["run_id"]==RUN_ID and ckpt["variant"]==variant
        model.load_state_dict(ckpt["model"])
        df=test_crowns04(model,variant,TEST_CROWNS)
        assert df.label.nunique()==2
        df.to_csv(folder/"heldout_pilot_crown_predictions.csv",index=False)
        TEST_PRED04.append(df)
        print("EVALUATED",variant,"crowns",len(df),flush=True)
        del model;gc.collect()
        if DEVICE=="cuda":torch.cuda.empty_cache()
    HOLDOUT04=pd.concat(TEST_PRED04,ignore_index=True)
    HOLDOUT04.to_csv(RUN_OUT/"04_heldout_pilot_crown_predictions.csv",index=False)


## 08 — Compare pilot results and save handoff

Compare validation-selected checkpoints on a held-out forest, alongside run-time budgets. Keep results labelled as **pilot**, and make no claims about Luki or wall-to-wall canopy cover. Subsequent complete four-fold LOSO remains the authority; revised Luki dark-area annotations must have explicit reference versions.

In [ ]:
def summarize04(d):
    y=d.label.to_numpy(dtype=int)
    p=d.predicted_probability.to_numpy(float)
    q=(p>=.5).astype(int)
    return dict(n=len(y),musanga=int(y.sum()),other=int((1-y).sum()),
        auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision=float(precision_score(y,q,zero_division=0)),
        recall=float(recall_score(y,q,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,q)),
        f1=float(f1_score(y,q,zero_division=0)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
if RUN_TRAINING:
    SUMMARY04=pd.DataFrame([dict(variant=v,**summarize04(HOLDOUT04[HOLDOUT04.variant==v]))
                           for v in RUN_VARIANTS])
    SUMMARY04.to_csv(RUN_OUT/"04_heldout_pilot_comparison.csv",index=False)
    display(SUMMARY04.round(3))
manifest=dict(stage="Musanga V2 Notebook 04: pilot U-Net",
    heldout=HELDOUT,training_sites=training_sites,
    architecture="segmentation_models_pytorch.Unet resnet18 imagenet",
    spatial_splits_source=str(REF),source_files=SOURCE_FINGERPRINT,
    run_id=RUN_ID,run_dir=str(RUN_OUT),
    patch_size=PATCH,seed=SEED,
    training_caps_per_site=PILOT_TRAIN_CROWNS_PER_SITE,
    validation_caps_per_site=PILOT_VAL_CROWNS_PER_SITE,
    heldout_crown_pilot_cap=PILOT_TEST_CROWNS,
    training_epochs=MAX_EPOCHS,patience=PATIENCE,variants=RUN_VARIANTS,
    normalization_reference_sites=training_sites,
    normalization_reference_luminance=reference,site_gains=GAIN,
    m2_interpretation="unlabeled held-out image statistics used for site gain: unsupervised target adaptation",
    heldout_selection_policy="test labels never choose checkpoint or photometric settings",
    luki_policy="Luki excluded from model fitting, checkpoint selection and pilot testing; new dark annotations are future versioned external references",
    spatial_qa="single-crown labelled patches, all other pixels ignored, split buffers inherited from Notebook 01",
    output_policy="All generated files exclusively under Musanga/KVO_work/V2")
(RUN_OUT/"04_pilot_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Pilot run:",RUN_ID)
print("Outputs:",RUN_OUT)
print("NEXT: inspect pilot results and decide whether full four-fold LOSO is warranted.")


## 09 — Full four-site LOSO: configuration and safe state isolation

This is **the full experiment inside Notebook 04**, not a new notebook. It uses all eligible Notebook 01 **training** and **validation** crowns in each of the four LOSO folds, and tests **all eligible held-out crowns**, with no artificial class balancing. Two variants: M0 original modest augmentation, M1 stronger photometric augmentation. Pixel-level validation AP selects checkpoints; independent held-out **crown-level ROC-AUC and AP** are primary ranking metrics. Additional F1 at 0.5 is descriptive only.

**Reproducibility and efficiency:** reuse Section 03/04 cache and U-Net functions; write into a separate `full_LOSO/` folder; freeze pilot globals after this stage; checkpoints every epoch and short streamed training progress. Default maximum 20 epochs and early stopping after four non-improving epochs. Training-crown **centre crops** approximate sampled crown recognition, not wall-to-wall U-Net pixel evaluation. Luki remains untouched.

In [ ]:
# Run the existing pilot once, then run Sections 09–11 to execute full LOSO.
RUN_FULL_LOSO=True
FULL_FOLDS=["Baego","MOS_1_2","Yoko","Yangambi"]
FULL_VARIANTS=["M0_native_historical","M1_native_photometric"]
FULL_MAX_EPOCHS=20
FULL_PATIENCE=4
FULL_BATCH_SIZE=12
FULL_ROOT=OUT/"full_LOSO"
FULL_ROOT.mkdir(parents=True,exist_ok=True)
assert len(FULL_FOLDS)==4 and set(FULL_FOLDS)==set(SITES)
assert set(FULL_VARIANTS).issubset(set(RUN_VARIANTS))
FULL_SUMMARY=FULL_ROOT/"04_full_LOSO_fold_metrics.csv"
# Sections 03–08 deliberately used shared notebook globals for simple pilot reuse.
# Snapshot these before substituting a complete fold and restore in Section 11.
PILOT_GLOBALS04={k:globals()[k] for k in [
    "HELDOUT","training_sites","reference","GAIN","RUN_ID","RUN_OUT",
    "NPZ","pilot_indices","TEST_CROWNS","X","Y","S","SI","CI",
    "RUN_VARIANTS","MAX_EPOCHS","PATIENCE","BATCH_SIZE","FORCE_RESTART"]}
print("Full LOSO folds:",FULL_FOLDS, "| models:",FULL_VARIANTS,flush=True)
print("Max epochs:",FULL_MAX_EPOCHS,"early-stop patience:",FULL_PATIENCE,flush=True)
print("Outputs:",FULL_ROOT,flush=True)

## 10 — Execute full LOSO with independent cache and checkpoint per fold

**Run once.** Each fold computes the training-only normalization reference, builds or reuses its own fingerprinted full-crown patch cache, trains or resumes M0 and M1, and evaluates the entire held-out forest using the existing immutable crown annotations. Completion is streamed after every fold. The pilot is never overwritten.

**Interpretation:** M0/M1 use native RGB for held-out testing (no target-domain normalization); the fold reference is recorded for reproducibility but unused by M0/M1. No SAM and no Luki in this workflow. The held-out labels are accessed **only after training** for evaluation, not to select the best epoch.

In [ ]:
def fold_cache04():
    # All eligible crowns, including natural species prevalences; split buffers excluded.
    global X,Y,S,SI,CI,NPZ
    if NPZ.exists() and not FORCE_REBUILD_CACHE:
        with np.load(NPZ,allow_pickle=False) as obj:
            X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
        print("CACHED full-crown patches",len(X),NPZ.name,flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(training_sites)
    assert (S==0).any() and (S==1).any()
    return len(X)

def evaluate_fold04(variant):
    ck=RUN_OUT/variant/"best_model.pt"
    assert ck.exists(),f"Missing selected checkpoint: {ck}"
    seed_all(SEED)
    model=new_model04()
    state=torch.load(ck,map_location=DEVICE,weights_only=False)
    assert state["run_id"]==RUN_ID and state["variant"]==variant
    model.load_state_dict(state["model"])
    pred=test_crowns04(model,variant,TEST_CROWNS)
    del model
    if DEVICE=="cuda":torch.cuda.empty_cache()
    gc.collect()
    assert len(pred)>0 and pred.label.nunique()==2
    assert pred.crown_id.is_unique
    assert set(pred.crown_id).issubset(set(TEST_CROWNS.crown_id))
    pred.to_csv(RUN_OUT/variant/"full_heldout_crown_predictions.csv",index=False)
    return dict(fold=HELDOUT,variant=variant,
                reference_crowns=int(len(TEST_CROWNS)),
                evaluated_crowns=int(len(pred)),
                **summarize04(pred))

def run_full_fold04(holdout):
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE
    assert holdout in SITES
    HELDOUT=holdout
    training_sites=[s for s in SITES if s!=HELDOUT]
    assert len(training_sites)==3
    # This reference uses ONLY the three development forests of this fold.
    reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
    GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),
                           *GLOBAL_GAIN_BOUNDS)) for s in SITES}
    pilot_indices={(s,sp):ANN[s][ANN[s].split==sp].copy()
                   for s in training_sites for sp in ("training","validation")}
    TEST_CROWNS=ANN[HELDOUT].copy()  # ALL annotated crowns; original prevalence
    RUN_VARIANTS=FULL_VARIANTS.copy()
    MAX_EPOCHS=FULL_MAX_EPOCHS
    PATIENCE=FULL_PATIENCE
    BATCH_SIZE=FULL_BATCH_SIZE
    configuration=dict(stage="V2_04_full_loso_crown_patches_v1",
      fold=HELDOUT,train_sites=training_sites,
      train_ids={s:sorted(pilot_indices[s,"training"].crown_id.astype(str).tolist())
                 for s in training_sites},
      validation_ids={s:sorted(pilot_indices[s,"validation"].crown_id.astype(str).tolist())
                      for s in training_sites},
      patch=PATCH,seed=SEED,
      source={s:SOURCE_FINGERPRINT[s] for s in training_sites},
      photo_aug_version=1)
    RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
    RUN_OUT=FULL_ROOT/f"{HELDOUT}_{RUN_ID}"
    RUN_OUT.mkdir(exist_ok=True)
    NPZ=CACHE/f"full_{HELDOUT}_{RUN_ID}.npz"
    # A complete fold is resumable independently, including if Colab disconnects.
    summary_path=RUN_OUT/"04_fold_results.csv"
    if summary_path.exists():
        existing=pd.read_csv(summary_path)
        if set(existing.variant)==set(FULL_VARIANTS) and len(existing)==len(FULL_VARIANTS):
            print("REUSING COMPLETE FULL LOSO FOLD",HELDOUT,flush=True)
            return existing
    n=fold_cache04()
    print(f"\nFULL FOLD {HELDOUT}: train={int((S==0).sum())}, "
          f"val={int((S==1).sum())}, test={len(TEST_CROWNS)}; cached {n}",flush=True)
    results=[]
    for variant in FULL_VARIANTS:
        print("TRAIN:",HELDOUT,variant,flush=True)
        selected=train_variant04(variant)
        print("TEST:",HELDOUT,variant,flush=True)
        scores=evaluate_fold04(variant)
        scores["best_training_validation_ap"]=selected["best_validation_ap"]
        scores["run_id"]=RUN_ID
        results.append(scores)
        # Write cumulative results after every model, even if later folds fail.
        pd.DataFrame(results).to_csv(summary_path,index=False)
        print("DONE",HELDOUT,variant,
              "crown AUC",round(scores["auc"],4),
              "AP",round(scores["average_precision"],4),
              "F1@0.5",round(scores["f1"],4),flush=True)
    return pd.DataFrame(results)

ALL_FULL04=[]
if RUN_FULL_LOSO:
    assert DEVICE=="cuda","Choose a GPU runtime for full LOSO."
    for fold in FULL_FOLDS:
        print("\n"+"="*55+"\nFULL LOSO HELD-OUT:",fold,flush=True)
        df=run_full_fold04(fold)
        ALL_FULL04.append(df)
        pd.concat(ALL_FULL04,ignore_index=True).to_csv(FULL_SUMMARY,index=False)
    print("\nAll four folds finished. Saved:",FULL_SUMMARY,flush=True)
else:
    print("Full LOSO disabled. Set RUN_FULL_LOSO=True and rerun section 10.")

## 11 — Aggregate ROC-AUC / AP, calibration-sensitive metrics and versioned completion manifest

Report crown-level ROC-AUC and AP as **primary**, alongside threshold-dependent precision, recall, F1 and balanced accuracy. Macro averaging across forests avoids treating the largest forest as automatically most important. Flag missing predictions; preserve per-crown outputs for later absolute RGB brightness joins. **No claim of external Luki accuracy here.**

In [ ]:
if FULL_SUMMARY.exists():
    result=pd.read_csv(FULL_SUMMARY)
    assert set(result.fold)==set(FULL_FOLDS),(
        "Full LOSO incomplete; existing folds:",sorted(result.fold.unique()))
    assert len(result)==len(FULL_FOLDS)*len(FULL_VARIANTS)
    order=["fold","variant","reference_crowns","evaluated_crowns",
           "auc","average_precision","precision","recall","balanced_accuracy","f1",
           "false_positives","false_negatives","best_training_validation_ap"]
    display(result[order].round(4))
    macro=result.groupby("variant",as_index=False).agg(
        folds=("fold","nunique"),
        macro_crown_auc=("auc","mean"),
        macro_crown_ap=("average_precision","mean"),
        macro_f1_at_0p5=("f1","mean"),
        macro_balanced_accuracy_at_0p5=("balanced_accuracy","mean"),
        total_crowns=("evaluated_crowns","sum"),
        missing_crowns=("reference_crowns",lambda x:0)) # recomputed below
    missing=result.groupby("variant").apply(
        lambda d:int((d.reference_crowns-d.evaluated_crowns).sum()),
        include_groups=False)
    macro["missing_crowns"]=macro.variant.map(missing)
    macro.to_csv(FULL_ROOT/"04_full_LOSO_macro_summary.csv",index=False)
    display(macro.round(4))
    contract=dict(stage="Musanga V2/04 complete LOSO using manual crowns",
      folds=FULL_FOLDS,variants=FULL_VARIANTS,heldout_Luki=False,
      patch_size_px=PATCH,
      trained_on="All eligible train split crowns in three forests per fold",
      validated_on="All eligible validation split crowns in three forests per fold",
      test_on="All eligible reference crowns from the fourth forest",
      reference_split_source=str(REF),
      primary_metrics=["heldout_crown_ROC_AUC","heldout_crown_average_precision"],
      secondary_metrics=["F1_at_probability_0.5","balanced_accuracy","precision","recall"],
      image_input="native UAV RGB only; no SAM, no spatial resampling",
      threshold_policy="fixed 0.5 is diagnostic, not chosen on held-out test",
      production_warning="Crown-centred patch pilot/training is NOT full orthomosaic pixel-level test",
      luki="held aside for future, versioned new dark-region annotations",
      full_metrics=str(FULL_SUMMARY),
      macro_metrics=str(FULL_ROOT/"04_full_LOSO_macro_summary.csv"))
    (FULL_ROOT/"04_full_LOSO_manifest.json").write_text(json.dumps(contract,indent=2))
    print("Full LOSO finished; all pilot outputs remain intact.",flush=True)
else:
    print("Section 10 has not yet produced full LOSO metrics.")
# Restore pilot state, so revisiting earlier sections does not cross-contaminate.
if "PILOT_GLOBALS04" in globals():
    globals().update(PILOT_GLOBALS04)
    print("Restored pilot configuration and arrays.")

## 12 — Fresh-runtime setup for the final four-site M1 model

**Start here in a NEW Colab GPU runtime.** This section independently mounts Drive, loads the immutable four-site Notebook 01 references and split registers, defines the original U-Net training and patch functions, and locks the versioned Luki reference. It does **not** execute the pilot or four LOSO folds. After this cell, run Sections **13 and 14 only**.

The final training cache is separately fingerprinted and resumable; completed pilot and LOSO checkpoints remain untouched. No GPU is needed until Section 13, but select a GPU runtime before starting.

In [3]:
# FRESH-RUNTIME BOOTSTRAP: only the essential earlier DEFINITIONS are used.
# No pilot inference or four-fold LOSO is launched by this section.
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=False
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("PILOT heldout:",HELDOUT,"epochs:",MAX_EPOCHS,"output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)

ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("LOSO TEST:",HELDOUT,"uses all annotation classes but only a bounded pilot subsample.")

def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI

MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("Each variant has identical architecture, patch inventory and RNG seed.")


def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on three training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary

FULL_BATCH_SIZE=12

# --------------------- FINAL-MODEL CONFIGURATION ---------------------
FINAL04=OUT/"final_four_site_M1";FINAL04.mkdir(parents=True,exist_ok=True)
FINAL_VARIANT04="M1_native_photometric"
FINAL_MAX_EPOCHS04=20
FINAL_PATIENCE04=5
RUN_FINAL_M1_04=True
RUN_LUKI_04=True
LUKI_RGB04=GEO/"Luki.tif"
LUKI_META04=REF/"01_luki_current_reference_version.json"
assert LUKI_META04.exists(),"First run updated Notebook 01 with the new Luki annotations"
LUKI_MANIFEST04=json.loads(LUKI_META04.read_text())
LUKI_VERSION04=LUKI_MANIFEST04["reference_id"]
LUKI_REFERENCE04=Path(LUKI_MANIFEST04["frozen_copy"])
assert LUKI_REFERENCE04.exists(),"Missing frozen external Luki version"
LUKI_CROWNS04=gpd.read_file(LUKI_REFERENCE04,layer="external_evaluation")
assert len(LUKI_CROWNS04)==int(LUKI_MANIFEST04["crowns"])
assert int(LUKI_CROWNS04.label.sum())==int(LUKI_MANIFEST04["musanga"])
assert LUKI_CROWNS04.crown_id.is_unique
assert LUKI_CROWNS04.crs is not None
with rasterio.open(LUKI_RGB04) as src:
    assert LUKI_CROWNS04.crs==src.crs
print("Final M1 source forests",list(SITES))
print("Luki independent source",LUKI_VERSION04,
      len(LUKI_CROWNS04),"crowns (reserved until model fully frozen)")
print("Final 4-site M1 directory:",FINAL04)


GPU: NVIDIA L4
PILOT heldout: Baego epochs: 4 output: /content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training
MOS_1_2 : 1132 crowns; train 863 val 240 buffered 29
Yoko : 1792 crowns; train 1215 val 417 buffered 160
Yangambi : 944 crowns; train 735 val 183 buffered 26
Baego : 749 crowns; train 534 val 198 buffered 17
LOSO TEST: Baego uses all annotation classes but only a bounded pilot subsample.
Each variant has identical architecture, patch inventory and RNG seed.
Final M1 source forests ['MOS_1_2', 'Yoko', 'Yangambi', 'Baego']
Luki independent source d957b570055bf5dd 479 crowns (reserved until model fully frozen)
Final 4-site M1 directory: /content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training/final_four_site_M1


## 13 — Final M1 training on all four original forests (resumable)

Run directly after Section 12 in a fresh GPU runtime. Read or build one four-forest native-RGB crown-centred cache using the frozen Notebook 01 training/validation splits. Train **M1 only** with the previously validated photometric recipe; save epoch checkpoints, select the model using development-forest validation pixel AP, and never consult Luki labels. No preceding pilot or LOSO cells are required.

In [4]:
FINAL_CONFIG04=dict(stage="four_site_M1_v1",seed=SEED,patch=PATCH,
  training_sites=list(SITES),train_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="training","crown_id"].astype(str).tolist())
      for site in SITES},
  val_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="validation","crown_id"].astype(str).tolist())
      for site in SITES},
  source_fingerprints=SOURCE_FINGERPRINT,
  photo_augmentation="M1_native_photometric_same_as_LOSO",
  max_epochs=FINAL_MAX_EPOCHS04,patience=FINAL_PATIENCE04)
FINAL_RUN_ID04=hashlib.sha256(json.dumps(FINAL_CONFIG04,sort_keys=True).encode()).hexdigest()[:12]
FINAL_RUN04=FINAL04/FINAL_RUN_ID04
FINAL_RUN04.mkdir(parents=True,exist_ok=True)
FINAL_CACHE04=CACHE/f"final_four_site_M1_{FINAL_RUN_ID04}.npz"

def final_prepare04():
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,X,Y,S,SI,CI
    global RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE,FORCE_RESTART
    # No fifth held-out development site: change training_sites to ALL four.
    HELDOUT="FINAL_FOUR_SITE_M1"
    training_sites=list(SITES)
    reference=np.nan # M1 uses native RGB: no site reference or gain is required
    GAIN={s:1.0 for s in SITES} # M1 never uses site gains
    pilot_indices={(site,sp):ANN[site][ANN[site].split==sp].copy()
                   for site in training_sites for sp in ("training","validation")}
    TEST_CROWNS=gpd.GeoDataFrame()
    RUN_ID=FINAL_RUN_ID04
    RUN_OUT=FINAL_RUN04
    NPZ=FINAL_CACHE04
    RUN_VARIANTS=[FINAL_VARIANT04]
    MAX_EPOCHS=FINAL_MAX_EPOCHS04
    PATIENCE=FINAL_PATIENCE04
    BATCH_SIZE=FULL_BATCH_SIZE
    FORCE_RESTART=False
    if NPZ.exists():
        with np.load(NPZ,allow_pickle=False) as cache:
            X,Y,S,SI,CI=[cache[k] for k in ["X","Y","S","SI","CI"]]
        print("REUSED full four-site training cache:",len(X),flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(SITES)
    assert (S==0).any() and (S==1).any()
    print("Four-site data:",len(X),"patches",
          int((S==0).sum()),"training",int((S==1).sum()),"validation",flush=True)
    print("Run ID:",RUN_ID,flush=True)

FINAL_CKPT04=FINAL_RUN04/FINAL_VARIANT04/"best_model.pt"
if RUN_FINAL_M1_04:
    assert DEVICE=="cuda","Select a Colab GPU runtime."
    final_prepare04()
    FINAL_TRAIN04=train_variant04(FINAL_VARIANT04)
    assert FINAL_CKPT04.exists()
    (FINAL_RUN04/"04_final_model_contract.json").write_text(json.dumps(dict(
        run_id=FINAL_RUN_ID04,architecture="smp.Unet(resnet18,ImageNet encoder)",
        model=FINAL_VARIANT04,source_forests=list(SITES),
        best_checkpoint=str(FINAL_CKPT04),
        spatial_training_and_validation="Notebook 01 250m blocks/30m buffer",
        class_labels="Class_ID=1 Musanga, other numeric Class_ID non-Musanga",
        best_validation_ap=FINAL_TRAIN04["best_validation_ap"],
        epochs_completed=FINAL_TRAIN04["epochs_completed"],
        sampling="256px native GSD crown-centred single annotated crown pixel loss",
        validation="spatial validation pixels across all four forests",
        luki="Not seen during training, validation, checkpoint selection or hyperparameter tuning",
        source_fingerprints=SOURCE_FINGERPRINT,
        limitation="Not wall-to-wall probability map; spatial seams and shadow patches require further QA"
    ),indent=2))
    print("FINAL FOUR-SITE M1 READY:",FINAL_CKPT04,flush=True)

else:
    print("Training disabled; expecting an existing final checkpoint:",FINAL_CKPT04)


MOS_1_2/training:   0%|          | 0/863 [00:00<?, ?it/s]

MOS_1_2/validation:   0%|          | 0/240 [00:00<?, ?it/s]

Yoko/training:   0%|          | 0/1215 [00:00<?, ?it/s]

Yoko/validation:   0%|          | 0/417 [00:00<?, ?it/s]

Yangambi/training:   0%|          | 0/735 [00:00<?, ?it/s]

...

Caching 4385 patches: 862.1 MB raw RGB
Four-site data: 4385 patches 3347 training 1038 validation
Run ID: 337073e7b74f


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

  M1_native_photometric epoch 1 batch 1/279 loss 0.6495
  M1_native_photometric epoch 1 batch 25/279 loss 0.5369
  M1_native_photometric epoch 1 batch 50/279 loss 0.4473
  M1_native_photometric epoch 1 batch 75/279 loss 0.4086
  M1_native_photometric epoch 1 batch 100/279 loss 0.3871
  M1_native_photometric epoch 1 batch 125/279 loss 0.3785
  M1_native_photometric epoch 1 batch 150/279 loss 0.3613
  M1_native_photometric epoch 1 batch 175/279 loss 0.3470
  M1_native_photometric epoch 1 batch 200/279 loss 0.3484
  M1_native_photometric epoch 1 batch 225/279 loss 0.3404
  M1_native_photometric epoch 1 batch 250/279 loss 0.3323
  M1_native_photometric epoch 1 batch 275/279 loss 0.3221
  M1_native_photometric epoch 1 batch 279/279 loss 0.3231
PROGRESS M1_native_photometric epoch 1/20: loss=0.3231, AP=0.9704, AUC=0.9738, best_AP=0.9704, 46.4s
  M1_native_photometric epoch 2 batch 1/279 loss 0.2852
  M1_native_photometric epoch 2 batch 25/279 loss 0.2582
  M1_native_photometric epoch 2 batch

## 14 — Independent external Luki crown ROC-AUC/AP, plus illumination diagnostics

Evaluate **exactly the frozen reference version** printed in Section 12. For each manually delineated crown read a native RGB window centred on the crown, predict with the frozen M1 model, and average probability over its rasterized geometry. Reject crowns whose polygon cannot fit inside the inference window or lack sufficient valid RGB support rather than silently biasing crown statistics. **ROC-AUC and AP remain primary**; F1 at 0.5 is secondary. Also join the actual Luki native-RGB luminance from Notebook 01, and summarize errors by **common 0–1 absolute brightness bins** and **Luki-relative quartiles**.

The crown diagnostic does **not** test broad unannotated shadow areas, ortho seams or model behaviour on full probability maps. Later we will generate those maps and test their suitability for satellite targets.

In [5]:
def predict_luki_crowns04(model,gdf):
    rows=[]
    model.eval()
    with rasterio.open(LUKI_RGB04) as src:
        for _,rec in tqdm(gdf.iterrows(),total=len(gdf),desc="Final M1 external Luki"):
            win=centered_window(rec.geometry,PATCH,src)
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0)
            raw=np.moveaxis(rgb,0,-1).astype(np.uint8)
            allowed=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                      ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            inside=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                transform=src.window_transform(win),invert=True)
            # Reject truncated crowns so the inference subset matches reference geometry.
            expected=max(1.,float(rec.geometry.area)/abs(src.transform.a*src.transform.e))
            coverage=float(inside.sum()/expected)
            mask=inside&allowed
            base=dict(site="Luki",crown_id=str(rec.crown_id),label=int(rec.label),
                valid_pixels=int(mask.sum()),estimated_geometry_coverage=coverage)
            if coverage<.90 or mask.sum()<10:
                rows.append(dict(**base,score=np.nan,qa="truncated_or_low_support"))
                continue
            img=raw.astype(np.float32)/255.
            arr=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(arr[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            rows.append(dict(**base,score=float(np.mean(p[mask])),qa="ok"))
    return pd.DataFrame(rows)

LUKI_EVAL_DIR04=FINAL_RUN04/"luki_external"/LUKI_VERSION04
LUKI_EVAL_DIR04.mkdir(parents=True,exist_ok=True)
RESULT_PATH04=LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions.csv"
if RUN_LUKI_04:
    assert FINAL_CKPT04.exists(),"Run Section 13 to train the 4-site final model first."
    if RESULT_PATH04.exists():
        preds=pd.read_csv(RESULT_PATH04,dtype={"crown_id":"string"})
        print("REUSED version-locked external predictions",RESULT_PATH04)
    else:
        model=new_model04()
        state=torch.load(FINAL_CKPT04,map_location=DEVICE,weights_only=False)
        assert state["run_id"]==FINAL_RUN_ID04
        model.load_state_dict(state["model"])
        preds=predict_luki_crowns04(model,LUKI_CROWNS04)
        preds.to_csv(RESULT_PATH04,index=False)
        del model
        if DEVICE=="cuda":torch.cuda.empty_cache()
    assert len(preds)==len(LUKI_CROWNS04)
    assert preds.crown_id.is_unique
    light=pd.read_csv(REF/"01_crown_illumination.csv",dtype={"crown_id":"string"})
    luki_light=light.loc[light.site=="Luki",["crown_id","median_luminance"]]
    assert luki_light.crown_id.is_unique
    preds=preds.merge(luki_light,on="crown_id",how="left",validate="one_to_one")
    assert preds.median_luminance.notna().all()
    preds["fixed_RGB_bin"]=pd.cut(preds.median_luminance,bins=np.linspace(0,1,11),include_lowest=True)
    quartiles=preds.median_luminance.quantile([.25,.75])
    preds["relative_light"]=np.where(preds.median_luminance<quartiles.loc[.25],
        "darkest_25pct",np.where(preds.median_luminance>quartiles.loc[.75],
        "brightest_25pct","middle_50pct"))
    evaluated=preds[preds.qa=="ok"].copy()
    print("Luki reference crowns",len(preds),"with valid centre patch",len(evaluated),
          "excluded due to truncation/coverage",len(preds)-len(evaluated),flush=True)
    if evaluated.label.nunique()!=2:raise RuntimeError("Both species classes required for AUC/AP")
    p=evaluated.score.to_numpy(float);y=evaluated.label.to_numpy(int)
    q=(p>=.5).astype(int)
    metrics=dict(model="final_four_site_M1",reference_version=LUKI_VERSION04,
        reference_crowns=len(preds),evaluated_crowns=len(evaluated),
        roc_auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision_0p5=float(precision_score(y,q,zero_division=0)),
        recall_0p5=float(recall_score(y,q,zero_division=0)),
        f1_0p5=float(f1_score(y,q,zero_division=0)),
        balanced_accuracy_0p5=float(balanced_accuracy_score(y,q)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
    pd.DataFrame([metrics]).to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_metrics.csv",index=False)
    display(pd.DataFrame([metrics]).round(4))
    details=[]
    for name,col in [("absolute_RGB_bin","fixed_RGB_bin"),("relative_brightness","relative_light")]:
        for value,gg in evaluated.groupby(col,observed=True):
            pos=gg[gg.label==1];neg=gg[gg.label==0]
            details.append(dict(stratum_type=name,stratum=str(value),n=len(gg),
                musanga=len(pos),other=len(neg),
                mean_probability=float(gg.score.mean()),
                musanga_recall=float((pos.score>=.5).mean()) if len(pos) else np.nan,
                other_false_positive_rate=float((neg.score>=.5).mean()) if len(neg) else np.nan,
                auc=float(roc_auc_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan,
                ap=float(average_precision_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan))
    strata=pd.DataFrame(details)
    strata.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_brightness_metrics.csv",index=False)
    display(strata.round(4))
    preds.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions_with_brightness.csv",index=False)
    print("NO wall-to-wall probability map has been generated here.",flush=True)
    print("NEXT: independently compare spatial shadow artefacts and satellite target transfer.",flush=True)
else:
    print("Luki evaluation disabled.")


Final M1 external Luki:   0%|          | 0/479 [00:00<?, ?it/s]

Luki reference crowns 479 with valid centre patch 406 excluded due to truncation/coverage 73


,model,reference_version,reference_crowns,evaluated_crowns,roc_auc,average_precision,precision_0p5,recall_0p5,f1_0p5,balanced_accuracy_0p5,false_positives,false_negatives
0,final_four_site_M1,d957b570055bf5dd,479,406,0.9845,0.9824,0.974,0.9414,0.9574,0.9527,6,14


,stratum_type,stratum,n,musanga,other,mean_probability,musanga_recall,other_false_positive_rate,auc,ap
0,absolute_RGB_bin,"(0.3, 0.4]",3,3,0,0.9415,1.0000,NaN,NaN,NaN
1,absolute_RGB_bin,"(0.4, 0.5]",47,44,3,0.8946,1.0000,0.3333,0.9924,0.9995
2,absolute_RGB_bin,"(0.5, 0.6]",173,138,35,0.7570,0.9638,0.0857,0.9681,0.9811
3,absolute_RGB_bin,"(0.6, 0.7]",137,46,91,0.2972,0.8478,0.0110,0.9859,0.9773
4,absolute_RGB_bin,"(0.7, 0.8]",46,8,38,0.1336,0.7500,0.0263,0.9342,0.8079
5,relative_brightness,brightest_25pct,101,19,82,0.1337,0.6316,0.0122,0.9499,0.8629
6,relative_brightness,darkest_25pct,109,98,11,0.8658,1.0000,0.2727,0.9972,0.9997
7,relative_brightness,middle_50pct,196,122,74,0.5858,0.9426,0.0270,0.9827,0.9745


NO wall-to-wall probability map has been generated here.
NEXT: independently compare spatial shadow artefacts and satellite target transfer.


## 15 — Luki: illumination–Musanga probability correlations, frozen U-Net versus M1

**Run after Section 14.** No training or new inference is needed. The comparison uses the **same Luki reference version and identical crown IDs**, joining the existing Notebook 02 frozen-U-Net predictions and Section 14 final M1 predictions. Spearman rank correlations between native RGB crown luminance and predicted Musanga probability are reported overall and **separately for Musanga and non-Musanga crowns**, with paired bootstrap confidence intervals.

The frozen model averaged predictions across entire polygons, whereas current M1 prediction is based on a 256-pixel centered patch; thus different spatial support remains a limitation even for matched crowns. The crucial later test is wall-to-wall probability maps, shadow patterns and satellite transfer; crown correlations alone cannot prove absence of illumination artefacts.

In [ ]:
# Section 15: matched-crown, species-conditional illumination sensitivity.
# Uses only stored outputs; no extra training or GPU required.
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
OLD15=V2/"02_frozen_baseline"/"by_luki_reference"/LUKI_VERSION04/"02_frozen_crown_predictions.csv"
NEW15=FINAL_RUN04/"luki_external"/LUKI_VERSION04/"04_final_M1_luki_crown_predictions_with_brightness.csv"
REPORT15=FINAL_RUN04/"luki_external"/LUKI_VERSION04/"illumination_comparison"
REPORT15.mkdir(parents=True,exist_ok=True)
assert OLD15.exists(),f"Notebook 02 frozen Luki predictions missing: {OLD15}"
assert NEW15.exists(),f"Section 14 final M1 Luki predictions missing: {NEW15}"
frozen15=pd.read_csv(OLD15,dtype={"crown_id":str})
frozen15=frozen15.loc[(frozen15.site=="Luki")&(frozen15.qa=="ok"),
                       ["crown_id","label","mean_p"]].rename(
                       columns={"label":"frozen_label","mean_p":"frozen_p"})
m115=pd.read_csv(NEW15,dtype={"crown_id":str})
m115=m115.loc[m115.qa=="ok",["crown_id","label","score","median_luminance"]].rename(
                      columns={"score":"M1_p"})
assert frozen15.crown_id.is_unique and m115.crown_id.is_unique
paired15=m115.merge(frozen15,on="crown_id",validate="one_to_one")
assert len(paired15)>0 and paired15.label.nunique()==2
assert (paired15.label==paired15.frozen_label).all(),"Label discrepancy between references"
assert paired15[["median_luminance","frozen_p","M1_p"]].notna().all().all()
print("Matching crown sample:",len(paired15),"of",len(LUKI_CROWNS04),
      "Luki crowns; Musanga",sum(paired15.label==1),"non-Musanga",sum(paired15.label==0))
rng15=np.random.default_rng(SEED+1500)
results15=[]
for name,gg in [("All (confounded)",paired15),
                ("Musanga only",paired15[paired15.label==1]),
                ("Non-Musanga only",paired15[paired15.label==0])]:
    lum=gg.median_luminance.to_numpy(float)
    original=gg.frozen_p.to_numpy(float)
    newer=gg.M1_p.to_numpy(float)
    oldrho=float(spearmanr(lum,original).statistic)
    newrho=float(spearmanr(lum,newer).statistic)
    samples=[]
    for k in range(1000):
        ix=rng15.integers(0,len(gg),len(gg))
        if len(np.unique(lum[ix]))<3:continue
        a=float(spearmanr(lum[ix],original[ix]).statistic)
        b=float(spearmanr(lum[ix],newer[ix]).statistic)
        if np.isfinite(a) and np.isfinite(b):
            samples.append([a,b,b-a,abs(b)-abs(a)])
    ci=np.quantile(np.asarray(samples),[.025,.975],axis=0)
    results15.append(dict(group=name,n=len(gg),
        frozen_rho=oldrho,M1_rho=newrho,
        signed_delta=newrho-oldrho,change_in_absolute_rho=abs(newrho)-abs(oldrho),
        frozen_95CI_low=ci[0,0],frozen_95CI_high=ci[1,0],
        M1_95CI_low=ci[0,1],M1_95CI_high=ci[1,1],
        absolute_change_95CI_low=ci[0,3],absolute_change_95CI_high=ci[1,3]))
correlations15=pd.DataFrame(results15)
correlations15.to_csv(REPORT15/"15_Luki_illumination_correlations.csv",index=False)
display(correlations15.round(3))

# Same matched crown sample, same luminance bins for both models.
paired15["brightness_bin"]=pd.cut(paired15.median_luminance,
    np.linspace(0,1,11),include_lowest=True)
rows15=[]
for band,gg in paired15.groupby("brightness_bin",observed=True):
    for model,col in [("Frozen","frozen_p"),("M1","M1_p")]:
        positives=gg[gg.label==1];negatives=gg[gg.label==0]
        rows15.append(dict(bin=str(band),model=model,crowns=len(gg),
            musanga=len(positives),non_musanga=len(negatives),
            median_luminance=float(gg.median_luminance.median()),
            mean_probability=float(gg[col].mean()),
            musanga_recall=float((positives[col]>=.5).mean()) if len(positives) else np.nan,
            non_musanga_false_positive_rate=float((negatives[col]>=.5).mean()) if len(negatives) else np.nan))
bins15=pd.DataFrame(rows15)
bins15.to_csv(REPORT15/"15_Luki_matched_brightness_bins.csv",index=False)
display(bins15.round(3))
fig,axes=plt.subplots(2,2,figsize=(12,9),sharex=True,sharey=True,constrained_layout=True)
for row,cls in enumerate([1,0]):
    group=paired15[paired15.label==cls]
    for col,(name,score) in enumerate([("Frozen production U-Net","frozen_p"),
                                        ("Final four-site M1","M1_p")]):
        ax=axes[row,col]
        ax.scatter(group.median_luminance,group[score],alpha=.4,s=16)
        rho=spearmanr(group.median_luminance,group[score]).statistic
        ax.set_title(f"{'Musanga' if cls else 'Non-Musanga'} — {name}\nn={len(group)}; Spearman ρ={rho:.3f}")
        ax.set(xlabel="Crown native-RGB luminance",ylabel="Predicted P(Musanga)",
               xlim=(0,1),ylim=(0,1))
        ax.axhline(.5,color="gray",ls="--",lw=1)
fig.savefig(REPORT15/"15_Luki_illumination_vs_probability.png",dpi=160)
plt.show()
paired15.to_csv(REPORT15/"15_Luki_matched_crown_probabilities.csv",index=False)
print("Section 15 complete:",REPORT15)
print("These are crown-level correlations, NOT proof that wall-to-wall mosaics lack shadow artefacts.")


## 16 — Final M1: full-resolution Luki Musanga probability map (GPU, resumable)

Produce a **native-grid, native-resolution** probability GeoTIFF from the already trained final four-site M1 checkpoint. The output is stored exclusively under `KVO_work/V2/04_illumination_robust_training/final_four_site_M1/<run_id>/luki_wall_to_wall/` and never overwrites an original UAV or historical production probability map.

**Efficient edge-safe inference:** each non-overlapping **512 × 512 output tile** is predicted from a **640 × 640 input window with a 64-pixel context halo** on all sides; the network's halo predictions are discarded. This avoids dividing a ~1.5-billion-pixel orthomosaic into large RAM arrays and minimizes tile-edge artefacts. The raster is written directly, one tile at a time, as `uint16` probabilities scaled 0–10000, with `65535` as no-data, matching the existing frozen probability-map convention. Pixel-validity rules use **RGB only** (exclude black 0/0/0, white 255/255/255, and pure blue 0/0/255); **never Luki's unreliable 4th band/alpha**. Zero illumination corrections are applied at inference: this is a true test of M1's photometric robustness.

Run **Section 12 first in this runtime** (for common model definitions/path variables). Section 13 is needed only when the final M1 checkpoint does not yet exist. Section 16 reuses an existing checkpoint and has row-wise progress/resume state on Drive. This processes the **entire Luki image**, so budget potentially considerable GPU time and ~3 GB for the final compressed/uncompressed output; a 3-row smoke-test option is provided. Do not compare satellite transfer until the completed map has passed illumination QA.

**Note:** halo tiling is not weighted overlap-blending; it discards edge predictions while retaining context. Inspect tile seams in the resulting map before treating it as a scientific product.

In [ ]:
# V2 / NB04 / Section 16: memory-bounded native Luki inference.
# Depends on Section 12 definitions, NOT previous pilot/LOSO execution.
import os, time, json, hashlib
from pathlib import Path
import numpy as np, rasterio, torch
import segmentation_models_pytorch as smp
from rasterio.windows import Window
from tqdm.auto import tqdm

assert torch.cuda.is_available(),"Use a GPU runtime for full-image prediction."
assert "FINAL_RUN04" in globals() and "FINAL_CKPT04" in globals(), \
    "Run Section 12, then Section 13 if the final M1 model has not been trained."
assert FINAL_CKPT04.exists(),f"Missing final M1 checkpoint: {FINAL_CKPT04}"
assert LUKI_RGB04.exists(),f"Missing Luki source RGB: {LUKI_RGB04}"

OUTPUT_TILE16=512      # output grid stride: no full-mosaic in-memory buffers
HALO16=64             # contextual overlap on each side (input size 640, multiple of 32)
BATCH16=4             # 4 x 640px fits typical L4 GPU; lower to 2 if needed
TEST_ROWS16=None      # e.g. 3 for a small test; set None for full Luki production
FORCE_NEW16=False     # True only to intentionally restart/replace incomplete output
OUT16=FINAL_RUN04/"luki_wall_to_wall"
OUT16.mkdir(parents=True,exist_ok=True)
PARTIAL16=OUT16/"Luki_M1_probability_uint16_NATIVE.partial.tif"
FINAL16=OUT16/"Luki_M1_probability_uint16_NATIVE.tif"
PROGRESS16=OUT16/"Luki_M1_inference_progress.json"
MANIFEST16=OUT16/"Luki_M1_inference_manifest.json"
SCALE16=10000
NODATA16=65535
VARIANT16="M1_native_photometric"

# Exact checkpoint, no guessing by filename or file modification time.
ckpt16=torch.load(FINAL_CKPT04,map_location="cpu",weights_only=False)
assert ckpt16.get("run_id")==FINAL_RUN_ID04 and ckpt16.get("variant")==VARIANT16, \
    "Checkpoint does not match final four-forest M1 run"
# Encoder architecture matches training; ImageNet weights need NOT be re-downloaded,
# since the complete trained checkpoint contains every encoder and decoder weight.
model16=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
model16.load_state_dict(ckpt16["model"],strict=True)
model16=model16.to("cuda").eval()
mean16=torch.tensor([.485,.456,.406],device="cuda").view(1,3,1,1)
std16=torch.tensor([.229,.224,.225],device="cuda").view(1,3,1,1)

with rasterio.open(LUKI_RGB04) as src:
    width16,height16=src.width,src.height
    assert src.count>=3 and src.dtypes[0]=="uint8", \
        "Validate unexpected Luki RGB channel type before inference."
    source_meta16=dict(width=src.width,height=src.height,
        crs=str(src.crs),transform=tuple(src.transform),
        source_size=LUKI_RGB04.stat().st_size,
        source_mtime_ns=LUKI_RGB04.stat().st_mtime_ns)
    profile16=src.profile.copy()

rows16=(height16+OUTPUT_TILE16-1)//OUTPUT_TILE16
cols16=(width16+OUTPUT_TILE16-1)//OUTPUT_TILE16
config16=dict(model_run=FINAL_RUN_ID04,checkpoint=str(FINAL_CKPT04),
    checkpoint_bytes=FINAL_CKPT04.stat().st_size,
    checkpoint_mtime_ns=FINAL_CKPT04.stat().st_mtime_ns,
    RGB_source=str(LUKI_RGB04),RGB_meta=source_meta16,
    output_tile=OUTPUT_TILE16,halo=HALO16,scale=SCALE16,
    nodata=NODATA16,normalization="ImageNet mean/std",
    input_bands=[1,2,3],
    excluded_RGB=["0/0/0","255/255/255","0/0/255"],
    no_alpha_or_raster_dataset_mask=True,illumination_gain="none",
    prediction="sigmoid binary Musanga model; context-halo tiled, no overlap-blending")
fingerprint16=hashlib.sha256(json.dumps(config16,sort_keys=True).encode()).hexdigest()[:16]
print("Native Luki image:",width16,"x",height16,"pixels")
print("Output windows:",rows16,"rows x",cols16,"columns =",rows16*cols16,"tiles")
print("Run:",FINAL_RUN_ID04,"| inference contract:",fingerprint16)
print("Output:",FINAL16)
print("Test mode:",TEST_ROWS16 is not None,"| GPU:",torch.cuda.get_device_name(0),flush=True)

if FINAL16.exists() and not FORCE_NEW16:
    assert MANIFEST16.exists(),"Found completed map without manifest; inspect before reusing"
    saved16=json.loads(MANIFEST16.read_text())
    assert saved16["fingerprint"]==fingerprint16, "Finished raster contract differs from requested run"
    print("REUSING FINISHED PROBABILITY MAP:",FINAL16,flush=True)
else:
    if FORCE_NEW16:
        for p in [PARTIAL16,PROGRESS16,MANIFEST16]:
            if p.exists():p.unlink()
        # Preserve completed prior outputs unless intentionally archived/renamed by user.
        assert not FINAL16.exists(),"Move/archive the completed map before requesting a new run."

    profile16.update(driver="GTiff",count=1,dtype="uint16",
        nodata=NODATA16,compress="deflate",predictor=2,
        tiled=True,blockxsize=512,blockysize=512,bigtiff="YES",
        interleave="band")
    if PARTIAL16.exists():
        assert PROGRESS16.exists(),"Partial raster exists without progress metadata; inspect before restarting."
        prev16=json.loads(PROGRESS16.read_text())
        assert prev16["fingerprint"]==fingerprint16,"Partial raster belongs to a different checkpoint/configuration"
        starting_row16=int(prev16["completed_output_rows"])
        print("RESUMING at completed output tile-row",starting_row16,flush=True)
        mode16="r+"
    else:
        starting_row16=0
        mode16="w"
    stopping_row16=rows16 if TEST_ROWS16 is None else min(rows16,starting_row16+int(TEST_ROWS16))

    def valid16(rgb):
        # Only source RGB; Luki alpha / dataset mask and nodata metadata are unreliable.
        return ~(np.all(rgb==0,axis=2)|np.all(rgb==255,axis=2)|
                 ((rgb[:,:,0]==0)&(rgb[:,:,1]==0)&(rgb[:,:,2]==255)))

    def tile16(src,x,y):
        # Input includes 64 px context, even outside mosaic (RGB padding = zero).
        # Output centre is exactly 512x512 at the native source transform.
        W=OUTPUT_TILE16+2*HALO16
        win=Window(x-HALO16,y-HALO16,W,W)
        rgb=np.moveaxis(src.read([1,2,3],window=win,boundless=True,fill_value=0),0,-1)
        central=rgb[HALO16:HALO16+OUTPUT_TILE16,HALO16:HALO16+OUTPUT_TILE16]
        return np.moveaxis(rgb,2,0),valid16(central)

    started16=time.monotonic()
    with rasterio.open(LUKI_RGB04) as src:
        with rasterio.open(PARTIAL16,mode16,**(profile16 if mode16=="w" else {})) as dst:
            for row16 in tqdm(range(starting_row16,stopping_row16),
                              desc="Luki M1 output tile-rows",unit="row"):
                y=row16*OUTPUT_TILE16
                for col0 in range(0,cols16,BATCH16):
                    batch=[];masks=[];positions=[]
                    for col in range(col0,min(cols16,col0+BATCH16)):
                        x=col*OUTPUT_TILE16
                        rgb,valid=tile16(src,x,y)
                        batch.append(rgb)
                        masks.append(valid)
                        positions.append((x,y,min(OUTPUT_TILE16,width16-x),
                                           min(OUTPUT_TILE16,height16-y)))
                    tensor=torch.from_numpy(np.stack(batch)).to("cuda",non_blocking=True).float()/255.
                    tensor=(tensor-mean16)/std16
                    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
                        pred=torch.sigmoid(model16(tensor)).float().cpu().numpy()[:,0]
                    for bi,(x,y,hwidth,hheight) in enumerate(positions):
                        center=pred[bi,HALO16:HALO16+hheight,
                                       HALO16:HALO16+hwidth]
                        mask=masks[bi][:hheight,:hwidth]
                        scaled=np.where(mask,np.rint(np.clip(center,0,1)*SCALE16),
                                        NODATA16).astype(np.uint16)
                        dst.write(scaled,1,window=Window(x,y,hwidth,hheight))
                    del tensor,pred,batch,masks
                # Only mark progress after the entire horizontal tile-row is saved.
                (PROGRESS16).write_text(json.dumps(dict(fingerprint=fingerprint16,
                    completed_output_rows=row16+1,expected_rows=rows16,
                    partial_output=str(PARTIAL16),last_updated_epoch=time.time()),indent=2))
                if (row16+1)%8==0 or row16==starting_row16:
                    elapsed=time.monotonic()-started16
                    print(f"OUTPUT PROGRESS: {row16+1}/{rows16} tile rows; "
                          f"elapsed {elapsed/60:.1f} min",flush=True)

    if stopping_row16<rows16:
        print(f"PARTIAL PREVIEW: {stopping_row16}/{rows16} tile rows saved; "
              "set TEST_ROWS16=None and rerun Section 16 to complete.",flush=True)
    else:
        # Check dimensions and source-grid equality after closing the writer.
        with rasterio.open(PARTIAL16) as written,rasterio.open(LUKI_RGB04) as src:
            assert (written.width,written.height)==(src.width,src.height)
            assert written.transform==src.transform and written.crs==src.crs
            assert written.nodata==NODATA16 and written.dtypes[0]=="uint16"
        os.replace(PARTIAL16,FINAL16)
        MANIFEST16.write_text(json.dumps(dict(
            fingerprint=fingerprint16,config=config16,
            output=str(FINAL16),all_tile_rows_completed=rows16,
            completed=True,source_native_grid_preserved=True,
            estimated_output_size_bytes=FINAL16.stat().st_size,
            elapsed_seconds=float(time.monotonic()-started16)),indent=2))
        if PROGRESS16.exists():PROGRESS16.unlink()
        print("COMPLETE LUKI FINAL M1 PROBABILITY MAP:",FINAL16,flush=True)
        print("DO NOT treat as illumination-corrected until spatial shadow QA is completed.",flush=True)


## 17 — Compare historical versus final M1 full-resolution Luki probability maps (CPU)

**Run directly after Section 16**, or after Section 12 with the completed M1 map on Drive. This reads the **historical frozen production Luki probability GeoTIFF** and the **new final M1 GeoTIFF** on the exact same pixel grid, using independent RGB-only validity masks. It performs a **complete streamed pixel census by default**, never loading the full 1.47-billion-pixel maps into RAM, and saves a compact CSV summary, probability histograms and four-panel maps (RGB, old, new, difference).

We compare *probabilities and thresholded predicted cover* on pixels valid in both maps, with no threshold retuning. Coverage may be affected by probability calibration; a change in the fraction of pixels with P ≥ 0.5 is **not automatically a real change in Musanga abundance**. We also summarize brightness versus probability in spatial windows to reveal systematic illumination relationships. Interpret the window correlations cautiously because tree-species composition also varies spatially.

For a quick smoke test choose `FULL_SCAN17=False` (approximately 150 spatially distributed windows); the default `True` measures all pixels. Neither image is altered; outputs go to `V2/04_illumination_robust_training/final_four_site_M1/<run_id>/luki_wall_to_wall/comparison/`.

In [ ]:
# Section 17: exact grid-aligned old vs M1 native Luki probability comparison.
# CPU-only, zero model inference and bounded-memory rasterio windows.
from pathlib import Path
import math,json
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from scipy.stats import spearmanr
from IPython.display import display
from tqdm.auto import tqdm

ROOT17=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
OLD17=ROOT17/"KVO_work/KVO_04_final_multisite/probability_maps/Luki_musanga_probability_uint16_FULL.tif"
NEW17=ROOT17/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f/luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
RGB17=ROOT17/"data/Geo_Ref/Luki.tif"
OUT17=NEW17.parent/"comparison"
OUT17.mkdir(parents=True,exist_ok=True)
FULL_SCAN17=True      # False = fast, spatially distributed 150-window sample
SAMPLE_WINDOWS17=150
WINDOW17=512
PREVIEW17=1024
SCALE17=10000.
NODATA17=65535
THRESH17=.50
assert all(p.exists() for p in [OLD17,NEW17,RGB17]), \
    "Missing a raster; check OLD17, NEW17 and RGB17 paths."
def rgb_valid17(c):
    return ~(np.all(c==0,axis=0)|np.all(c==255,axis=0)|
             ((c[0]==0)&(c[1]==0)&(c[2]==255)))
def usable17(v,src):
    return (v!=NODATA17)&(v<=SCALE17)&np.isfinite(v)

# Stream window pairs to avoid >10GB in-memory arrays.
with rasterio.open(OLD17) as old,rasterio.open(NEW17) as new,rasterio.open(RGB17) as rgb:
    assert (old.width,old.height,old.transform,old.crs)== \
           (new.width,new.height,new.transform,new.crs), \
           "Original and new probability rasters are not on identical grids."
    assert (rgb.width,rgb.height,rgb.transform,rgb.crs)== \
           (new.width,new.height,new.transform,new.crs), \
           "RGB and probability grids differ; investigate before comparing."
    assert old.count==new.count==1
    assert old.dtypes[0]==new.dtypes[0]=="uint16"
    assert old.nodata==new.nodata==NODATA17, \
        "Unexpected no-data conventions: check rather than silently compare."
    nx=math.ceil(old.width/WINDOW17)
    ny=math.ceil(old.height/WINDOW17)
    all_windows=[(ix,iy) for iy in range(ny) for ix in range(nx)]
    if FULL_SCAN17:
        selected=all_windows
    else:
        indices=np.unique(np.linspace(0,len(all_windows)-1,
                                       min(SAMPLE_WINDOWS17,len(all_windows))).round().astype(int))
        selected=[all_windows[int(i)] for i in indices]
    print("Mode:", "FULL MAP CENSUS" if FULL_SCAN17 else "SPATIAL SUBSAMPLE")
    print("Output windows:",len(selected),"/",len(all_windows))
    totals=dict(compared_pixels=0,original_only_pixels=0,new_only_pixels=0,
                rgb_valid_pixels=0,old_sum=0.,new_sum=0.,difference_sum=0.,
                difference_squared_sum=0.,old_above=0,new_above=0,
                old_very_high=0,new_very_high=0)
    hist_old=np.zeros(50,dtype=np.int64)
    hist_new=np.zeros(50,dtype=np.int64)
    window_rows=[]
    for ix,iy in tqdm(selected,desc="Paired Luki maps (CPU windows)"):
        x0=ix*WINDOW17;y0=iy*WINDOW17
        w=min(WINDOW17,old.width-x0);h=min(WINDOW17,old.height-y0)
        win=Window(x0,y0,w,h)
        o=old.read(1,window=win)
        n=new.read(1,window=win)
        c=rgb.read([1,2,3],window=win)
        rv=rgb_valid17(c)
        ov=rv&usable17(o,old)
        nv=rv&usable17(n,new)
        valid=ov&nv
        totals["rgb_valid_pixels"]+=int(rv.sum())
        totals["original_only_pixels"]+=int((ov&~nv).sum())
        totals["new_only_pixels"]+=int((nv&~ov).sum())
        count=int(valid.sum())
        if count<100:continue
        old_p=o[valid].astype(np.float32)/SCALE17
        new_p=n[valid].astype(np.float32)/SCALE17
        diff=new_p-old_p
        totals["compared_pixels"]+=count
        totals["old_sum"]+=float(old_p.sum(dtype=np.float64))
        totals["new_sum"]+=float(new_p.sum(dtype=np.float64))
        totals["difference_sum"]+=float(diff.sum(dtype=np.float64))
        totals["difference_squared_sum"]+=float(np.square(diff).sum(dtype=np.float64))
        totals["old_above"]+=int((old_p>=THRESH17).sum())
        totals["new_above"]+=int((new_p>=THRESH17).sum())
        totals["old_very_high"]+=int((old_p>=.9).sum())
        totals["new_very_high"]+=int((new_p>=.9).sum())
        hist_old+=np.histogram(old_p,bins=50,range=(0,1))[0]
        hist_new+=np.histogram(new_p,bins=50,range=(0,1))[0]
        brightness=(.2126*c[0].astype(np.float32)+
                    .7152*c[1].astype(np.float32)+
                    .0722*c[2].astype(np.float32))/255.
        window_rows.append(dict(ix=ix,iy=iy,n=count,
             median_rgb_brightness=float(np.median(brightness[valid])),
             original_mean_p=float(old_p.mean()),M1_mean_p=float(new_p.mean()),
             mean_delta_p=float(diff.mean()),
             original_cover_0p5=float((old_p>=THRESH17).mean()),
             M1_cover_0p5=float((new_p>=THRESH17).mean())))
    assert totals["compared_pixels"]>0,"No jointly valid source pixels; verify map alignment and no-data."
    z=totals["compared_pixels"]
    summary17=dict(scope="all_RGB_and_both_model_valid_pixels" if FULL_SCAN17 else "sampled_RGB_and_both_model_valid_pixels",
        spatial_windows=len(selected),compared_pixels=z,
        valid_RGB_pixels=totals["rgb_valid_pixels"],
        original_only_valid_pixels=totals["original_only_pixels"],
        M1_only_valid_pixels=totals["new_only_pixels"],
        original_mean_probability=totals["old_sum"]/z,
        M1_mean_probability=totals["new_sum"]/z,
        mean_M1_minus_original=totals["difference_sum"]/z,
        RMSE_probability_difference=math.sqrt(totals["difference_squared_sum"]/z),
        original_pixel_fraction_ge_0p5=totals["old_above"]/z,
        M1_pixel_fraction_ge_0p5=totals["new_above"]/z,
        change_fraction_ge_0p5=(totals["new_above"]-totals["old_above"])/z,
        original_pixel_fraction_ge_0p9=totals["old_very_high"]/z,
        M1_pixel_fraction_ge_0p9=totals["new_very_high"]/z)
    df17=pd.DataFrame([summary17])
    windows17=pd.DataFrame(window_rows)
    df17.to_csv(OUT17/"17_Luki_map_comparison_summary.csv",index=False)
    windows17.to_csv(OUT17/"17_Luki_window_brightness_vs_probability.csv",index=False)
    print("Paired full-map probability and cover comparison:")
    display(df17.T.rename(columns={0:"value"}).round(5))
    if len(windows17)>5:
        print("Window-level Spearman: median RGB brightness vs mean Musanga probability")
        rho_rows=[]
        for name,col in [("Historical U-Net","original_mean_p"),("Final M1","M1_mean_p")]:
            rr=spearmanr(windows17.median_rgb_brightness,windows17[col])
            rho_rows.append(dict(model=name,n_windows=len(windows17),
                                 rho=float(rr.statistic),p_value=float(rr.pvalue)))
        display(pd.DataFrame(rho_rows).round(4))
    fig,ax=plt.subplots(figsize=(8,4))
    centers=(np.arange(50)+.5)/50
    ax.plot(centers,hist_old/z,label="Original")
    ax.plot(centers,hist_new/z,label="Final M1")
    ax.set(xlabel="Pixel P(Musanga)",ylabel="Fraction of compared valid pixels",
           title="Luki: matched-pixel probability distribution")
    ax.legend()
    fig.savefig(OUT17/"17_Luki_probability_distributions.png",dpi=150,bbox_inches="tight")
    plt.show()

    # Representative high-support window: choose from the middle of eligible sample windows.
    eligible=windows17.loc[windows17.n>=WINDOW17*WINDOW17*.65].copy()
    if eligible.empty:eligible=windows17.copy()
    if len(eligible):
        ordered=eligible.sort_values("median_rgb_brightness")
        picks=[ordered.iloc[int(q*(len(ordered)-1))] for q in [.15,.5,.85]]
        for j,rec in enumerate(picks,1):
            cx=int(rec.ix)*WINDOW17+WINDOW17//2
            cy=int(rec.iy)*WINDOW17+WINDOW17//2
            px=max(0,min(old.width-PREVIEW17,cx-PREVIEW17//2))
            py=max(0,min(old.height-PREVIEW17,cy-PREVIEW17//2))
            pw=Window(px,py,min(PREVIEW17,old.width-px),min(PREVIEW17,old.height-py))
            src_img=np.moveaxis(rgb.read([1,2,3],window=pw),0,-1)
            po=old.read(1,window=pw)
            pn=new.read(1,window=pw)
            valid=rgb_valid17(np.moveaxis(src_img,2,0))&usable17(po,old)&usable17(pn,new)
            a=np.where(valid,po.astype(np.float32)/SCALE17,np.nan)
            b=np.where(valid,pn.astype(np.float32)/SCALE17,np.nan)
            fig,axes=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
            axes[0].imshow(src_img);axes[0].set_title("Luki RGB")
            for ax,im,title in [(axes[1],a,"Historical probability"),
                                (axes[2],b,"Final M1 probability")]:
                ax.imshow(im,cmap="viridis",vmin=0,vmax=1);ax.set_title(title)
            axes[3].imshow(b-a,cmap="coolwarm",vmin=-1,vmax=1)
            axes[3].set_title("M1 − historical")
            for ax in axes:ax.axis("off")
            fig.savefig(OUT17/f"17_Luki_matched_RGB_old_M1_difference_{j}.png",
                        dpi=140,bbox_inches="tight")
            plt.show()
    (OUT17/"17_comparison_manifest.json").write_text(json.dumps(dict(
        original_probability=str(OLD17),M1_probability=str(NEW17),
        source_RGB=str(RGB17),full_pixel_census=FULL_SCAN17,
        compared_pixels=z,threshold=THRESH17,summary=summary17),indent=2))
    print("SAVED CPU-ONLY LUKI MAP COMPARISON:",OUT17,flush=True)
    print("Fractions above 0.5 are thresholded probability-map fractions, NOT validated species cover.")


## 18 — Native-resolution final M1 maps for all four development sites (GPU; resumable)

Apply the **exact final four-site M1 checkpoint** used for Luki, without retraining, to all four development sites. All four forests were included in the final four-site training dataset, so **these maps are *in-sample visual diagnostics*, not independent transfer tests**. The earlier LOSO checkpoints provide independent held-out crown scores, but those are different models.

This reuses Section 16's approach: 512 px output tiles, 64 px context halo, input RGB bands 1–3, ImageNet normalization, no illumination gain or alpha/mask filtering. Pixels are classified as invalid only when RGB is black, white or pure blue. Stream directly to an independently resumable `uint16` (0–10000; NoData 65535) GeoTIFF on each raster's untouched native grid.

**Fresh runtime:** Run Section 12 first to load basic variables/model definitions (Section 13 is unnecessary if the final checkpoint already exists). Then run Section 18. By default **Yoko, Baego, MOS_1_2 and Yangambi** run sequentially. Existing completed maps are verified and skipped. These are very large mosaics (Baego alone ~5.5 billion native pixels), so you can limit `SITES18` to one site, and interrupted maps resume from completed rows. `TEST_TILE_ROWS18=3` allows an initial test; set back to `None` for the complete map.

Outputs are under the current run's `final_four_site_M1/<run_id>/site_wall_to_wall/<site>/`. Section 19 provides a lightweight comparison preview using exactly aligned old/new maps and native RGB; no whole-mosaic RAM allocation.

In [ ]:
# V2 / NB04 / Section 18 — Yoko + Baego native-wall-to-wall final M1 inference.
# Run after Section 12; no Section 13 training needed when checkpoint already exists.
import os,json,time,math,hashlib
from pathlib import Path
import numpy as np,rasterio,torch
import segmentation_models_pytorch as smp
from rasterio.windows import Window
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Choose Colab GPU for wall-to-wall M1 inference."
assert "FINAL_CKPT04" in globals() and FINAL_CKPT04.exists(), \
    "Run Section 12 (and train Section 13 only if no final M1 checkpoint exists)."
assert "FINAL_RUN_ID04" in globals() and "FINAL_RUN04" in globals(), \
    "Run Section 12 first to resolve the exact final M1 run."

SITES18=["Yoko","Baego","MOS_1_2","Yangambi"] # resume completed sites; select a subset if preferred
TILE18=512
HALO18=64
BATCH18=4
TEST_TILE_ROWS18=None    # 3 for a quick resumable smoke test, None for complete image
SCALE18=10000
NODATA18=65535
assert SITES18 and set(SITES18).issubset(set(SITES))
assert (TILE18+2*HALO18)%32==0
assert TILE18>0 and HALO18>0
assert BATCH18>=1

# Load explicitly selected checkpoint, not whatever .pt happens to be newest.
ckpt18=torch.load(FINAL_CKPT04,map_location="cpu",weights_only=False)
assert ckpt18.get("run_id")==FINAL_RUN_ID04
assert ckpt18.get("variant")=="M1_native_photometric"
model18=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
model18.load_state_dict(ckpt18["model"],strict=True)
model18=model18.to("cuda").eval()
mean18=torch.tensor([.485,.456,.406],device="cuda").view(1,3,1,1)
std18=torch.tensor([.229,.224,.225],device="cuda").view(1,3,1,1)
print("Final M1 checkpoint:",FINAL_CKPT04,flush=True)
print("GPU:",torch.cuda.get_device_name(0),flush=True)

def valid_rgb18(c):
    # c: bands x height x width; never consult the 4th band/alpha.
    return ~(np.all(c==0,axis=0)|np.all(c==255,axis=0)|
             ((c[0]==0)&(c[1]==0)&(c[2]==255)))

def process_site18(site):
    src_path=GEO/SITES[site][0]
    assert src_path.exists(),f"Missing RGB for {site}: {src_path}"
    output_dir=FINAL_RUN04/"site_wall_to_wall"/site
    output_dir.mkdir(parents=True,exist_ok=True)
    final=output_dir/f"{site}_M1_probability_uint16_NATIVE.tif"
    partial=output_dir/f"{site}_M1_probability_uint16_NATIVE.partial.tif"
    progress=output_dir/"18_inference_progress.json"
    manifest=output_dir/"18_inference_manifest.json"

    with rasterio.open(src_path) as src:
        assert src.count>=3 and src.dtypes[0]=="uint8"
        profile=src.profile.copy()
        width,height=src.width,src.height
        nrows=math.ceil(height/TILE18)
        ncols=math.ceil(width/TILE18)
        identity=dict(site=site,RGB_path=str(src_path),RGB_bytes=src_path.stat().st_size,
            RGB_mtime_ns=src_path.stat().st_mtime_ns,source_shape=[width,height],
            source_crs=str(src.crs),source_transform=list(src.transform),
            checkpoint=str(FINAL_CKPT04),checkpoint_bytes=FINAL_CKPT04.stat().st_size,
            checkpoint_mtime_ns=FINAL_CKPT04.stat().st_mtime_ns,
            model_run_id=FINAL_RUN_ID04,model="four_site_M1",
            tile=TILE18,halo=HALO18,scale=SCALE18,nodata=NODATA18,
            input="RGB1-3 only, black/white/blue invalid, no alpha mask",
            network="SMP Unet ResNet18; ImageNet mean/std; sigmoid; centre of halo-tile")
        fp=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]
        print(f"\n{site}: {width} x {height}; {nrows*ncols:,} tiles; output {final}",flush=True)

        if final.exists():
            assert manifest.exists(),f"Completed raster without manifest: {final}"
            oldmeta=json.loads(manifest.read_text())
            assert oldmeta["fingerprint"]==fp, \
                f"{site}: completed map from different model/input; inspect before reuse"
            print(f"{site}: already complete, verified matching fingerprint — skipped",flush=True)
            return final

        if partial.exists():
            assert progress.exists(),f"{site}: partial map lacks resume metadata"
            state=json.loads(progress.read_text())
            assert state["fingerprint"]==fp,f"{site}: resume contract mismatch"
            start=int(state["completed_rows"])
            assert 0<=start<=nrows
            mode="r+"
            print(f"{site}: RESUMING from completed tile-row {start}/{nrows}",flush=True)
        else:
            assert not progress.exists(),f"{site}: stale progress without partial TIFF"
            start=0
            mode="w"

        finish=nrows if TEST_TILE_ROWS18 is None else min(nrows,start+int(TEST_TILE_ROWS18))
        if mode=="w":
            profile.update(driver="GTiff",count=1,dtype="uint16",nodata=NODATA18,
                compress="deflate",predictor=2,tiled=True,
                blockxsize=512,blockysize=512,bigtiff="YES",interleave="band")
        begun=time.monotonic()
        with rasterio.open(partial,mode,**(profile if mode=="w" else {})) as dst:
            for row in tqdm(range(start,finish),desc=f"{site} M1 tile rows",unit="row"):
                y=row*TILE18
                for firstcol in range(0,ncols,BATCH18):
                    patches=[];valids=[];windows=[]
                    for col in range(firstcol,min(ncols,firstcol+BATCH18)):
                        x=col*TILE18
                        rgb=src.read([1,2,3],window=Window(x-HALO18,y-HALO18,
                            TILE18+2*HALO18,TILE18+2*HALO18),
                            boundless=True,fill_value=0)
                        center=rgb[:,HALO18:HALO18+TILE18,HALO18:HALO18+TILE18]
                        patches.append(rgb)
                        valids.append(valid_rgb18(center))
                        windows.append((x,y,min(TILE18,width-x),min(TILE18,height-y)))
                    inputs=torch.from_numpy(np.stack(patches)).to("cuda").float()/255.
                    inputs=(inputs-mean18)/std18
                    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
                        pred=torch.sigmoid(model18(inputs)).float().cpu().numpy()[:,0]
                    for j,(x,y,w,h) in enumerate(windows):
                        output=pred[j,HALO18:HALO18+h,HALO18:HALO18+w]
                        valid=valids[j][:h,:w]
                        data=np.where(valid,np.rint(np.clip(output,0,1)*SCALE18),
                                      NODATA18).astype("uint16")
                        dst.write(data,1,window=Window(x,y,w,h))
                    del inputs,pred,patches
                # Only a fully written row is marked complete. Safe restart redoes partial row.
                progress.write_text(json.dumps(dict(fingerprint=fp,site=site,
                    completed_rows=row+1,total_rows=nrows),indent=2))
                if (row+1)%10==0 or row==start:
                    print(f"{site}: {row+1}/{nrows} tile-rows; "
                          f"{(time.monotonic()-begun)/60:.1f} min",flush=True)

        if finish<nrows:
            print(f"{site}: saved partial {finish}/{nrows} tile rows; rerun with "
                  "TEST_TILE_ROWS18=None to finish",flush=True)
        else:
            with rasterio.open(partial) as predicted:
                assert predicted.width==src.width and predicted.height==src.height
                assert predicted.transform==src.transform and predicted.crs==src.crs
                assert predicted.count==1 and predicted.nodata==NODATA18
            os.replace(partial,final)
            manifest.write_text(json.dumps(dict(fingerprint=fp,config=identity,
                completed=True,output=str(final),total_rows=nrows,
                elapsed_seconds=time.monotonic()-begun),indent=2))
            progress.unlink(missing_ok=True)
            print(f"{site}: COMPLETE {final}",flush=True)
    return final if final.exists() else partial

FINISHED18={}
for site18 in SITES18:
    FINISHED18[site18]=process_site18(site18)
print("Site outputs:",FINISHED18)


## 19 — Visual RGB / historical / final M1 panels for all four development forests (CPU)

After Section 18 completes, this generates **matched native-grid panels** for each available site: RGB, historical probability, M1 probability and M1 minus historical. Three 1024 × 1024 pixel windows are selected near the image centre (and horizontally spaced) for quick visual comparison; no full-image array is loaded. Both probability maps must be complete. Saves PNG previews in each site's `comparison_previews` folder.

These sites were used to train final M1; **do not present their mapped behaviour as external generalization validation**. The comparisons show spatial artefacts and apparent calibration shifts, not ground-truth species cover.

In [ ]:
# Section 19: matching RGB / old production / new M1 preview windows.
# CPU-only; can run in same session or after Section 12.
from pathlib import Path
import numpy as np,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window

for site19 in ["Yoko","Baego","MOS_1_2","Yangambi"]:
    rgb19=GEO/SITES[site19][0]
    old19=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/f"{site19}_musanga_probability_uint16_FULL.tif"
    new19=FINAL_RUN04/"site_wall_to_wall"/site19/f"{site19}_M1_probability_uint16_NATIVE.tif"
    if not new19.exists():
        print(f"{site19}: full M1 map not yet complete; skip preview.")
        continue
    if not old19.exists():
        print(f"{site19}: historical map absent; skip preview ({old19}).")
        continue
    out19=new19.parent/"comparison_previews";out19.mkdir(parents=True,exist_ok=True)
    with rasterio.open(rgb19) as rgb,rasterio.open(old19) as old,rasterio.open(new19) as new:
        assert rgb.width==new.width==old.width and rgb.height==new.height==old.height
        assert rgb.transform==new.transform==old.transform and rgb.crs==new.crs==old.crs
        size=1024
        for j,fraction in enumerate([.25,.5,.75],1):
            x0=max(0,min(rgb.width-size,int(rgb.width*fraction-size/2)))
            y0=max(0,min(rgb.height-size,int(rgb.height*.50-size/2)))
            window=Window(x0,y0,size,size)
            colors=rgb.read([1,2,3],window=window)
            a=old.read(1,window=window)
            b=new.read(1,window=window)
            valid=~(np.all(colors==0,axis=0)|np.all(colors==255,axis=0)|
                    ((colors[0]==0)&(colors[1]==0)&(colors[2]==255)))
            valid&=(a!=65535)&(b!=65535)&(a<=10000)&(b<=10000)
            old_p=np.where(valid,a.astype(np.float32)/10000.,np.nan)
            new_p=np.where(valid,b.astype(np.float32)/10000.,np.nan)
            fig,ax=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
            ax[0].imshow(np.moveaxis(colors,0,-1))
            ax[0].set_title(f"{site19} RGB")
            ax[1].imshow(old_p,cmap="viridis",vmin=0,vmax=1)
            ax[1].set_title("Historical")
            ax[2].imshow(new_p,cmap="viridis",vmin=0,vmax=1)
            ax[2].set_title("Final M1")
            ax[3].imshow(new_p-old_p,cmap="coolwarm",vmin=-1,vmax=1)
            ax[3].set_title("M1 − historical")
            for a0 in ax:a0.axis("off")
            fig.savefig(out19/f"{site19}_old_M1_RGB_preview_{j}.png",
                        dpi=140,bbox_inches="tight")
            plt.show()
        print(site19,"visual previews:",out19)


## 20 — Critical audit: full-map crown scores versus crown-centred scores (CPU)

**Run Section 20 without retraining.** Extract the Musanga probability from the **complete final M1 Luki map** over each of the same 479 manually annotated crown polygons, including 73 crowns omitted by the 256-pixel crown-centred evaluation. Apply the exact same whole-crown extraction and valid-pixel requirements as Notebook 02, then compare original frozen and final M1 on (a) all jointly evaluated crowns and (b) the matched 406 crown-centred test subset. This isolates how much the apparent performance depends on inference context and the training/evaluation patch-centering convention.

This audit also saves a crown-by-crown comparison, including `full_M1 − centered_M1` and whether the crown was absent from the centred evaluation. It checks exact native raster grid alignment and does not use Luki labels for model training or threshold tuning. The `full-map` scores are generated by the **existing 512-pixel output tiling with a 64-pixel halo**, so these metrics will still reflect any systematic tiling artefacts.

**Interpretation:** Strong full-map performance would support real species discrimination *within annotated crowns*, but it would not validate unannotated canopy or total forest cover. Large deterioration would indicate that the centred-patch evaluation was overly optimistic for wall-to-wall mapping.

In [ ]:
# NB04 / Section 20 — exact native crown polygon sampling of both completed full maps.
# CPU-only. Standalone if the original and final GeoTIFFs and versioned Luki GPKG are present.
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,math
from rasterio.windows import Window,from_bounds
from rasterio.features import geometry_mask
from sklearn.metrics import (roc_auc_score,average_precision_score,precision_score,
                             recall_score,f1_score,balanced_accuracy_score)
from IPython.display import display
from tqdm.auto import tqdm
ROOT20=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
V220=ROOT20/"KVO_work"/"V2"
OLD20=ROOT20/"KVO_work/KVO_04_final_multisite/probability_maps/Luki_musanga_probability_uint16_FULL.tif"
RUN20="337073e7b74f"
NEW20=V220/"04_illumination_robust_training"/"final_four_site_M1"/RUN20/"luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
VERSION20="d957b570055bf5dd"
REFERENCE20=V220/"01_reference_splits"/"luki_reference_versions"/VERSION20/"01_Luki_frozen_external_reference.gpkg"
CENTERED20=V220/"04_illumination_robust_training"/"final_four_site_M1"/RUN20/"luki_external"/VERSION20/"04_final_M1_luki_crown_predictions.csv"
OUT20=NEW20.parent/"full_map_crown_audit"
OUT20.mkdir(parents=True,exist_ok=True)
for p in (OLD20,NEW20,REFERENCE20,CENTERED20):
    assert p.exists(),f"Required file missing: {p}"
crowns20=gpd.read_file(REFERENCE20,layer="external_evaluation")
assert len(crowns20)==479 and int(crowns20.label.sum())==249
assert crowns20.crown_id.is_unique
center20=pd.read_csv(CENTERED20,dtype={"crown_id":"string"})
assert center20.crown_id.is_unique
MAX_WINDOW20=2048
MIN_PIXELS20=10
MIN_COVERAGE20=.90
SCALE20=10000
def score_polygon20(geom,src):
    win=from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
    left=max(0,int(win.col_off));top=max(0,int(win.row_off))
    right=min(src.width,int(math.ceil(win.col_off+win.width)))
    bottom=min(src.height,int(math.ceil(win.row_off+win.height)))
    w=right-left;h=bottom-top
    if w<=0 or h<=0:return np.nan,0,0.,"outside"
    if w>MAX_WINDOW20 or h>MAX_WINDOW20:return np.nan,0,0.,"window_too_large"
    window=Window(left,top,w,h)
    mask=geometry_mask([geom],out_shape=(h,w),transform=src.window_transform(window),invert=True)
    data=src.read(1,window=window)
    valid=mask&(data!=65535)&(data<=SCALE20)
    count=int(valid.sum());coverage=count/max(int(mask.sum()),1)
    if count<MIN_PIXELS20 or coverage<MIN_COVERAGE20:
        return np.nan,count,coverage,"insufficient_valid"
    return float(data[valid].astype(np.float32).mean()/SCALE20),count,coverage,"ok"
rows20=[]
with rasterio.open(OLD20) as old,rasterio.open(NEW20) as new:
    assert old.width==new.width and old.height==new.height
    assert old.transform==new.transform and old.crs==new.crs
    assert crowns20.crs==new.crs
    for _,rec in tqdm(crowns20.iterrows(),total=len(crowns20),desc="Luki crown full-map comparison"):
        p0,k0,c0,q0=score_polygon20(rec.geometry,old)
        p1,k1,c1,q1=score_polygon20(rec.geometry,new)
        rows20.append(dict(crown_id=str(rec.crown_id),label=int(rec.label),
                           historical_map_p=p0,M1_full_map_p=p1,
                           historical_QA=q0,M1_full_QA=q1,
                           original_valid_pixels=k0,M1_valid_pixels=k1,
                           original_coverage=c0,M1_coverage=c1))
full20=pd.DataFrame(rows20).merge(
    center20[["crown_id","score","qa"]].rename(
        columns={"score":"M1_centered_p","qa":"M1_centered_QA"}),
    on="crown_id",how="left",validate="one_to_one")
full20["full_minus_centered"]=full20.M1_full_map_p-full20.M1_centered_p
full20.to_csv(OUT20/"20_all_479_Luki_crown_map_and_centred_scores.csv",index=False)
def summarize20(group,name,column):
    d=group[group[column].notna()].copy()
    if d.empty or d.label.nunique()<2:
        return dict(subset=name,model=column,n=len(d))
    y=d.label.to_numpy(int);p=d[column].to_numpy(float);pred=(p>=.5).astype(int)
    return dict(subset=name,model=column,n=len(d),musanga=int(y.sum()),other=int((y==0).sum()),
        AUC=float(roc_auc_score(y,p)),AP=float(average_precision_score(y,p)),
        precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),
        F1=float(f1_score(y,pred,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        FP=int(((pred==1)&(y==0)).sum()),FN=int(((pred==0)&(y==1)).sum()))
common=full20[(full20.historical_QA=="ok")&(full20.M1_full_QA=="ok")]
paired406=common[common.M1_centered_QA=="ok"].copy()
metrics20=[]
for label,part in [("all_both_maps",common),("matched_centered_crowns",paired406)]:
    for col in (["historical_map_p","M1_full_map_p"]+
                (["M1_centered_p"] if label=="matched_centered_crowns" else [])):
        metrics20.append(summarize20(part,label,col))
report20=pd.DataFrame(metrics20)
report20.to_csv(OUT20/"20_Luki_full_map_vs_centered_metrics.csv",index=False)
print("Evaluation counts: original",len(crowns20),
      "both full-map valid",len(common),"matched centered",len(paired406))
print("Excluded from centred evaluation:",(full20.M1_centered_QA!="ok").sum())
print("Species counts in common:",common.label.value_counts().to_dict())
display(report20.round(4))
if len(paired406):
    delta=paired406.full_minus_centered.dropna()
    print("Same-crown M1 full map minus crown-centred probability:",
          dict(n=len(delta),mean=float(delta.mean()),median=float(delta.median()),
               p10=float(delta.quantile(.1)),p90=float(delta.quantile(.9))))
    print("Centre/full M1 probability Spearman:",
          paired406[["M1_full_map_p","M1_centered_p"]].corr(method="spearman").iloc[0,1])
print("Crown audit saved to:",OUT20)


## 21 — Test the reported 512-pixel tiling artefact (CPU; old vs M1 vs RGB)

A tiled segmentation model can perform well on **crown-centred 256-pixel chips** yet produce systematic edge patterns when inferred wall-to-wall on **640-pixel inputs with a 64-pixel context halo**, writing 512-pixel outputs. This block explicitly tests discontinuities across every 512-pixel output-grid boundary using spatially distributed matched cross-boundary and within-tile pixel pairs.

It compares the mean **absolute jump in probability** across tile edges to the jump 256 pixels inside tiles, on the same source imagery, and checks the corresponding **RGB brightness jump**. It repeats the comparison for the original historical probability map. A large **M1-only edge/control ratio**, without matching RGB changes, would strongly support an inference seam artefact.

This is a diagnostic, not a perfect causal test: canopy textures also vary and the original map could have its own tiles. The block saves numeric seam statistics and example 2048 px visual panels with 512-grid lines. If a tiling effect is confirmed, the next step should be **a controlled re-inference experiment with larger halos or shifted grids**, not immediately changing the U-Net architecture or Luki training labels.

In [ ]:
# NB04 / Section 21 — CPU-only seam test, no inference, bounded raster window reads.
from pathlib import Path
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt,math
from rasterio.windows import Window
from IPython.display import display
from scipy.stats import spearmanr
ROOT21=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
OLD21=ROOT21/"KVO_work/KVO_04_final_multisite/probability_maps/Luki_musanga_probability_uint16_FULL.tif"
NEW21=ROOT21/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f/luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
RGB21=ROOT21/"data/Geo_Ref/Luki.tif"
OUT21=NEW21.parent/"tiling_audit"
OUT21.mkdir(parents=True,exist_ok=True)
GRID21=512
# Use multiple spatially distributed boundaries and pixel stretches.
RNG21=np.random.default_rng(20261013)
PAIRS_PER_DIRECTION21=150   # comparable seam/control pairs per orientation
STRIP21=512
def v21(rgb):
    return ~(np.all(rgb==0,axis=0)|np.all(rgb==255,axis=0)|
             ((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)))
def adjacent21(src,orient,position,offset,length):
    if orient=="vertical":
        # Pair at x=position+offset-1 and x=position+offset.
        return src.read(window=Window(position+offset-1,length[0],2,length[1]))
    return src.read(window=Window(length[0],position+offset-1,length[1],2))
def paired_jump21(old,new,rgb,orient,pos,start,span,offset):
    # One edge/control transition sampled at identical orientation and segment.
    if orient=="vertical":
        win=Window(pos+offset-1,start,2,span)
    else:
        win=Window(start,pos+offset-1,span,2)
    a=old.read(1,window=win);b=new.read(1,window=win);c=rgb.read([1,2,3],window=win)
    valid=v21(c)
    valid=valid[0]&valid[1] if orient=="horizontal" else valid[:,0]&valid[:,1]
    if orient=="horizontal":
        v0=valid&(a[0]!=65535)&(a[1]!=65535)&(b[0]!=65535)&(b[1]!=65535)
        v0 &= (a[0]<=10000)&(a[1]<=10000)&(b[0]<=10000)&(b[1]<=10000)
        aa=np.abs(a[0].astype(np.float32)-a[1].astype(np.float32))/10000.
        bb=np.abs(b[0].astype(np.float32)-b[1].astype(np.float32))/10000.
        lum=(.2126*c[0]+.7152*c[1]+.0722*c[2])/255.
        cc=np.abs(lum[0]-lum[1])
    else:
        v0=valid&(a[:,0]!=65535)&(a[:,1]!=65535)&(b[:,0]!=65535)&(b[:,1]!=65535)
        v0 &= (a[:,0]<=10000)&(a[:,1]<=10000)&(b[:,0]<=10000)&(b[:,1]<=10000)
        aa=np.abs(a[:,0].astype(np.float32)-a[:,1].astype(np.float32))/10000.
        bb=np.abs(b[:,0].astype(np.float32)-b[:,1].astype(np.float32))/10000.
        lum=(.2126*c[0].astype(np.float32)+.7152*c[1].astype(np.float32)+.0722*c[2].astype(np.float32))/255.
        cc=np.abs(lum[:,0]-lum[:,1])
    if v0.sum()<32:return None
    return (float(aa[v0].mean()),float(bb[v0].mean()),float(cc[v0].mean()),int(v0.sum()))
rows21=[]
with rasterio.open(OLD21) as old,rasterio.open(NEW21) as new,rasterio.open(RGB21) as rgb:
    assert (old.width,old.height,old.transform,old.crs)==\
           (new.width,new.height,new.transform,new.crs)
    assert (rgb.width,rgb.height,rgb.transform,rgb.crs)==\
           (new.width,new.height,new.transform,new.crs)
    for orient in ("vertical","horizontal"):
        limit=new.width if orient=="vertical" else new.height
        cross=new.height if orient=="vertical" else new.width
        boundaries=np.arange(GRID21,limit-GRID21,GRID21)
        assert len(boundaries)>0
        for i in range(PAIRS_PER_DIRECTION21):
            pos=int(RNG21.choice(boundaries))
            start=int(RNG21.integers(0,max(1,cross-STRIP21)))
            span=min(STRIP21,cross-start)
            # Edge transition is at position-1 -> position.
            # Control transition is 256 pixels inside NEXT output tile.
            if pos+GRID21//2>=limit:continue
            at_edge=paired_jump21(old,new,rgb,orient,pos,start,span,0)
            control=paired_jump21(old,new,rgb,orient,pos,start,span,GRID21//2)
            if at_edge is None or control is None:continue
            rows21.append(dict(orientation=orient,boundary_pos=pos,segment_start=start,
                n_edge=at_edge[3],n_control=control[3],
                old_boundary=at_edge[0],new_boundary=at_edge[1],rgb_boundary=at_edge[2],
                old_control=control[0],new_control=control[1],rgb_control=control[2]))
    # Illustrate spatial patterns explicitly with visible 512-pixel boundary positions.
    for j,(xf,yf) in enumerate([(.35,.5),(.6,.5),(.5,.7)],1):
        side=2048
        x0=int(max(0,min(new.width-side,round(new.width*xf-side/2))))
        y0=int(max(0,min(new.height-side,round(new.height*yf-side/2))))
        win=Window(x0,y0,side,side)
        c=rgb.read([1,2,3],window=win)
        o=old.read(1,window=win)
        m=new.read(1,window=win)
        mask=v21(c)&(o!=65535)&(m!=65535)&(o<=10000)&(m<=10000)
        oo=np.where(mask,o.astype(np.float32)/10000,np.nan)
        mm=np.where(mask,m.astype(np.float32)/10000,np.nan)
        fig,ax=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
        ax[0].imshow(np.moveaxis(c,0,-1));ax[0].set_title("Native RGB")
        ax[1].imshow(oo,vmin=0,vmax=1,cmap="viridis");ax[1].set_title("Historical")
        ax[2].imshow(mm,vmin=0,vmax=1,cmap="viridis");ax[2].set_title("M1 full map")
        ax[3].imshow(mm-oo,vmin=-1,vmax=1,cmap="coolwarm");ax[3].set_title("M1 minus historical")
        # Green grid follows the actual 512px full-map output tile boundaries, not plot grid.
        for a in ax[1:]:
            for xp in range((x0//GRID21+1)*GRID21,x0+side,GRID21):
                a.axvline(xp-x0,color="lime",alpha=.5,linewidth=.75)
            for yp in range((y0//GRID21+1)*GRID21,y0+side,GRID21):
                a.axhline(yp-y0,color="lime",alpha=.5,linewidth=.75)
        for a in ax:a.axis("off")
        fig.savefig(OUT21/f"21_Luki_RGB_old_M1_512grid_{j}.png",
                    dpi=130,bbox_inches="tight")
        plt.show()
seams21=pd.DataFrame(rows21)
assert len(seams21)>20,"Too few valid edge/control samples: inspect Luki spatial coverage."
seams21.to_csv(OUT21/"21_Luki_sampled_512px_tile_seams.csv",index=False)
summary21=[]
for ori,g in [("all",seams21)]+list(seams21.groupby("orientation")):
    for name in ("old","new","rgb"):
        boundary=g[f"{name}_boundary"].mean()
        control=g[f"{name}_control"].mean()
        summary21.append(dict(orientation=ori,model=name,n_pairs=len(g),
            mean_abs_jump_at_tile_edge=float(boundary),
            mean_abs_jump_tile_interior=float(control),
            edge_to_interior_ratio=float(boundary/control) if control>0 else np.nan,
            excess_edge_jump=float(boundary-control)))
report21=pd.DataFrame(summary21)
report21.to_csv(OUT21/"21_Luki_512px_tile_seam_summary.csv",index=False)
print("512-pixel inference-tile boundary discontinuity vs within-tile control:")
display(report21.round(5))
print("OUTPUT:",OUT21)
print("A strong M1-only seam effect with no matching RGB effect implicates inference tiling.")


## 22 — Four-forest M1 full-map census, calibration and seam audit (CPU only)

**The main diagnostic for the Musanga–satellite workflow.** This section evaluates **complete final-M1 inference probability maps**, not crown-centred model outputs. For Yoko and Baego, it streams native-grid RGB, historical U-Net probability and final-M1 probability in 512-pixel windows, comparing **only the same mutually valid pixels**. It reports global mean P(Musanga), fractions above 0.5 / 0.9, histogram distributions, and 512 × 512 window brightness correlations. It additionally reports within-site **full-map polygon scores** for the existing reference crowns as a secondary diagnostic. All maps must have exactly identical native transforms: no resampling or synthetic alignment.

**Critical tiling test:** Across random 512-pixel inference-grid boundaries it contrasts the jump in M1 probability at the tile boundary with a transition in the tile interior, and computes the same diagnostics for the historical probability map and native RGB. The ratio is evidence of a *grid-locked discontinuity* rather than a model performance metric. It is possible to have **excellent annotation crown AUC and terrible inference seams**, which is precisely why both are reported. It also saves side-by-side RGB, old map, M1 map, and M1−old difference panels with the 512-pixel output-grid boundaries highlighted.

This is a **read-only CPU audit**. The default processes **all four sites** only when their M1 maps are finished. The default is `AUDIT_SITES22=["Yoko","Baego"]`, using the exact Section 18 filenames. Full pixel census can take several minutes to tens of minutes depending on Google Drive throughput. Crown discrimination over these sites is **in-sample for final M1**; original four-fold LOSO metrics remain the independent site-transfer evaluation. If seams are strong, stop before producing satellite labels; prioritize a small controlled shifted-grid / larger-halo inference experiment.

In [ ]:
# Section 22 — streamed full-mosaic comparison + native 512-pixel seam and crown audit.
# CPU-only; the final M1 maps must already exist. No inference or training here.
from pathlib import Path
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from rasterio.windows import Window,from_bounds
from rasterio.features import geometry_mask
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score
from scipy.stats import spearmanr
from IPython.display import display
from tqdm.auto import tqdm
import math, json

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT22=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO22=ROOT22/"data/Geo_Ref"
REF22=ROOT22/"KVO_work/V2/01_reference_splits"
HIST22=ROOT22/"KVO_work/KVO_04_final_multisite/probability_maps"
RUN22=ROOT22/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f"
OUT22=RUN22/"four_site_full_map_audit"
OUT22.mkdir(parents=True,exist_ok=True)
SITES22={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")
}
AUDIT_SITES22=["Yoko","Baego"]  # completed sites only
FULL_CENSUS22=True
CENSUS_SAMPLE22=200        # used only when FULL_CENSUS22=False
WINDOW22=512
TILE_GRID22=512
SEAM_SAMPLES22=160        # samples for each seam orientation and site
STRIP22=512
PREVIEW22=1024
assert set(AUDIT_SITES22).issubset(SITES22) and len(AUDIT_SITES22)>0

def valid22(rgb):
    return ~(np.all(rgb==0,axis=0)|np.all(rgb==255,axis=0)|
             ((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)))
def valid_probability22(arr):
    return (arr!=65535)&(arr<=10000)

rng22=np.random.default_rng(20261013)
rows22=[];window_rows22=[];seams22=[];crown_rows22=[]
for site22 in AUDIT_SITES22:
    rgb_path=GEO22/SITES22[site22][0]
    old_path=HIST22/f"{site22}_musanga_probability_uint16_FULL.tif"
    # Exact previously supplied Section-18 final map locations.
    NEW_MAPS22={
        "Yoko": ROOT22/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f/site_wall_to_wall/Yoko/Yoko_M1_probability_uint16_NATIVE.tif",
        "Baego": ROOT22/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f/site_wall_to_wall/Baego/Baego_M1_probability_uint16_NATIVE.tif",
    }
    new_path=NEW_MAPS22[site22] if site22 in NEW_MAPS22 else RUN22/"site_wall_to_wall"/site22/f"{site22}_M1_probability_uint16_NATIVE.tif"
    if not new_path.exists():
        parent=new_path.parent
        partial=parent/f"{site22}_M1_probability_uint16_NATIVE.partial.tif"
        print(f"\n{site22}: exact map file not visible in current Colab runtime: {new_path}")
        print("Drive mounted:",Path("/content/drive/MyDrive").is_dir())
        print("Folder exists:",parent.is_dir(),"partial exists:",partial.exists())
        if parent.is_dir():
            print("Destination files:",[p.name for p in parent.iterdir()][:30])
        continue
    for file in [rgb_path,old_path]:
        assert file.exists(),f"{site22} source missing: {file}"
    site_dir=OUT22/site22
    site_dir.mkdir(parents=True,exist_ok=True)
    print(f"\n=== {site22}: exact native-grid FULL INFERENCE evaluation ===",flush=True)
    with rasterio.open(rgb_path) as rgb,rasterio.open(old_path) as old,rasterio.open(new_path) as new:
        assert (rgb.width,rgb.height,rgb.crs,rgb.transform)==(old.width,old.height,old.crs,old.transform), \
            f"{site22}: source and old map differ in grid; refusing comparison"
        assert (rgb.width,rgb.height,rgb.crs,rgb.transform)==(new.width,new.height,new.crs,new.transform), \
            f"{site22}: source and final M1 map differ in grid; refusing comparison"
        assert rgb.count>=3 and old.count==1 and new.count==1
        assert old.dtypes[0]==new.dtypes[0]=="uint16"
        counts=dict(joint=0,rgb_valid=0,only_old=0,only_new=0,
                    old_sum=0.,new_sum=0.,difference_sq=0.,
                    old_cover05=0,new_cover05=0,old_cover09=0,new_cover09=0)
        h0=np.zeros(50,dtype=np.int64);h1=np.zeros(50,dtype=np.int64)
        ncols=math.ceil(rgb.width/WINDOW22);nrows=math.ceil(rgb.height/WINDOW22)
        jobs=[(j,i) for i in range(nrows) for j in range(ncols)]
        if FULL_CENSUS22:
            chosen=jobs
        else:
            ix=np.unique(np.linspace(0,len(jobs)-1,min(CENSUS_SAMPLE22,len(jobs))).round().astype(int))
            chosen=[jobs[int(i)] for i in ix]
        for ix,iy in tqdm(chosen,desc=f"{site22}: native paired windows",leave=True):
            x=ix*WINDOW22;y=iy*WINDOW22
            w=min(WINDOW22,rgb.width-x);h=min(WINDOW22,rgb.height-y)
            window=Window(x,y,w,h)
            colors=rgb.read([1,2,3],window=window)
            o=old.read(1,window=window);m=new.read(1,window=window)
            v=valid22(colors);ov=v&valid_probability22(o);mv=v&valid_probability22(m)
            joint=ov&mv
            counts["rgb_valid"]+=int(v.sum())
            counts["only_old"]+=int((ov&~mv).sum())
            counts["only_new"]+=int((mv&~ov).sum())
            z=int(joint.sum())
            if z<32:continue
            a=o[joint].astype(np.float32)/10000.
            b=m[joint].astype(np.float32)/10000.
            d=b-a
            counts["joint"]+=z
            counts["old_sum"]+=float(a.sum(dtype=np.float64))
            counts["new_sum"]+=float(b.sum(dtype=np.float64))
            counts["difference_sq"]+=float(np.square(d).sum(dtype=np.float64))
            counts["old_cover05"]+=int((a>=.5).sum())
            counts["new_cover05"]+=int((b>=.5).sum())
            counts["old_cover09"]+=int((a>=.9).sum())
            counts["new_cover09"]+=int((b>=.9).sum())
            h0+=np.histogram(a,bins=50,range=(0,1))[0]
            h1+=np.histogram(b,bins=50,range=(0,1))[0]
            lum=(.2126*colors[0].astype(np.float32)+
                 .7152*colors[1].astype(np.float32)+
                 .0722*colors[2].astype(np.float32))/255.
            window_rows22.append(dict(site=site22,ix=ix,iy=iy,n_valid=z,
                median_brightness=float(np.median(lum[joint])),
                old_mean_p=float(a.mean()),M1_mean_p=float(b.mean()),
                old_cover05=float((a>=.5).mean()),M1_cover05=float((b>=.5).mean())))
        assert counts["joint"]>0,f"{site22}: zero jointly valid pixels"
        z=counts["joint"]
        site_windows=pd.DataFrame([r for r in window_rows22 if r["site"]==site22])
        rh0=float(spearmanr(site_windows.median_brightness,site_windows.old_mean_p).statistic)
        rh1=float(spearmanr(site_windows.median_brightness,site_windows.M1_mean_p).statistic)
        rows22.append(dict(site=site22,scope="full" if FULL_CENSUS22 else "sample",
            RGB_valid=counts["rgb_valid"],joint_valid=z,
            old_only_valid=counts["only_old"],M1_only_valid=counts["only_new"],
            old_mean_probability=counts["old_sum"]/z,M1_mean_probability=counts["new_sum"]/z,
            mean_shift=(counts["new_sum"]-counts["old_sum"])/z,
            difference_RMSE=math.sqrt(counts["difference_sq"]/z),
            old_fraction_Pge05=counts["old_cover05"]/z,
            M1_fraction_Pge05=counts["new_cover05"]/z,
            change_Pge05=(counts["new_cover05"]-counts["old_cover05"])/z,
            old_fraction_Pge09=counts["old_cover09"]/z,
            M1_fraction_Pge09=counts["new_cover09"]/z,
            brightness_old_rho=rh0,brightness_M1_rho=rh1,
            measured_windows=len(site_windows)))
        pd.DataFrame({"midpoint":(np.arange(50)+.5)/50,
                      "original_fraction":h0/z,"M1_fraction":h1/z}).to_csv(
                      site_dir/"22_probability_histogram.csv",index=False)

        # *Independent* seam diagnostic: random tile-edge pairs versus 256px interior.
        for orientation in ("vertical","horizontal"):
            length=rgb.width if orientation=="vertical" else rgb.height
            cross=rgb.height if orientation=="vertical" else rgb.width
            boundaries=np.arange(TILE_GRID22,length-TILE_GRID22,TILE_GRID22)
            if not len(boundaries):continue
            def jump(position,start,orientation):
                win=(Window(position-1,start,2,STRIP22) if orientation=="vertical"
                     else Window(start,position-1,STRIP22,2))
                cc=rgb.read([1,2,3],window=win)
                oo=old.read(1,window=win)
                mm=new.read(1,window=win)
                valid=valid22(cc)&valid_probability22(oo)&valid_probability22(mm)
                if orientation=="vertical":
                    pair=valid[:,0]&valid[:,1]
                    ao=np.abs(oo[:,1].astype(np.float32)-oo[:,0].astype(np.float32))/10000.
                    am=np.abs(mm[:,1].astype(np.float32)-mm[:,0].astype(np.float32))/10000.
                else:
                    pair=valid[0,:]&valid[1,:]
                    ao=np.abs(oo[1,:].astype(np.float32)-oo[0,:].astype(np.float32))/10000.
                    am=np.abs(mm[1,:].astype(np.float32)-mm[0,:].astype(np.float32))/10000.
                lum=(.2126*cc[0].astype(np.float32)+.7152*cc[1].astype(np.float32)+
                     .0722*cc[2].astype(np.float32))/255.
                al=np.abs(lum[:,1]-lum[:,0]) if orientation=="vertical" else np.abs(lum[1,:]-lum[0,:])
                if pair.sum()<32:return None
                return float(ao[pair].mean()),float(am[pair].mean()),float(al[pair].mean()),int(pair.sum())
            for k in range(SEAM_SAMPLES22):
                edge=int(rng22.choice(boundaries))
                start=int(rng22.integers(0,cross-STRIP22))
                e=jump(edge,start,orientation)
                control=jump(edge+TILE_GRID22//2,start,orientation)
                if e is None or control is None:continue
                seams22.append(dict(site=site22,orientation=orientation,edge=edge,
                    old_edge=e[0],M1_edge=e[1],RGB_edge=e[2],
                    old_interior=control[0],M1_interior=control[1],
                    RGB_interior=control[2],n_edge=e[3],n_interior=control[3]))

        # Crown scores from WHOLE map; final M1 is in-sample for these 4 training sites.
        annotations=gpd.read_file(GEO22/"Annotations"/SITES22[site22][1])
        if annotations.crs!=new.crs:annotations=annotations.to_crs(new.crs)
        annotations=annotations.loc[
            annotations.geometry.notna()&(~annotations.geometry.is_empty)&
            annotations.geometry.is_valid].copy()
        cls=pd.to_numeric(annotations.Class_ID,errors="coerce")
        annotations=annotations.loc[cls.notna()].copy()
        annotations["label"]=(cls.loc[annotations.index].astype(int)==1).astype(int)
        annotations=annotations.loc[(annotations.geometry.area>=5)].copy()
        def crown_score(src,geom):
            win=from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
            x=max(0,int(win.col_off));y=max(0,int(win.row_off))
            xe=min(src.width,int(math.ceil(win.col_off+win.width)))
            ye=min(src.height,int(math.ceil(win.row_off+win.height)))
            if xe<=x or ye<=y or xe-x>2048 or ye-y>2048:return np.nan
            boxwin=Window(x,y,xe-x,ye-y)
            inside=geometry_mask([geom],out_shape=(ye-y,xe-x),
                                 transform=src.window_transform(boxwin),invert=True)
            arr=src.read(1,window=boxwin)
            v=inside&valid_probability22(arr)
            if v.sum()<10 or v.sum()/max(1,inside.sum())<.9:return np.nan
            return float(np.mean(arr[v].astype(np.float32))/10000)
        species=[]
        for _,rec in tqdm(annotations.iterrows(),total=len(annotations),
                          desc=f"{site22} full-map reference crowns",leave=False):
            species.append(dict(site=site22,label=int(rec.label),
                old_p=crown_score(old,rec.geometry),M1_p=crown_score(new,rec.geometry)))
        crown_rows22.extend(species)
        print(f"{site22}: map statistics and crown scores complete.",flush=True)
        pd.DataFrame(rows22).to_csv(OUT22/"22_four_site_full_map_summary.csv",index=False)
        pd.DataFrame(window_rows22).to_csv(OUT22/"22_four_site_brightness_windows.csv",index=False)
        pd.DataFrame(seams22).to_csv(OUT22/"22_four_site_grid_seam_samples.csv",index=False)
        pd.DataFrame(crown_rows22).to_csv(OUT22/"22_four_site_full_map_crown_scores.csv",index=False)

        # 3 small native resolution inspection panels per site
        for j,(fx,fy) in enumerate([(.25,.5),(.5,.5),(.75,.5)],1):
            x0=max(0,min(rgb.width-PREVIEW22,int(rgb.width*fx-PREVIEW22/2)))
            y0=max(0,min(rgb.height-PREVIEW22,int(rgb.height*fy-PREVIEW22/2)))
            view=Window(x0,y0,PREVIEW22,PREVIEW22)
            colors=rgb.read([1,2,3],window=view)
            p0=old.read(1,window=view);p1=new.read(1,window=view)
            mask=valid22(colors)&valid_probability22(p0)&valid_probability22(p1)
            aa=np.where(mask,p0.astype(np.float32)/10000.,np.nan)
            bb=np.where(mask,p1.astype(np.float32)/10000.,np.nan)
            import matplotlib.pyplot as plt
            fig,axs=plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
            axs[0].imshow(np.moveaxis(colors,0,-1));axs[0].set_title("RGB")
            axs[1].imshow(aa,cmap="viridis",vmin=0,vmax=1);axs[1].set_title("Historical")
            axs[2].imshow(bb,cmap="viridis",vmin=0,vmax=1);axs[2].set_title("Final M1")
            axs[3].imshow(bb-aa,cmap="coolwarm",vmin=-1,vmax=1);axs[3].set_title("M1 − old")
            for ax in axs[1:]:
                for tx in range((x0//512+1)*512,x0+PREVIEW22,512):
                    ax.axvline(tx-x0,color="lime",lw=.8,alpha=.5)
                for ty in range((y0//512+1)*512,y0+PREVIEW22,512):
                    ax.axhline(ty-y0,color="lime",lw=.8,alpha=.5)
            for ax in axs:ax.axis("off")
            fig.savefig(site_dir/f"22_RGB_old_M1_difference_tiles_{j}.png",dpi=130)
            plt.show()
            plt.close(fig)

if not rows22:
    raise RuntimeError("No completed final M1 site maps found. Run Section 18 first.")
summary22=pd.DataFrame(rows22)
print("\nFULL MOSAIC comparisons (all four sites when available):")
display(summary22.round(4))
seams_df22=pd.DataFrame(seams22)
seam_summary22=[]
if not seams_df22.empty:
    for (site,orient),g in seams_df22.groupby(["site","orientation"]):
        for model in ["old","M1","RGB"]:
            edge=float(g[f"{model}_edge"].mean())
            inside=float(g[f"{model}_interior"].mean())
            seam_summary22.append(dict(site=site,orientation=orient,model=model,
                n_pairs=len(g),edge_jump=edge,interior_jump=inside,
                edge_to_interior_ratio=edge/inside if inside else np.nan,
                excess_edge_jump=edge-inside))
seam_summary_df22=pd.DataFrame(seam_summary22)
seam_summary_df22.to_csv(OUT22/"22_four_site_grid_seam_summary.csv",index=False)
print("\nGrid-aligned 512px edge discontinuities (ratio >1 indicates extra jumps at edges):")
display(seam_summary_df22.round(4))
crowns_df22=pd.DataFrame(crown_rows22)
metrics22=[]
for site,g in crowns_df22.groupby("site"):
    for model,col in [("Historical","old_p"),("Final M1","M1_p")]:
        valid=g.loc[g[col].notna()]
        if len(valid)<10 or valid.label.nunique()!=2:continue
        y=valid.label.to_numpy(int);p=valid[col].to_numpy(float);pred=(p>=.5).astype(int)
        metrics22.append(dict(site=site,model=model,n=len(valid),AUC=float(roc_auc_score(y,p)),
            AP=float(average_precision_score(y,p)),
            precision=float(precision_score(y,pred,zero_division=0)),
            recall=float(recall_score(y,pred,zero_division=0)),
            F1=float(f1_score(y,pred,zero_division=0))))
crown_metrics22=pd.DataFrame(metrics22)
crown_metrics22.to_csv(OUT22/"22_full_map_reference_crown_metrics.csv",index=False)
print("\nSECONDARY: full-map crown metrics (final M1 sites in training, not external tests):")
display(crown_metrics22.round(4))
print("AUDIT RESULTS:",OUT22,flush=True)
print("Interpret satellite suitability from full-map output distributions AND 512px seam metrics, NOT crown AUC alone.")


## 23 — Fix the grid artefact: controlled inference geometry test (GPU, small areas only)

**Run Section 23 only** (after installing `segmentation-models-pytorch` if needed), using the existing final M1 checkpoint. Section 23 predicts a 2048 × 2048 native-resolution area in **Yoko and Luki** with four strategies: **A** current 640px input / 512px output / 64px halo; **B** training-scale 256px input / 128px output / 64px halo; **C** wider-context 768px input / 512px output / 128px halo; **D** two 640/512 grids offset by 256px in *both* directions, with overlapping predictions averaged. Same frozen checkpoint, RGB and normalization; **no retraining, no full forest inference, no existing maps overwritten**.

The script uses mosaic-global 512-aligned 2048px test windows (chosen from mostly valid-RGB regions), reproduces the old map A via an exact raster comparison, evaluates each configuration's own grid discontinuities against within-tile controls, measures output disagreements and outputs matched visual PNGs + summary CSVs. It uses native RGB only and excludes black/white/pure-blue source pixels, just like Sections 16/18.

**Decision rule:** if B or C reduces grid jumps while preserving predicted crown pattern, move toward that production geometry. If D is best, consider overlapping-grid averaging. If none works, investigate input normalization, model padding and training supervision before another full-site export. Testing only selected areas means it cannot by itself validate site-wide probability calibration.

In [ ]:
# Section 23: small-area inference geometry experiments, using EXACT final M1 weights.
# Run Section 12 first. No training, no overwrites of completed full-map predictions.
from pathlib import Path
from rasterio.windows import Window
import numpy as np, pandas as pd, rasterio, torch, math, json
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO=ROOT/"data/Geo_Ref"
SITES={"Yoko":("Yoko_kvo_georef.tif",)}
FINAL_RUN_ID04="337073e7b74f"
FINAL_RUN04=ROOT/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1"/FINAL_RUN_ID04
FINAL_CKPT04=FINAL_RUN04/"M1_native_photometric/best_model.pt"
assert torch.cuda.is_available(),"Section 23 requires a GPU runtime (L4 recommended)."
assert FINAL_CKPT04.is_file(),f"Missing saved checkpoint: {FINAL_CKPT04}"
print("Reusing final M1 checkpoint:",FINAL_CKPT04,flush=True)
TEST_SITES23=["Yoko","Luki"]
ROI_SIZE23=2048               # divisible by 512; all comparisons at exact native pixels
BATCH23=4
OUT23=FINAL_RUN04/"inference_geometry_diagnostic"
OUT23.mkdir(parents=True,exist_ok=True)
ck23=torch.load(FINAL_CKPT04,map_location="cpu",weights_only=False)
assert ck23.get("run_id")==FINAL_RUN_ID04 and ck23.get("variant")=="M1_native_photometric"
net23=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
net23.load_state_dict(ck23["model"],strict=True)
net23=net23.cuda().eval()
mean23=torch.tensor([.485,.456,.406],device="cuda").view(1,3,1,1)
std23=torch.tensor([.229,.224,.225],device="cuda").view(1,3,1,1)
def rgb_valid23(c):
    return ~(np.all(c==0,axis=0)|np.all(c==255,axis=0)|
             ((c[0]==0)&(c[1]==0)&(c[2]==255)))
def raster_site23(site):
    return GEO/"Luki.tif" if site=="Luki" else GEO/SITES[site][0]
def oldmap_site23(site):
    return (FINAL_RUN04/"luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
            if site=="Luki" else FINAL_RUN04/"site_wall_to_wall"/site/
            f"{site}_M1_probability_uint16_NATIVE.tif")
def select_roi23(src):
    # Choose the most valid central-ish area among several globally 512-aligned candidates.
    side=ROI_SIZE23
    assert src.width>=side and src.height>=side
    proposals=[]
    for fy in (.3,.5,.7):
        for fx in (.3,.5,.7):
            x=int(np.clip(round((src.width*fx-side/2)/512)*512,0,
                          ((src.width-side)//512)*512))
            y=int(np.clip(round((src.height*fy-side/2)/512)*512,0,
                          ((src.height-side)//512)*512))
            win=Window(x,y,side,side)
            # nearest out_shape limits read I/O; major valid regions still detected
            c=src.read([1,2,3],window=win,out_shape=(3,256,256),
                       resampling=rasterio.enums.Resampling.nearest)
            proposals.append((float(rgb_valid23(c).mean()),x,y))
    frac,x0,y0=max(proposals)
    assert frac>.50,f"No adequately valid native region; top fraction={frac:.3f}"
    return x0,y0,frac

def infer_geometry23(src,x0,y0,step,halo,shiftx=0,shifty=0):
    # Predicted ROI shape 2048x2048; output-window origins vary by shift.
    # Exact mosaic-native coordinates retained regardless of ROI positioning.
    side=ROI_SIZE23;size=step+2*halo
    assert size%32==0 and step%32==0
    pred=np.zeros((side,side),np.float32)
    visited=np.zeros((side,side),np.uint8)
    # Each grid is a complete tiling; negative origins are permitted via boundless read.
    xs=list(range(x0+shiftx-step,x0+side,step))
    ys=list(range(y0+shifty-step,y0+side,step))
    tasks=[]
    for py in ys:
        for px in xs:
            xa=max(px,x0);xb=min(px+step,x0+side)
            ya=max(py,y0);yb=min(py+step,y0+side)
            if xa<xb and ya<yb:tasks.append((px,py,xa,xb,ya,yb))
    for begin in tqdm(range(0,len(tasks),BATCH23),desc=f"step{step}_halo{halo}_shift{shiftx}",leave=False):
        pack=tasks[begin:begin+BATCH23]
        crops=[]
        for px,py,*_ in pack:
            crops.append(src.read([1,2,3],
                window=Window(px-halo,py-halo,size,size),
                boundless=True,fill_value=0))
        ten=torch.from_numpy(np.stack(crops)).to("cuda",non_blocking=True).float()/255.
        ten=(ten-mean23)/std23
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
            out=torch.sigmoid(net23(ten)).float().cpu().numpy()[:,0]
        for i,(px,py,xa,xb,ya,yb) in enumerate(pack):
            patch=out[i,halo+(ya-py):halo+(yb-py),
                           halo+(xa-px):halo+(xb-px)]
            pred[ya-y0:yb-y0,xa-x0:xb-x0]=patch
            visited[ya-y0:yb-y0,xa-x0:xb-x0]+=1
    assert np.all(visited==1),f"Uncovered/overlapping ROI pixels: {np.unique(visited,return_counts=True)}"
    return pred

def seam23(p,valid,step,shift=0):
    # Aggregate native adjacent-pixel gradients at method's own tile seams,
    # with control transitions at equivalent positions step//2 inside tiles.
    edge=[];interior=[]
    h,w=p.shape
    for ori in ("x","y"):
        n=w if ori=="x" else h
        for pos in range(step+shift,n,step):
            if pos<=0 or pos+step//2>=n:continue
            if ori=="x":
                v=valid[:,pos-1]&valid[:,pos]
                vc=valid[:,pos+step//2-1]&valid[:,pos+step//2]
                d=np.abs(p[:,pos]-p[:,pos-1])
                dc=np.abs(p[:,pos+step//2]-p[:,pos+step//2-1])
            else:
                v=valid[pos-1,:]&valid[pos,:]
                vc=valid[pos+step//2-1,:]&valid[pos+step//2,:]
                d=np.abs(p[pos,:]-p[pos-1,:])
                dc=np.abs(p[pos+step//2,:]-p[pos+step//2-1,:])
            if v.sum()>=32:edge.append(d[v])
            if vc.sum()>=32:interior.append(dc[vc])
    if not edge or not interior:return np.nan,np.nan,np.nan
    e=float(np.concatenate(edge).mean());q=float(np.concatenate(interior).mean())
    return e,q,e/q if q else np.nan

report23=[]
for site in TEST_SITES23:
    src_path=raster_site23(site)
    previous=oldmap_site23(site)
    assert previous.exists(),f"Missing previous full map for {site}: {previous}"
    outdir=OUT23/site;outdir.mkdir(parents=True,exist_ok=True)
    with rasterio.open(src_path) as src,rasterio.open(previous) as reference:
        assert (src.width,src.height,src.transform,src.crs)==(
                reference.width,reference.height,reference.transform,reference.crs)
        x0,y0,rough=select_roi23(src)
        print(f"\n{site} region: x={x0},y={y0}, size={ROI_SIZE23}, coarse valid={rough:.2%}",flush=True)
        win=Window(x0,y0,ROI_SIZE23,ROI_SIZE23)
        colors=src.read([1,2,3],window=win)
        valid=rgb_valid23(colors)
        reference_arr=reference.read(1,window=win)
        known=(reference_arr<=10000)&(reference_arr!=65535)&valid
        base=np.where(known,reference_arr.astype(np.float32)/10000.,np.nan)
        # The four test strategies. D averages four 2D half-step grid offsets.
        a=infer_geometry23(src,x0,y0,512,64)
        b=infer_geometry23(src,x0,y0,128,64)
        c=infer_geometry23(src,x0,y0,512,128)
        d=(a+infer_geometry23(src,x0,y0,512,64,256,0)+
           infer_geometry23(src,x0,y0,512,64,0,256)+
           infer_geometry23(src,x0,y0,512,64,256,256))/4.
        variants={"A_existing_640_512":a,"B_training_scale_256_128":b,
                  "C_halo128_768_512":c,"D_4shift_grid_average":d}
        # Test A must reproduce the original, byte-quantized prediction at exactly the same pixels.
        difference=np.abs(a[known]-base[known])
        print(f"{site}: current-method replication: mean |A - existing|={difference.mean():.6f}; "
              f"p99={np.quantile(difference,.99):.6f}",flush=True)
        assert float(np.mean(difference))<.01, \
            "Existing map and exact current-method re-inference differ. Resolve before comparing strategies."
        for name,p in variants.items():
            p=np.where(valid,p,np.nan)
            edge,inner,ratio=seam23(np.nan_to_num(p),valid,128 if name.startswith("B_") else 512)
            # Evaluate both model-original 512 boundaries and strategy-specific seams.
            e512,i512,r512=seam23(np.nan_to_num(p),valid,512)
            report23.append(dict(site=site,strategy=name,roi_x=x0,roi_y=y0,valid_pixels=int(valid.sum()),
                mean_p=float(np.nanmean(p)),fraction_ge_05=float(np.nanmean(p>=.5)),
                seam_jump=edge,interior_jump=inner,seam_ratio=ratio,
                seam512_edge=e512,seam512_interior=i512,seam512_ratio=r512,
                mean_abs_difference_from_A=float(np.nanmean(np.abs(p-a))),
                mean_abs_difference_from_existing=float(np.nanmean(np.abs(p-base)))))
        fig,axs=plt.subplots(2,3,figsize=(15,10),constrained_layout=True)
        axs=axs.ravel()
        axs[0].imshow(np.moveaxis(colors,0,-1));axs[0].set_title(f"{site} native RGB")
        axs[1].imshow(base,vmin=0,vmax=1,cmap="viridis");axs[1].set_title("Existing full M1 (512-grid)")
        for ax,(name,p) in zip(axs[2:],variants.items()):
            ax.imshow(np.where(valid,p,np.nan),vmin=0,vmax=1,cmap="viridis")
            ax.set_title(name)
        for ax in axs:ax.axis("off")
        fig.savefig(outdir/"23_inference_geometry_same_native_area.png",dpi=120)
        plt.show();plt.close(fig)
        # Save exact-grid raw comparison for ArcGIS inspection; never alter original full maps.
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="float32",nodata=-9999.,
                       tiled=True,blockxsize=512,blockysize=512,compress="deflate",
                       transform=src.window_transform(win))
        for name,p in variants.items():
            with rasterio.open(outdir/f"23_{site}_{name}.tif","w",**profile) as dst:
                dst.write(np.where(valid,p,-9999.).astype(np.float32),1)
        (outdir/"23_ROI_metadata.json").write_text(json.dumps(dict(
            site=site,x=x0,y=y0,size=ROI_SIZE23,checkpoint=str(FINAL_CKPT04),
            existing_map=str(previous),strategies=list(variants)),indent=2))
reportdf23=pd.DataFrame(report23)
reportdf23.to_csv(OUT23/"23_inference_geometry_comparison.csv",index=False)
display(reportdf23.round(4))
print("Saved experiment results:",OUT23)


## 24 — Seam-free candidate: overlapping inference with feather blending (GPU, two small ROIs)

Section 23 revealed that more context reduces edge jumps but does not remove them, and four hard-edged shifted-grid maps can simply **move the seam**. This section tests a production-oriented strategy: **512px output blocks placed every 256px**, with a **smooth two-dimensional weight window** and normalized accumulation of overlapping predictions, using the **same original final-M1 weights**. The overlap and weighting remove hard transitions between individual tile predictions.

Compare **E**: 768px input / 512px output / 128px halo / 256px stride, and **F**: 1024px input / 512px output / 256px halo / 256px stride. Both use an explicit cosine taper and normalize sum of weighted probabilities. Compute 256px and 512px seam statistics **at every phase**, along with mean probability, P≥0.5 area fraction, and direct comparison to saved old full-map probabilities. Predictions remain on the unmodified native raster grids and are saved as small GeoTIFFs for ArcGIS. **Nothing is trained or overwritten.**

**Run Section 24 alone** on GPU in a new Colab runtime (the cell mounts Drive and loads the checkpoint independently; install `segmentation-models-pytorch` if necessary). Existing Section 23 outputs are not needed. The results answer whether overlapping blended inference can repair the grid artefacts; they do **not** prove unbiased Musanga cover, which must be assessed separately.

In [ ]:
# Section 24: GPU native-grid overlapping inference + raised-cosine feathering.
# Standalone; no training and no changes to old maps.
from pathlib import Path
import math,json
import numpy as np,pandas as pd,rasterio,torch,matplotlib.pyplot as plt
from rasterio.windows import Window
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT24=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
RUN24=ROOT24/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f"
CKPT24=RUN24/"M1_native_photometric/best_model.pt"
OUT24=RUN24/"inference_geometry_diagnostic"/"feather_blending"
OUT24.mkdir(parents=True,exist_ok=True)
assert CKPT24.is_file() and torch.cuda.is_available(),"Need stored checkpoint and GPU runtime."
NET24=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
ck24=torch.load(CKPT24,map_location="cpu",weights_only=False)
assert ck24.get("run_id")=="337073e7b74f" and ck24.get("variant")=="M1_native_photometric"
NET24.load_state_dict(ck24["model"],strict=True)
NET24=NET24.cuda().eval()
mu24=torch.tensor([.485,.456,.406],device="cuda").reshape(1,3,1,1)
sd24=torch.tensor([.229,.224,.225],device="cuda").reshape(1,3,1,1)
SITES24=["Yoko","Luki"]
ROI24=2048
OUTPUT24=512
STRIDE24=256
BATCH24=2
def valid24(c):
    return ~(np.all(c==0,axis=0)|np.all(c==255,axis=0)|
             ((c[0]==0)&(c[1]==0)&(c[2]==255)))
def loc24(site):
    if site=="Luki":
        return ROOT24/"data/Geo_Ref/Luki.tif",RUN24/"luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
    return ROOT24/"data/Geo_Ref/Yoko_kvo_georef.tif",RUN24/"site_wall_to_wall/Yoko/Yoko_M1_probability_uint16_NATIVE.tif"
def best_roi24(src):
    choices=[]
    for fy in (.3,.5,.7):
        for fx in (.3,.5,.7):
            x=int(np.clip(round((src.width*fx-ROI24/2)/512)*512,0,((src.width-ROI24)//512)*512))
            y=int(np.clip(round((src.height*fy-ROI24/2)/512)*512,0,((src.height-ROI24)//512)*512))
            c=src.read([1,2,3],window=Window(x,y,ROI24,ROI24),
                       out_shape=(3,256,256),resampling=rasterio.enums.Resampling.nearest)
            choices.append((float(valid24(c).mean()),x,y))
    frac,x,y=max(choices)
    assert frac>.5,"Test ROI has insufficient valid RGB"
    return x,y,frac
def weight24():
    # Positive (nonzero) everywhere, smooth taper near each of the four edges.
    # Raised cosine reaches 1 in the central 256 pixels; 0.025 at the exterior edge.
    distance=np.minimum(np.arange(OUTPUT24),OUTPUT24-1-np.arange(OUTPUT24)).astype(np.float32)
    t=np.minimum(1.,distance/(OUTPUT24/4))
    taper=.025+.975*.5*(1.-np.cos(np.pi*t))
    return (taper[:,None]*taper[None,:]).astype(np.float32)
W24=weight24()
def feather24(src,x0,y0,halo):
    input_size=OUTPUT24+2*halo
    assert input_size%32==0
    sums=np.zeros((ROI24,ROI24),np.float32)
    weights=np.zeros((ROI24,ROI24),np.float32)
    starts_x=range(x0-OUTPUT24+STRIDE24,x0+ROI24,STRIDE24)
    starts_y=range(y0-OUTPUT24+STRIDE24,y0+ROI24,STRIDE24)
    tasks=[]
    for py in starts_y:
        for px in starts_x:
            xa=max(x0,px);xb=min(x0+ROI24,px+OUTPUT24)
            ya=max(y0,py);yb=min(y0+ROI24,py+OUTPUT24)
            if xa<xb and ya<yb:tasks.append((px,py,xa,xb,ya,yb))
    for k in tqdm(range(0,len(tasks),BATCH24),desc=f"overlap512_stride256_halo{halo}",leave=False):
        pack=tasks[k:k+BATCH24]
        raw=np.stack([src.read([1,2,3],window=Window(px-halo,py-halo,input_size,input_size),
                               boundless=True,fill_value=0) for px,py,*_ in pack])
        inputs=torch.from_numpy(raw).float().cuda()/255.
        inputs=(inputs-mu24)/sd24
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
            outputs=torch.sigmoid(NET24(inputs)).float().cpu().numpy()[:,0]
        for j,(px,py,xa,xb,ya,yb) in enumerate(pack):
            sx=slice(xa-px,xb-px);sy=slice(ya-py,yb-py)
            dx=slice(xa-x0,xb-x0);dy=slice(ya-y0,yb-y0)
            prediction=outputs[j,halo+sy.start:halo+sy.stop,halo+sx.start:halo+sx.stop]
            weight=W24[sy,sx]
            sums[dy,dx]+=prediction*weight
            weights[dy,dx]+=weight
    assert np.all(weights>0),f"Uncovered output pixels: {np.sum(weights<=0)}"
    return sums/weights,weights
def seam24(p,v,period,phase=0):
    # Mean native per-pixel differences at a given regular grid phase vs
    # an offset interior comparison. Both directions, all valid paired pixels.
    boundary=[];control=[]
    for orient in ("x","y"):
        dim=p.shape[1] if orient=="x" else p.shape[0]
        for i in range(period+phase,dim,period):
            j=i+period//2
            if j>=dim:continue
            if orient=="x":
                a=v[:,i-1]&v[:,i];b=v[:,j-1]&v[:,j]
                if a.any():boundary.append(np.abs(p[:,i]-p[:,i-1])[a])
                if b.any():control.append(np.abs(p[:,j]-p[:,j-1])[b])
            else:
                a=v[i-1,:]&v[i,:];b=v[j-1,:]&v[j,:]
                if a.any():boundary.append(np.abs(p[i,:]-p[i-1,:])[a])
                if b.any():control.append(np.abs(p[j,:]-p[j-1,:])[b])
    if not boundary or not control:return np.nan,np.nan,np.nan
    e=float(np.concatenate(boundary).mean());c=float(np.concatenate(control).mean())
    return e,c,e/c if c else np.nan
rows24=[]
for site in SITES24:
    raster,previous=loc24(site)
    assert raster.is_file() and previous.is_file(),f"Missing native source or baseline for {site}"
    siteout=OUT24/site;siteout.mkdir(parents=True,exist_ok=True)
    with rasterio.open(raster) as src,rasterio.open(previous) as old:
        assert (src.width,src.height,src.crs,src.transform)==(old.width,old.height,old.crs,old.transform)
        x0,y0,frac=best_roi24(src)
        print(f"\n{site}: ROI x={x0},y={y0}; coarse valid={frac:.1%}",flush=True)
        region=Window(x0,y0,ROI24,ROI24)
        rgb=src.read([1,2,3],window=region)
        v=valid24(rgb)
        a_raw=old.read(1,window=region)
        prior=np.where(v&(a_raw<=10000),a_raw.astype(np.float32)/10000.,np.nan)
        maps={"A_existing":prior}
        for halo in (128,256):
            predicted,weight=feather24(src,x0,y0,halo)
            maps[f"feather_halo{halo}"]=np.where(v,predicted,np.nan)
        for name,p in maps.items():
            finite=v&np.isfinite(p)
            stats=dict(site=site,strategy=name,x=x0,y=y0,n_valid=int(finite.sum()),
                       mean_probability=float(np.mean(p[finite])),
                       fraction_ge05=float(np.mean(p[finite]>=.5)))
            for period in (256,512):
                for phase in ([0,period//2]):
                    edge,interior,ratio=seam24(np.nan_to_num(p),finite,period,phase)
                    stats[f"seam_{period}_phase{phase}_jump"]=edge
                    stats[f"seam_{period}_phase{phase}_ratio"]=ratio
            rows24.append(stats)
        fig,axs=plt.subplots(1,4,figsize=(18,4),constrained_layout=True)
        axs[0].imshow(np.moveaxis(rgb,0,-1));axs[0].set_title(f"{site}: native RGB")
        for j,key in enumerate(maps):
            axs[j+1].imshow(maps[key],vmin=0,vmax=1,cmap="viridis")
            axs[j+1].set_title(key)
        for ax in axs:ax.axis("off")
        fig.savefig(siteout/"24_overlap_feather_comparison.png",dpi=140)
        plt.show();plt.close(fig)
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="float32",nodata=-9999.,
            transform=src.window_transform(region),tiled=True,
            blockxsize=512,blockysize=512,compress="deflate")
        for key in ("feather_halo128","feather_halo256"):
            with rasterio.open(siteout/f"24_{site}_{key}.tif","w",**profile) as dst:
                dst.write(np.where(v,maps[key],-9999.).astype(np.float32),1)
pd.DataFrame(rows24).to_csv(OUT24/"24_feather_blended_full_grid_phase_seams.csv",index=False)
display(pd.DataFrame(rows24).round(4))
print("Saved:",OUT24)


## 25 — Five-forest M1 feathered production (resumable; GPU; canonical maps OUTSIDE V2)

Produce the **same final four-site M1 model** on MOS_1_2, Yoko, Baego and Luki using the experimentally validated inference method: 512px central output, 128px context halo, 256px stride, raised-cosine feather blending, full native-resolution imagery and correct RGB validity rules. **Yangambi is intentionally excluded** (2,680 unreadable tiles in its earlier M1 output; rerun deferred).

The 1024px-high streaming stripes avoid holding whole mosaics in RAM; each successfully written stripe is resume-logged. All existing historical KVO_04 probability maps and temporary V2 hard-edge maps are left untouched.

**Canonical products**: `KVO_work/KVO_04_final_multisite/probability_maps_M1_feathered_v2/<site>_M1_feathered_probability_uint16_NATIVE.tif`. This intentionally keeps *production* maps outside V2 experiments. Each map has an exact-run manifest, original native CRS, 0–10000 uint16 probabilities and 65535 NoData. Luki retains its **own native CRS**, not reprojected here; Luki's CRS transformation belongs in the later satellite-target preparation.

**Run Section 25 only** in a GPU runtime. The code is fresh-runtime standalone and uses the existing checkpoint. Start with `TEST_STRIPES25=1` for one stripe per site, inspect and then change to `None` to finish. Incomplete products are named `.partial.tif` and never used downstream. Do **not** run another concurrent inference writing the same site. Production inference involves several billion pixels; with 50% overlap it will consume substantially more compute than the previous hard-edge maps.

After completion run Section 26 to compare the *complete* old and corrected maps. Only after successful comparison should temporary V2 probabilities be cleaned up.

In [ ]:
# Section 25 — native five-site M1 feathered production; standalone, GPU + Drive.
# All output paths are NEW, outside V2. Resume-safe at completed 1024px row stripes.
from pathlib import Path
import os,json,hashlib,math,time
import numpy as np,rasterio,torch
from rasterio.windows import Window
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
assert torch.cuda.is_available(),"Choose a Colab GPU runtime for Section 25."
ROOT25=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO25=ROOT25/"data/Geo_Ref"
RUN25=ROOT25/"KVO_work/V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f"
CKPT25=RUN25/"M1_native_photometric/best_model.pt"
OUT25=ROOT25/"KVO_work/KVO_04_final_multisite/probability_maps_M1_feathered_v2"
OUT25.mkdir(parents=True,exist_ok=True)
SOURCES25={
 "MOS_1_2":GEO25/"MOS_1_2_kvo.tif",
 "Yoko":GEO25/"Yoko_kvo_georef.tif",
 "Yangambi":GEO25/"Yangambi_georef.tif",
 "Baego":GEO25/"Baego_georef.tif",
 "Luki":GEO25/"Luki.tif"}
SITES25=["MOS_1_2","Yoko","Baego","Luki"]  # Yangambi quarantined: 2680/14726 unreadable TIFF blocks
TEST_STRIPES25=None        # full inference if manually enabled; existing four complete products are reused
STRIPE25=1024              # native rows; bounded RAM for large mosaics
OUTPUT25=512
STRIDE25=256
HALO25=128
BATCH25=2                  # 768px input batches; adjust to 1 for smaller GPU
NODATA25=65535;SCALE25=10000
assert STRIPE25%STRIDE25==0 and OUTPUT25%STRIDE25==0
assert CKPT25.is_file(),f"Missing final M1 checkpoint: {CKPT25}"
assert set(SITES25).issubset(SOURCES25)
checkpoint25=torch.load(CKPT25,map_location="cpu",weights_only=False)
assert checkpoint25.get("run_id")=="337073e7b74f"
assert checkpoint25.get("variant")=="M1_native_photometric"
model25=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
model25.load_state_dict(checkpoint25["model"],strict=True)
model25=model25.cuda().eval()
mu25=torch.tensor([.485,.456,.406],device="cuda").view(1,3,1,1)
sd25=torch.tensor([.229,.224,.225],device="cuda").view(1,3,1,1)
PINPUT25=OUTPUT25+2*HALO25
d25=np.minimum(np.arange(OUTPUT25),OUTPUT25-1-np.arange(OUTPUT25)).astype(np.float32)
t25=np.minimum(1.,d25/(OUTPUT25/4))
taper25=.025+.975*.5*(1-np.cos(np.pi*t25))
WEIGHT25=(taper25[:,None]*taper25[None,:]).astype(np.float32)

def valid_rgb25(arr):
    # bands x rows x cols; do not consult 4th band/alpha.
    return ~(np.all(arr==0,axis=0)|np.all(arr==255,axis=0)|
             ((arr[0]==0)&(arr[1]==0)&(arr[2]==255)))

def run_site25(site):
    srcpath=SOURCES25[site]
    assert srcpath.is_file(),f"Missing UAV source RGB: {srcpath}"
    final=OUT25/f"{site}_M1_feathered_probability_uint16_NATIVE.tif"
    partial=OUT25/f"{site}_M1_feathered_probability_uint16_NATIVE.partial.tif"
    progress=OUT25/f"{site}_M1_feathered_progress.json"
    manifest=OUT25/f"{site}_M1_feathered_manifest.json"
    started=time.monotonic()
    with rasterio.open(srcpath) as src:
        assert src.count>=3 and src.dtypes[0]=="uint8"
        width,height=src.width,src.height
        nstripes=math.ceil(height/STRIPE25)
        config=dict(site=site,rgb=str(srcpath),rgb_bytes=srcpath.stat().st_size,
            rgb_mtime_ns=srcpath.stat().st_mtime_ns,
            checkpoint=str(CKPT25),checkpoint_bytes=CKPT25.stat().st_size,
            checkpoint_mtime_ns=CKPT25.stat().st_mtime_ns,
            width=width,height=height,crs=str(src.crs),
            transform=list(src.transform),output=OUTPUT25,stride=STRIDE25,
            halo=HALO25,stripe=STRIPE25,batch=BATCH25,
            normalizer="ImageNet",weight="raised_cosine_floor_0p025",
            invalid_RGB="black-white-pure-blue",nodata=NODATA25,
            scale=SCALE25,model="M1_four_site_337073e7b74f")
        fingerprint=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:20]
        print(f"\n{site}: native {width} x {height}; {nstripes} stripes; {final}",flush=True)
        if final.exists():
            assert manifest.exists(),f"Finished map exists without manifest: {final}"
            meta=json.loads(manifest.read_text())
            assert meta.get("fingerprint")==fingerprint, \
                f"Finished {site} map uses another inference contract; do not overwrite."
            with rasterio.open(final) as dst:
                assert dst.width==width and dst.height==height
                assert dst.crs==src.crs and dst.transform==src.transform
            print(f"{site}: existing completed product (structural checks only); SKIP",flush=True)
            return final
        if partial.exists():
            assert progress.exists(),f"Partial {site} raster lacks resume record"
            state=json.loads(progress.read_text())
            assert state.get("fingerprint")==fingerprint
            first=int(state["completed_stripes"])
            assert 0<=first<=nstripes
            mode="r+"
        else:
            assert not progress.exists(),f"Orphan progress file for {site}"
            first=0
            mode="w"
        finish=nstripes if TEST_STRIPES25 is None else min(nstripes,first+int(TEST_STRIPES25))
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="uint16",nodata=NODATA25,
            compress="deflate",predictor=2,tiled=True,blockxsize=512,
            blockysize=512,bigtiff="YES",interleave="band")
        with rasterio.open(partial,mode,**(profile if mode=="w" else {})) as dst:
            for stripe_index in tqdm(range(first,finish),desc=f"{site} feathered stripes",unit="stripe"):
                y0=stripe_index*STRIPE25
                y1=min(height,y0+STRIPE25)
                h=y1-y0
                # RAM O(native width * stripe height), never O(whole mosaic).
                accum=np.zeros((h,width),dtype=np.float32)
                weights=np.zeros((h,width),dtype=np.float32)
                tasks=[]
                for py in range((y0-OUTPUT25+STRIDE25)//STRIDE25*STRIDE25,y1,STRIDE25):
                    for px in range(-OUTPUT25+STRIDE25,width,STRIDE25):
                        xa=max(0,px);xb=min(width,px+OUTPUT25)
                        ya=max(y0,py);yb=min(y1,py+OUTPUT25)
                        if xa<xb and ya<yb:tasks.append((px,py,xa,xb,ya,yb))
                for offset in range(0,len(tasks),BATCH25):
                    batch=tasks[offset:offset+BATCH25]
                    inputs=np.stack([src.read([1,2,3],window=Window(
                        px-HALO25,py-HALO25,PINPUT25,PINPUT25),
                        boundless=True,fill_value=0) for px,py,*_ in batch])
                    tens=torch.from_numpy(inputs).float().cuda()/255.
                    tens=(tens-mu25)/sd25
                    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
                        prob=torch.sigmoid(model25(tens)).float().cpu().numpy()[:,0]
                    for j,(px,py,xa,xb,ya,yb) in enumerate(batch):
                        sy=slice(ya-py,yb-py);sx=slice(xa-px,xb-px)
                        dy=slice(ya-y0,yb-y0);dx=slice(xa,xb)
                        arr=prob[j,HALO25+sy.start:HALO25+sy.stop,
                                   HALO25+sx.start:HALO25+sx.stop]
                        wgt=WEIGHT25[sy,sx]
                        accum[dy,dx]+=arr*wgt
                        weights[dy,dx]+=wgt
                    del inputs,tens,prob
                assert np.all(weights>0),f"{site} uncovered pixels in stripe {stripe_index}"
                # Source RGB validity uses the EXACT same 3-channel rule as old products.
                rgb=src.read([1,2,3],window=Window(0,y0,width,h))
                valid=valid_rgb25(rgb)
                prob=accum/weights
                # Row-block writes limit intermediate uint16 allocations.
                for ry in range(0,h,256):
                    rh=min(256,h-ry)
                    values=np.where(valid[ry:ry+rh],
                        np.rint(np.clip(prob[ry:ry+rh],0,1)*SCALE25),
                        NODATA25).astype(np.uint16)
                    dst.write(values,1,window=Window(0,y0+ry,width,rh))
                # Write resume marker only after whole stripe was persisted.
                progress.write_text(json.dumps(dict(fingerprint=fingerprint,
                    completed_stripes=stripe_index+1,total_stripes=nstripes,site=site),indent=2))
                if stripe_index==first or (stripe_index+1)%8==0:
                    print(f"{site} stripe {stripe_index+1}/{nstripes}; "
                          f"{(time.monotonic()-started)/60:.1f} min",flush=True)
        if finish<nstripes:
            print(f"{site}: partial {finish}/{nstripes}; rerun Section 25 to resume.",flush=True)
            return partial
        with rasterio.open(partial) as dst:
            assert dst.crs==src.crs and dst.transform==src.transform
            assert dst.width==width and dst.height==height
            assert dst.count==1 and dst.nodata==NODATA25
        # Validate every compressed TIFF block before finalizing. If a block is unreadable,
        # preserve the .partial.tif for inspection; do not mark it complete.
        with rasterio.open(partial) as verify:
            for _,block_window in tqdm(verify.block_windows(1),
                desc=f"{site} verify compressed TIFF blocks",leave=False):
                verify.read(1,window=block_window)
        os.replace(partial,final)
        manifest.write_text(json.dumps(dict(fingerprint=fingerprint,config=config,
            final=str(final),complete=True,elapsed_seconds=time.monotonic()-started),indent=2))
        progress.unlink(missing_ok=True)
        print(f"{site} COMPLETE: {final}",flush=True)
    return final

OUTPUTS25={}
for site in SITES25:
    OUTPUTS25[site]=run_site25(site)
print("Section 25 site outputs:",OUTPUTS25,flush=True)


## 26 — Four healthy sites **full probability map** evaluation: historical KVO_04 versus feathered M1 (CPU)

Uses the entire **completed** native-resolution maps for MOS_1_2, Yoko, Baego and Luki (Yangambi M1 excluded: 2,680 unreadable blocks), not crown-centred predictions. Counts jointly valid RGB pixels, mean probability, area fractions P≥0.5 and P≥0.9, RMSE, 50-bin distributions and window-level luminance correlation. Evaluates residual native 512px seams on paired source windows and saves matched RGB/old/corrected panels. Includes all five forests including Luki, with each comparison in its original **native CRS**. **No resampling is done here.**

Only completed products with matching inference manifests are eligible; `.partial.tif` files will not be used. Results live in `KVO_work/KVO_04_final_multisite/probability_maps_M1_feathered_v2/comparison_to_historical/`. Section 26 is standalone, read-only and CPU-only. Set `COMPARE_SITES26=["Yoko","Luki"]` for an early comparison when those maps are completed.

In [ ]:
# Section 26: exact full-mosaic census, KVO_04 historical versus corrected final M1.
# Self-contained; no GPU and no retraining. Requires completed feathered maps.
from pathlib import Path
import math,json
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT26=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
SRC26=ROOT26/"data/Geo_Ref"
HIST26=ROOT26/"KVO_work/KVO_04_final_multisite/probability_maps"
NEW26=ROOT26/"KVO_work/KVO_04_final_multisite/probability_maps_M1_feathered_v2"
OUT26=NEW26/"comparison_to_historical";OUT26.mkdir(parents=True,exist_ok=True)
IMAGES26={
    "MOS_1_2":"MOS_1_2_kvo.tif","Yoko":"Yoko_kvo_georef.tif",
    "Baego":"Baego_georef.tif","Luki":"Luki.tif"}
COMPARE_SITES26=list(IMAGES26)
WINDOW26=512
SEAM_PAIRS26=160
rng26=np.random.default_rng(20261014)
def valid_rgb26(a):
    return ~(np.all(a==0,axis=0)|np.all(a==255,axis=0)|
             ((a[0]==0)&(a[1]==0)&(a[2]==255)))
def valid_prob26(a):return (a<=10000)&(a!=65535)
rows26=[];seam_rows26=[]
for site in COMPARE_SITES26:
    rgb_path=SRC26/IMAGES26[site]
    old_path=HIST26/f"{site}_musanga_probability_uint16_FULL.tif"
    new_path=NEW26/f"{site}_M1_feathered_probability_uint16_NATIVE.tif"
    manifest=NEW26/f"{site}_M1_feathered_manifest.json"
    if not new_path.exists() or not manifest.exists():
        print(f"{site}: corrected map incomplete or missing; skip.",flush=True)
        continue
    assert json.loads(manifest.read_text()).get("complete") is True
    assert rgb_path.is_file() and old_path.is_file(), \
        f"{site}: missing RGB or original KVO_04 output: {rgb_path}, {old_path}"
    with rasterio.open(rgb_path) as rgb,rasterio.open(old_path) as old,rasterio.open(new_path) as new:
        identity=lambda z:(z.width,z.height,z.transform,z.crs)
        assert identity(rgb)==identity(old)==identity(new), \
            f"{site}: native rasters differ in resolution, alignment or CRS; do not silently resample"
        assert old.count==new.count==1
        ncols=math.ceil(rgb.width/WINDOW26);nrows=math.ceil(rgb.height/WINDOW26)
        count=0;s_old=0.;s_new=0.;sq=0.;hi_old=0;hi_new=0;strong_old=0;strong_new=0
        valid_rgb_count=0;only_old=0;only_new=0
        hist_old=np.zeros(50,np.int64);hist_new=np.zeros(50,np.int64)
        brightness=[];window_old=[];window_new=[]
        for row in tqdm(range(nrows),desc=f"{site}: full-map census"):
            for col in range(ncols):
                x=col*WINDOW26;y=row*WINDOW26
                w=min(WINDOW26,rgb.width-x);h=min(WINDOW26,rgb.height-y)
                win=Window(x,y,w,h)
                colors=rgb.read([1,2,3],window=win)
                a=old.read(1,window=win);b=new.read(1,window=win)
                v=valid_rgb26(colors)
                aa=v&valid_prob26(a);bb=v&valid_prob26(b)
                both=aa&bb
                valid_rgb_count+=int(v.sum())
                only_old+=int((aa&~bb).sum());only_new+=int((bb&~aa).sum())
                if both.sum()<32:continue
                ap=a[both].astype(np.float32)/10000.
                bp=b[both].astype(np.float32)/10000.
                count+=len(ap)
                s_old+=float(ap.sum(dtype=np.float64))
                s_new+=float(bp.sum(dtype=np.float64))
                sq+=float(np.square(bp-ap).sum(dtype=np.float64))
                hi_old+=int((ap>=.5).sum());hi_new+=int((bp>=.5).sum())
                strong_old+=int((ap>=.9).sum());strong_new+=int((bp>=.9).sum())
                hist_old+=np.histogram(ap,bins=50,range=(0,1))[0]
                hist_new+=np.histogram(bp,bins=50,range=(0,1))[0]
                lum=(.2126*colors[0].astype(np.float32)+.7152*colors[1].astype(np.float32)+
                     .0722*colors[2].astype(np.float32))/255.
                brightness.append(float(np.median(lum[both])))
                window_old.append(float(ap.mean()))
                window_new.append(float(bp.mean()))
        assert count>0,f"{site}: no jointly valid pixels"
        rows26.append(dict(site=site,RGB_valid=valid_rgb_count,compared_pixels=count,
            original_only_valid=only_old,M1_only_valid=only_new,
            old_mean=s_old/count,feather_M1_mean=s_new/count,mean_shift=(s_new-s_old)/count,
            RMSE=math.sqrt(sq/count),old_frac_ge05=hi_old/count,
            feather_M1_frac_ge05=hi_new/count,change_ge05=(hi_new-hi_old)/count,
            old_frac_ge09=strong_old/count,feather_M1_frac_ge09=strong_new/count,
            old_luminance_rho=float(spearmanr(brightness,window_old).statistic),
            feather_M1_luminance_rho=float(spearmanr(brightness,window_new).statistic),
            windows=len(brightness),native_CRS=str(rgb.crs)))
        siteout=OUT26/site;siteout.mkdir(exist_ok=True)
        pd.DataFrame({"bin_mid":(np.arange(50)+.5)/50,
            "historical":hist_old/count,"M1_feathered":hist_new/count}).to_csv(
            siteout/"26_probability_distribution.csv",index=False)
        # Boundary jumps at old 512-grid edges vs tile-interior control, matched RGB.
        edges_x=np.arange(512,rgb.width-512,512)
        edges_y=np.arange(512,rgb.height-512,512)
        for orientation,edges,across in (("vertical",edges_x,rgb.height),
                                         ("horizontal",edges_y,rgb.width)):
            if not len(edges) or across<=512:continue
            for k in range(SEAM_PAIRS26):
                pos=int(rng26.choice(edges))
                start=int(rng26.integers(0,across-512))
                def jump(position):
                    win=(Window(position-1,start,2,512) if orientation=="vertical"
                         else Window(start,position-1,512,2))
                    colors=rgb.read([1,2,3],window=win)
                    p0=old.read(1,window=win);p1=new.read(1,window=win)
                    vm=valid_rgb26(colors)&valid_prob26(p0)&valid_prob26(p1)
                    if orientation=="vertical":
                        pair=vm[:,0]&vm[:,1]
                        d0=np.abs(p0[:,0].astype(np.float32)-p0[:,1].astype(np.float32))/10000.
                        d1=np.abs(p1[:,0].astype(np.float32)-p1[:,1].astype(np.float32))/10000.
                    else:
                        pair=vm[0,:]&vm[1,:]
                        d0=np.abs(p0[0].astype(np.float32)-p0[1].astype(np.float32))/10000.
                        d1=np.abs(p1[0].astype(np.float32)-p1[1].astype(np.float32))/10000.
                    if pair.sum()<32:return None
                    return float(d0[pair].mean()),float(d1[pair].mean())
                e=jump(pos);inside=jump(pos+256)
                if e is not None and inside is not None:
                    seam_rows26.append(dict(site=site,orientation=orientation,
                        historical_boundary=e[0],feather_boundary=e[1],
                        historical_interior=inside[0],feather_interior=inside[1]))
        # Georeferenced previews for visual comparison at native pixel scale.
        for i,(xf,yf) in enumerate([(.3,.5),(.5,.5),(.7,.5)],1):
            side=1024
            x=max(0,min(rgb.width-side,int(xf*rgb.width-side/2)))
            y=max(0,min(rgb.height-side,int(yf*rgb.height-side/2)))
            win=Window(x,y,side,side)
            c=rgb.read([1,2,3],window=win)
            a=old.read(1,window=win);b=new.read(1,window=win)
            v=valid_rgb26(c)&valid_prob26(a)&valid_prob26(b)
            ap=np.where(v,a.astype(np.float32)/10000.,np.nan)
            bp=np.where(v,b.astype(np.float32)/10000.,np.nan)
            fig,axs=plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
            axs[0].imshow(np.moveaxis(c,0,-1));axs[0].set_title(f"{site}: native RGB")
            axs[1].imshow(ap,vmin=0,vmax=1,cmap="viridis");axs[1].set_title("KVO_04 historical")
            axs[2].imshow(bp,vmin=0,vmax=1,cmap="viridis");axs[2].set_title("M1 feathered")
            axs[3].imshow(bp-ap,vmin=-1,vmax=1,cmap="coolwarm");axs[3].set_title("M1 minus historical")
            for ax in axs:ax.axis("off")
            fig.savefig(siteout/f"26_RGB_old_feather_M1_{i}.png",dpi=120)
            plt.show();plt.close(fig)
        pd.DataFrame(rows26).to_csv(OUT26/"26_five_site_full_map_comparison.csv",index=False)
        pd.DataFrame(seam_rows26).to_csv(OUT26/"26_512px_seam_samples.csv",index=False)
if not rows26:raise RuntimeError("No completed healthy-site M1 maps for comparison.")
summary26=pd.DataFrame(rows26)
display(summary26.round(4))
seam_summary26=[]
for (site,ori),g in pd.DataFrame(seam_rows26).groupby(["site","orientation"]):
    for label in ("historical","feather"):
        edge=float(g[f"{label}_boundary"].mean())
        center=float(g[f"{label}_interior"].mean())
        seam_summary26.append(dict(site=site,orientation=ori,model=label,n=len(g),
             edge_jump=edge,interior_jump=center,
             edge_ratio=edge/center if center else np.nan))
seam_df26=pd.DataFrame(seam_summary26)
seam_df26.to_csv(OUT26/"26_512px_seam_summary.csv",index=False)
display(seam_df26.round(4))
print("FULL MAP EVALUATION:",OUT26)


## 27 — Reuse KVO_07 / KVO_08 for satellite inference: inspect and prepare NEW 10 m UAV targets (CPU)

**Luki registration:** New M1 source coordinates receive the original KVO_08 Block 03L 3-GCP affine correction before aggregation. Run Section 27 again once; it skips three completed sites and replaces only the new Luki target, not the historical source.

**Yangambi is excluded from updated M1 UAV target creation because its feathered TIFF is corrupted. Keep its historical KVO_08 target unchanged for separately labelled exploratory comparisons. Do not silently mix historical Yangambi into a uniform M1 training set.**

**Do not download or recompute satellite predictor features.** KVO_07 already provides GEE upload/asset products and KVO_08 already includes `uav_targets_10m`, `satellite_features`, `raw_s2_benchmark`, `multiscale`, and model diagnostics. We reuse these established assets, predictor variables and spatial evaluation splits; only the **UAV response/label layer** changes to the corrected feathered M1 output. Previously fitted satellite models should **not** be presented as newly validated against changed targets; we need to rerun the established fitting and evaluation procedures with the new target.

This section **discovers** KVO_08 existing target rasters as templates (no assumptions about their precise filename). For each site with a validated, completed feathered map and an **unambiguous site-matching 10 m GeoTIFF template**, it calculates the area-averaged UAV probability at the **exact existing satellite target grid**, respecting its CRS, affine transform and NoData. Crucially **Luki is reprojected on the fly** from its separate native UAV projection to its existing 10 m target grid using `WarpedVRT` + area-average resampling. Neither the native Luki RGB nor the corrected native UAV probability map is reprojected or overwritten. New labels are written to a separate `KVO_08_satellite_fraction/uav_targets_10m_feathered_M1_v2` folder.

If existing target templates are stored only as GEE assets or tabular data (and no GeoTIFF template exists), the section prints what it finds and skips that site rather than silently inventing a grid. In that case use the original KVO_08 target grid definition / EE export logic when proceeding. All native corrected maps must be completed first.

**Run Section 27 after Sections 25–26.** It is standalone and CPU-only. We will subsequently re-run KVO_08's existing modelling blocks using the new 10 m label rasters and its cached predictors; do not rebuild KVO_07/08 inputs.

In [ ]:
# Section 27 — preserve established KVO_07/08 features, regrid only UAV labels.
# CPU-only, existing 10m raster grids, no inference and no satellite redownloads.
from pathlib import Path
import json,os,numpy as np,rasterio
from affine import Affine
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from google.colab import drive
from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT27=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
WORK27=ROOT27/"KVO_work"
KVO0827=WORK27/"KVO_08_satellite_fraction"
KVO0727=WORK27/"KVO_07_GEE_direct_upload_optimized"
OLD_TARGETS27=KVO0827/"uav_targets_10m"
NEW_TARGETS27=KVO0827/"uav_targets_10m_feathered_M1_v2"
NEW_TARGETS27.mkdir(parents=True,exist_ok=True)
PROD27=WORK27/"KVO_04_final_multisite/probability_maps_M1_feathered_v2"
SITES27=["MOS_1_2","Yoko","Baego","Luki"]  # exclude damaged Yangambi M1; historical target stays untouched
# OPTIONAL override if original KVO_08 grids are GEE assets or have ambiguous names:
# Set this to an existing same-site 10 m georeferenced raster, NOT an unrelated site.
SAT_TEMPLATE_OVERRIDES27={
    "Luki": OLD_TARGETS27/"Luki_musanga_uav_target_S2grid_10m.tif"
}  # corrected georeferenced Luki S2 grid; never use BEFORE_GEOREF backup
DRY_RUN27=False
REBUILD_LUKI27=True  # replace only the NEW Luki target to apply historical 3-GCP source correction
assert OLD_TARGETS27.is_dir(),f"Missing established KVO_08 targets: {OLD_TARGETS27}"
def inspect_inventory27():
    for label,folder in [("KVO_07",KVO0727),("KVO_08 old target 10m",OLD_TARGETS27),
                         ("KVO_08 feature cache",KVO0827/"satellite_features"),
                         ("KVO_08 raw S2",KVO0827/"satellite_raw_s2_10m")]:
        print(f"\n{label}: {folder} exists={folder.is_dir()}")
        if folder.is_dir():
            files=list(folder.rglob("*.tif"))
            print("GeoTIFF count:",len(files))
            for p in files[:15]:print(" ",p.relative_to(folder))
            for ext in ("*.parquet","*.csv","*.npz"):
                print(ext, "count:",len(list(folder.rglob(ext))))
inspect_inventory27()
templates27=list(OLD_TARGETS27.rglob("*.tif"))+list(OLD_TARGETS27.rglob("*.tiff"))
def locate_template27(site):
    if site in SAT_TEMPLATE_OVERRIDES27:
        p=Path(SAT_TEMPLATE_OVERRIDES27[site])
        assert p.exists(),f"Template override missing: {p}"
        return p
    # Require unambiguous site token to prevent taking the wrong 10m grid.
    matching=[p for p in templates27 if site.lower() in p.name.lower()]
    if len(matching)!=1:
        print(f"{site}: template candidates={len(matching)}:",
              [str(x) for x in matching[:12]],"— SKIPPING until grid is specified")
        return None
    return matching[0]
rows27=[]
for site in SITES27:
    native=PROD27/f"{site}_M1_feathered_probability_uint16_NATIVE.tif"
    manifest=PROD27/f"{site}_M1_feathered_manifest.json"
    if not native.is_file() or not manifest.is_file():
        print(f"{site}: corrected UAV map incomplete; skip 10m target")
        continue
    assert json.loads(manifest.read_text()).get("complete") is True
    template=locate_template27(site)
    if template is None:continue
    out=NEW_TARGETS27/f"{site}_M1_feathered_fraction_10m.tif"
    if out.exists() and not (site=="Luki" and REBUILD_LUKI27):
        print(f"{site}: new 10m target already exists, not replacing: {out}")
        continue
    with rasterio.open(native) as src,rasterio.open(template) as ref:
        assert src.count==1 and src.nodata==65535 and src.dtypes[0]=="uint16"
        assert ref.crs and ref.transform and ref.width>0 and ref.height>0
        # Same geographic position is allowed with DIFFERENT CRS: essential for Luki.
        print(f"\n{site}: source CRS={src.crs}, ref 10m CRS={ref.crs}, "
              f"source pixel={src.res}, template pixel={ref.res}",flush=True)
        assert 7<=abs(ref.transform.a)<=15 and 7<=abs(ref.transform.e)<=15, \
            f"{site}: candidate template is not approximately 10m: {template}"
        if DRY_RUN27:continue
        source_transform=src.transform
        if site=="Luki":
            # KVO_08 Block 03L: original XY -> corrected XY, in EPSG:32733.
            oldxy=np.array([[289355.331250,9378298.115416],
                            [289887.047604,9377653.025741],
                            [289176.719321,9377532.373705]],dtype=np.float64)
            newxy=np.array([[289360.828531,9378303.451012],
                            [289890.928037,9377658.231988],
                            [289180.994707,9377539.349260]],dtype=np.float64)
            origin=oldxy.mean(axis=0)
            coef=np.linalg.solve(np.column_stack((oldxy-origin,np.ones(3))),newxy)
            m=coef[:2,:].T
            b=coef[2,:]-m@origin
            correction=Affine(m[0,0],m[0,1],b[0],m[1,0],m[1,1],b[1])
            assert np.linalg.det(m)>0 and np.all(np.abs(np.linalg.svd(m,compute_uv=False)-1)<.01)
            source_transform=correction*src.transform
            print("Luki: applying KVO_08 Block 03L source affine correction",flush=True)
        staging=out.with_name(out.stem+".incoming.tif")
        if staging.exists():staging.unlink()
        profile=ref.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="float32",nodata=-9999.,
            compress="deflate",tiled=True,blockxsize=256,blockysize=256,bigtiff="IF_SAFER")
        # Area averages of valid UAV probabilities onto exact existing satellite lattice.
        # WarpedVRT performs native-to-template CRS transformation (esp. Luki).
        with WarpedVRT(src,src_transform=source_transform,crs=ref.crs,transform=ref.transform,
                       width=ref.width,height=ref.height,
                       src_nodata=65535,nodata=65535,
                       resampling=Resampling.average) as vrt:
            with rasterio.open(staging,"w",**profile) as dst:
                for _,window in tqdm(list(dst.block_windows(1)),desc=f"{site} UAV to 10m"):
                    a=vrt.read(1,window=window)
                    good=(a!=65535)&(a>=0)&(a<=10000)
                    target=np.where(good,a.astype(np.float32)/10000.,-9999.).astype(np.float32)
                    dst.write(target,1,window=window)
        with rasterio.open(staging) as check:
            assert (check.width,check.height,check.transform,check.crs)==(
                ref.width,ref.height,ref.transform,ref.crs)
            valid_count=0
            for _,win in check.block_windows(1):
                a=check.read(1,window=win)
                valid_count+=int(((a>=0)&(a<=1)).sum())
            assert valid_count>0, f"{site}: empty 10m target"
        os.replace(staging,out)
        provenance=dict(site=site,source_uav=str(native),template=str(template),
            source_crs=str(src.crs),target_crs=str(ref.crs),
            source_transform_used=list(source_transform)[:6],
            luki_correction="KVO_08_Block03L_affine_3_GCP_20261012" if site=="Luki" else None,
            method="GDAL WarpedVRT average of valid 0-10000 uint16 probabilities; invalid 65535",
            output=str(out),model="M1 feathered 337073e7b74f")
        out.with_suffix(".json").write_text(json.dumps(provenance,indent=2))
        rows27.append(provenance)
        print(f"{site}: new target written at exact KVO_08 grid: {out}",flush=True)
pd.DataFrame(rows27).to_csv(NEW_TARGETS27/"27_new_target_inventory.csv",index=False)
print("\nRe-use original KVO_08 feature caches, spatial splits, and model code;",
      "refit/evaluate with these NEW targets rather than treating old fitted-model scores as updated.")
print("NEW TARGET ROOT:",NEW_TARGETS27)


## 28 — Clear misleading experimental **probability maps from V2** (safe archive, after production)

Only after **the four healthy feathered production maps are complete and verified**, archive the outdated hard-edge inference GeoTIFFs from V2 into `KVO_work/KVO_04_final_multisite/archive_V2_hardedge_probability_maps`. This **removes the four replaced probability maps from V2** without permanently deleting research artifacts, preserves old KVO_04 production maps, and retains all V2 training checkpoints, metrics, crown results and modelling history.

**Default is dry-run** (`APPLY_ARCHIVE28=False`). Inspect the proposed file moves, set it to `True` and rerun when satisfied. Each move is recorded in a manifest. Never archive an incomplete map or a file whose full production replacement has not yet been verified. This section is intentionally **not** part of Section 25 to prevent deleting input evidence after partial or failed production runs.

In [ ]:
# Section 28 — validated cleanup of obsolete HARD-EDGE V2 map rasters.
# Archive, never delete. Independent GPU-free cell.
from pathlib import Path
import json,shutil,time
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT28=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
WORK28=ROOT28/"KVO_work"
RUN28=WORK28/"V2/04_illumination_robust_training/final_four_site_M1/337073e7b74f"
PROD28=WORK28/"KVO_04_final_multisite/probability_maps_M1_feathered_v2"
ARCHIVE28=WORK28/"KVO_04_final_multisite/archive_V2_hardedge_probability_maps"
APPLY_ARCHIVE28=False       # explicit opt-in after reviewing move plan
SITES28=["MOS_1_2","Yoko","Baego","Luki"]  # Yangambi excluded: retain its historical/V2 products
verified28=[]
for site in SITES28:
    completed=PROD28/f"{site}_M1_feathered_probability_uint16_NATIVE.tif"
    contract=PROD28/f"{site}_M1_feathered_manifest.json"
    assert completed.is_file() and contract.is_file(), \
        f"Cannot archive V2 experiment yet: {site} corrected full map missing"
    assert json.loads(contract.read_text()).get("complete") is True
    import rasterio
    with rasterio.open(completed) as src:
        assert src.count==1 and src.nodata==65535
        for _,w in src.block_windows(1):
            src.read(1,window=w)  # ensure archive never removes replacement for unreadable map
    verified28.append(site)
moves28=[]
for site in SITES28:
    old=(RUN28/"luki_wall_to_wall/Luki_M1_probability_uint16_NATIVE.tif"
         if site=="Luki" else RUN28/"site_wall_to_wall"/site/
             f"{site}_M1_probability_uint16_NATIVE.tif")
    if old.is_file():
        dest=ARCHIVE28/site/old.name
        assert not dest.exists(),f"Archive collision (inspect): {dest}"
        moves28.append((old,dest))
print("PRODUCTION REPLACEMENTS VERIFIED:",verified28)
print("Obsolete hard-edge V2 probability maps to archive:",len(moves28))
for old,dest in moves28:print(" MOVE",old,"\n   ->",dest)
if APPLY_ARCHIVE28:
    ARCHIVE28.mkdir(parents=True,exist_ok=True)
    done=[]
    for source,target in moves28:
        target.parent.mkdir(parents=True,exist_ok=True)
        # All folders on same MyDrive mount; move instead of creating duplicate gigabytes.
        shutil.move(str(source),str(target))
        done.append(dict(from_path=str(source),to_path=str(target)))
    (ARCHIVE28/"28_archived_V2_probability_maps.json").write_text(
        json.dumps(dict(archived_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                        replacements_verified=verified28,moves=done),indent=2))
    print("ARCHIVED",len(done),"V2 hard-edge probability maps; no permanent deletion")
else:
    print("DRY RUN ONLY. Change APPLY_ARCHIVE28=True after inspecting.")


## 29 — CPU-only four-site M1 satellite benchmark

Run revised Section 27 once more first: it applies the original KVO_08 Luki 3-GCP affine source correction, rather than just matching the destination raster grid. Three other new targets are skipped.

Section 29 reuses existing KVO_08 satellite feature files and the four healthy 10 m M1 probability targets. It checks exact grids, reports valid data coverage, and runs leave-one-site-out (LOSO) HistGradientBoosting regression with the original KVO_08 hyperparameters. No GEE exports or UAV inference are needed.

**Scientific distinction:** Current M1 10 m targets are mean UAV probability, not the fraction of pixels with probability >= 0.5. Historical KVO_08 canopy fraction and quality masks are retained, but are not substituted for new M1 fractions. We can add threshold-derived M1 cover fractions separately. Luki is the independent external site. Different feature families may have different valid-pixel support.


In [ ]:
# Section 29 — standalone CPU, original KVO_08 4-site LOSO design
from pathlib import Path
import json
import numpy as np,pandas as pd,rasterio
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
B29=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/KVO_08_satellite_fraction")
N29=B29/"uav_targets_10m_feathered_M1_v2"
O29=B29/"uav_targets_10m"
F29=B29/"satellite_features"
R29=B29/"satellite_raw_s2_10m"
OUT29=B29/"block29_M1_probability_transfer";OUT29.mkdir(exist_ok=True,parents=True)
S29=["MOS_1_2","Yoko","Baego","Luki"]  # exclude damaged Yangambi M1
Y29={"MOS_1_2":2024,"Yoko":2020,"Baego":2023,"Luki":2023}
T29={s:N29/f"{s}_M1_feathered_fraction_10m.tif" for s in S29}
H29={s:O29/f"{s}_musanga_uav_target_S2grid_10m.tif" for s in S29}
P29=["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL","S2_T3Y","EMB_T3Y",
     "S2_AMAP_QUARTERLY","S2_AMAP_T3Y","S2_BRDF_QUARTERLY","S2_BRDF_T3Y"]
COMB29={"FUSION":["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL"],
       "FUSION_T3Y":["S2_T3Y","EMB_T3Y"],
       "FUSION_AMAP":["RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL"],
       "FUSION_AMAP_T3Y":["S2_AMAP_T3Y","EMB_T3Y"]}
MAX_TRAIN29=4000;MAX_TEST29=8000;MIN29=100;SEED29=20261014
RUN_MODELS29=True
def same29(a,b,site):
    assert (a.width,a.height,a.crs)==(b.width,b.height,b.crs) and a.transform.almost_equals(b.transform), f"Grid mismatch {site}"
def path29(s,f):
    y=Y29[s]
    return R29/f"{s}_S2_{y}_RAW_10m.tif" if f=="RAW_ANNUAL" else F29/f/f"{s}_{f}_{y}_10m.tif"
for s in S29:
    assert T29[s].exists() and H29[s].exists(),s
    if s=="Luki":
        p=T29[s].with_suffix(".json")
        assert p.exists() and json.loads(p.read_text()).get("luki_correction")=="KVO_08_Block03L_affine_3_GCP_20261012", "Rerun corrected Section 27 for Luki"
    with rasterio.open(T29[s]) as a,rasterio.open(H29[s]) as b:
        same29(a,b,s)
        assert a.count==1 and a.nodata==-9999 and b.count>=3
files29={}
for f in P29:
    mapping={s:path29(s,f) for s in S29}
    if not all(p.exists() for p in mapping.values()):
        print("Missing family:",f);continue
    for s in S29:
        with rasterio.open(T29[s]) as a,rasterio.open(mapping[s]) as b:
            same29(a,b,f"{s}/{f}")
    files29[f]=mapping
    print("Available family:",f)
assert "RAW_ANNUAL" in files29
families29={f:[f] for f in files29}
families29.update({f:p for f,p in COMB29.items() if all(z in files29 for z in p)})
locations29={};cover29=[]
for i,s in enumerate(S29):
    with rasterio.open(T29[s]) as a:target=a.read(1)
    with rasterio.open(H29[s]) as b:coverage=b.read(3)
    rr,cc=np.where(np.isfinite(target)&(target>=0)&(target<=1)&
                   np.isfinite(coverage)&(coverage>=.9-1e-5))
    assert len(rr)>=MIN29,f"{s}: too few valid M1 cells"
    order=np.lexsort((cc,rr))
    rng=np.random.default_rng(SEED29+i)
    groups=np.array_split(order,min(MAX_TEST29,len(order)))
    idx=np.array([g[rng.integers(len(g))] for g in groups])
    locations29[s]=(rr[idx],cc[idx],target[rr[idx],cc[idx]].astype(np.float32))
    cover29.append(dict(site=s,valid_cells=len(rr),sampled=len(idx),mean_M1_probability=float(target[rr[idx],cc[idx]].mean())))
display(pd.DataFrame(cover29))
pd.DataFrame(cover29).to_csv(OUT29/"29_target_coverage.csv",index=False)
sample29={s:{} for s in S29};supports29=[]
for f,mapping in files29.items():
    for s in S29:
        rr,cc,_=locations29[s]
        with rasterio.open(mapping[s]) as ds: X=ds.read().astype(np.float32)[:,rr,cc].T.copy()
        good=np.isfinite(X).all(axis=1)&(X!=-9999).all(axis=1)
        if f=="RAW_ANNUAL":
            assert X.shape[1]>14
            good &= X[:,14]>0
        if f in ("S2_QUARTERLY","S2_AMAP_QUARTERLY","S2_BRDF_QUARTERLY"):
            assert X.shape[1]>43
            good &= (X[:,[10,21,32,43]]>0).all(axis=1)
        sample29[s][f]=(X,good)
        supports29.append(dict(site=s,family=f,supported=int(good.sum()),sampled=len(good),bands=X.shape[1]))
support29=pd.DataFrame(supports29)
support29.to_csv(OUT29/"29_feature_support.csv",index=False)
display(support29.pivot(index="site",columns="family",values="supported"))
if RUN_MODELS29:
    rows=[];luki=[]
    for family,parts in families29.items():
        data={}
        for s in S29:
            rr,cc,y=locations29[s]
            good=np.logical_and.reduce([sample29[s][p][1] for p in parts])
            X=np.concatenate([sample29[s][p][0] for p in parts],axis=1)
            data[s]=(X[good],y[good],rr[good],cc[good])
        for held in S29:
            train=[s for s in S29 if s!=held]
            if len(data[held][1])<MIN29 or any(len(data[s][1])<MIN29 for s in train):
                print("Sparse support; skip",family,held);continue
            Xtr=np.concatenate([data[s][0][:MAX_TRAIN29] for s in train])
            ytr=np.concatenate([data[s][1][:MAX_TRAIN29] for s in train])
            Xte,yte,rr,cc=data[held]
            model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=180,
                max_leaf_nodes=15,min_samples_leaf=60,l2_regularization=1.,
                loss="squared_error",random_state=SEED29)
            model.fit(Xtr,ytr)
            pred=np.clip(model.predict(Xte),0,1)
            base=np.full_like(yte,ytr.mean())
            for name,estimate in [("HGB",pred),("TRAIN_MEAN",base)]:
                rows.append(dict(family=family,heldout=held,model=name,n_train=len(ytr),
                    n_test=len(yte),rmse=float(np.sqrt(mean_squared_error(yte,estimate))),
                    mae=float(mean_absolute_error(yte,estimate)),
                    bias=float(np.mean(estimate-yte)),
                    r2=float(r2_score(yte,estimate)),
                    observed_mean=float(yte.mean()),predicted_mean=float(estimate.mean())))
            if held=="Luki":
                luki.append(pd.DataFrame(dict(family=family,row=rr,col=cc,observed=yte,predicted=pred)))
            print(f"{family} -> {held}: RMSE {rows[-2]['rmse']:.4f}, baseline {rows[-1]['rmse']:.4f}",flush=True)
    scores29=pd.DataFrame(rows)
    assert len(scores29),"No model comparisons completed"
    scores29.to_csv(OUT29/"29_LOSO_M1_mean_probability_metrics.csv",index=False)
    if luki:pd.concat(luki,ignore_index=True).to_csv(OUT29/"29_Luki_LOSO_predictions.csv",index=False)
    display(scores29[scores29.model=="HGB"].pivot(index="heldout",columns="family",values="rmse").round(4))
    print("Saved:",OUT29)
print("IMPORTANT: Target is mean M1 probability, NOT binary-thresholded canopy fraction.")


## 30 — Multiscale M1 versus historical KVO_08: matched-support independent-forest benchmarks

**Run Section 30 only** after completing corrected Sections 27 and 29. It reads existing UAV targets, cached Sentinel-2 and embedding rasters, and previous KVO_08 result CSVs; no GPU, Earth Engine, downloading or UAV inference.

Primary scales: **50, 75, 100, 150 m** (add 30 m if desired). The 75 m scale uses area-average reprojection directly from the aligned 10 m grid; it is **not** an integer 10 m block reduction. Every comparison group uses **identical valid 10 m input pixels across feature families and both UAV targets**, then requires at least 90% supported coarse-cell area. LOSO holds out entire forest sites; per-training-site sample caps prevent MOS from dominating.

Two genuinely *paired* target experiments are run at identical cells and with the same algorithm and training-site list: (i) **new feathered M1 mean UAV probability** and (ii) **historical KVO_08 mean UAV probability**. A third outcome, historical **Musanga canopy fraction P>=0.5**, is kept distinct and can be enabled as a separate benchmark.

The code reports RMSE, MAE, bias, R², Spearman rho and skill over the held-out training-mean baseline. It loads previous results under `multiscale`, `block09_model_diagnostics` and `block10_advanced_models` into an explicitly labelled historical comparison report. Prior KVO_08 results typically used five sites, different reference targets or algorithms, and cannot be treated as strictly paired with four-site M1 results.

Yangambi M1 remains excluded. Luki remains an independent forest transfer test. Luki multiannual S2 has insufficient support; those combinations are skipped, without dropping annual predictors.


In [ ]:
# Section 30 — matched-support multiscale M1 vs historical KVO_08 comparison
# Standalone CPU only. Prerequisite: Section 27 with Luki affine correction.
from pathlib import Path
import json,math,time
import numpy as np,pandas as pd,rasterio
from rasterio.warp import reproject,Resampling
from rasterio.transform import Affine
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
BASE30=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/KVO_work/KVO_08_satellite_fraction")
NEW30=BASE30/"uav_targets_10m_feathered_M1_v2"
HIST30=BASE30/"uav_targets_10m"
FEAT30=BASE30/"satellite_features"
RAW30=BASE30/"satellite_raw_s2_10m"
OUT30=BASE30/"block30_multiscale_M1_vs_KVO08";OUT30.mkdir(exist_ok=True,parents=True)
SITES30=["MOS_1_2","Yoko","Baego","Luki"]
YEARS30={"MOS_1_2":2024,"Yoko":2020,"Baego":2023,"Luki":2023}
SCALES30=[50,75,100,150]   # metres; optional: add 30
COVERAGE30=.90
TRAIN_CAP30=1200
MIN_CELLS30=15
RUN_MODELS30=True
INCLUDE_OLD_COVER30=False   # extra historical cover-fraction benchmark
GROUPS30={
    "ANNUAL":{"RAW_ANNUAL":["RAW_ANNUAL"],"EMB_ANNUAL":["EMB_ANNUAL"],
              "EMB_T3Y":["EMB_T3Y"]},
    "QUARTERLY":{"S2_QUARTERLY":["S2_QUARTERLY"],
                 "S2_AMAP_QUARTERLY":["S2_AMAP_QUARTERLY"],
                 "FUSION":["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL"],
                 "FUSION_AMAP":["RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL"]}
}
# Read original KVO_08 results for contextual historical comparison.
previous30=[]
for rel in ("multiscale/08_multiscale_LOSO_metrics_by_family.csv",
            "block09_model_diagnostics/09_LOSO_all_metrics.csv",
            "block10_advanced_models/10_advanced_LOSO_all_metrics.csv"):
    p=BASE30/rel
    if p.exists():
        df=pd.read_csv(p);df["historical_file"]=rel;previous30.append(df)
        print("PREVIOUS",rel,"rows",len(df),flush=True)
    else:
        print("Previous results unavailable:",rel,flush=True)
if previous30:
    prior30=pd.concat(previous30,ignore_index=True)
    prior30.to_csv(OUT30/"30_previous_KVO08_results_combined.csv",index=False)
    cols=[c for c in ["historical_file","scale_m","site","held_out_site","family",
                       "model","target","rmse","r2","skill_rmse","n","n_test"] if c in prior30]
    old_summary30=prior30[prior30.get("scale_m",pd.Series(np.nan,index=prior30.index)).isin(SCALES30)]
    print("Historical model scores found at requested scales:",len(old_summary30))
    if len(old_summary30):display(old_summary30[cols].head(25))
def file30(s,f):
    y=YEARS30[s]
    return RAW30/f"{s}_S2_{y}_RAW_10m.tif" if f=="RAW_ANNUAL" else FEAT30/f/f"{s}_{f}_{y}_10m.tif"
def assert_aligned30(a,b,desc):
    assert (a.width,a.height,a.crs)==(b.width,b.height,b.crs) and a.transform.almost_equals(b.transform),desc
def coarse30(a,mask,src_transform,crs,scale):
    # Area-average numerator and denominator; supports 75m and arbitrary 10m offsets.
    h,w=a.shape
    t=src_transform*Affine.scale(scale/10.,scale/10.)
    shape=(math.ceil(h*10/scale),math.ceil(w*10/scale))
    def average(src):
        dst=np.zeros(shape,dtype=np.float32)
        reproject(src.astype(np.float32),dst,
            src_transform=src_transform,src_crs=crs,
            dst_transform=t,dst_crs=crs,resampling=Resampling.average,
            init_dest_nodata=False)
        return dst
    fraction=average(mask.astype(np.float32))
    numerator=average(np.where(mask,a,0).astype(np.float32))
    out=np.divide(numerator,fraction,out=np.full(shape,np.nan,dtype=np.float32),
                  where=fraction>1e-6)
    return out,fraction
# Freeze the allowed comparison groups once: all families within each group
# must exist and share identical source grids at all four sites.
active30={}
for group,models in GROUPS30.items():
    eligible={}
    for name,parts in models.items():
        if all(file30(s,p).exists() for s in SITES30 for p in parts):
            eligible[name]=parts
        else:
            print("EXCLUDE missing data:",group,name,flush=True)
    if eligible:active30[group]=eligible
assert active30,"No complete cached feature groups"
print("ACTIVE GROUPS:",active30,flush=True)
def cache_site_group30(site,group,models):
    newfile=NEW30/f"{site}_M1_feathered_fraction_10m.tif"
    oldfile=HIST30/f"{site}_musanga_uav_target_S2grid_10m.tif"
    if site=="Luki":
        provenance=newfile.with_suffix(".json")
        assert provenance.exists() and json.loads(provenance.read_text()).get("luki_correction")=="KVO_08_Block03L_affine_3_GCP_20261012"
    with rasterio.open(newfile) as m,rasterio.open(oldfile) as h:
        assert_aligned30(m,h,site+" source targets")
        new=m.read(1).astype(np.float32)
        old=h.read([1,2,3]).astype(np.float32)
        transform,crs=m.transform,m.crs
    common=(np.isfinite(new)&(new>=0)&(new<=1)&
            np.isfinite(old).all(axis=0)&(old[0]>=0)&(old[0]<=1)&
            (old[1]>=0)&(old[1]<=1)&(old[2]>=.9-1e-5))
    unique_parts=sorted({p for parts in models.values() for p in parts})
    # Pass 1: common 10m validity mask for every family in this group.
    for part in unique_parts:
        with rasterio.open(file30(site,part)) as ds:
            with rasterio.open(newfile) as ref:assert_aligned30(ds,ref,site+"/"+part)
            for b in range(1,ds.count+1):
                arr=ds.read(b)
                common &= np.isfinite(arr)&(arr!=-9999)
            if part=="RAW_ANNUAL":common &= ds.read(15)>0
            if part in ("S2_QUARTERLY","S2_AMAP_QUARTERLY"):
                for idx in [11,22,33,44]:common &= ds.read(idx)>0
    print(f"  {site}/{group}: matched valid 10m = {int(common.sum())}",flush=True)
    # Pass 2: aggregate both targets and each feature using exactly the same 10m mask.
    cache={}
    for scale in SCALES30:
        target,coverage=coarse30(new,common,transform,crs,scale)
        oldprob,_=coarse30(old[0],common,transform,crs,scale)
        oldcover,_=coarse30(old[1],common,transform,crs,scale)
        keep=(coverage>=COVERAGE30-1e-6)&np.isfinite(target)&np.isfinite(oldprob)
        cache[scale]=dict(y_new=target,y_old=oldprob,y_cover=oldcover,
                          keep=keep,coverage=coverage,features={})
    for part in unique_parts:
        with rasterio.open(file30(site,part)) as ds:
            for scale in SCALES30:
                bands=[]
                for b in range(1,ds.count+1):
                    a=ds.read(b).astype(np.float32)
                    avg,_=coarse30(a,common,transform,crs,scale)
                    bands.append(avg)
                cache[scale]["features"][part]=np.stack(bands,axis=-1)
    for scale,d in cache.items():
        for part in unique_parts:d["keep"] &= np.isfinite(d["features"][part]).all(axis=-1)
        coords=np.column_stack(np.where(d["keep"]))
        d["coords"]=coords
        d["Y"]=np.column_stack([d[k][d["keep"]] for k in ("y_new","y_old","y_cover")]).astype(np.float32)
        d["X"]={f:np.concatenate([d["features"][p][d["keep"]] for p in parts],axis=1)
                for f,parts in models.items()}
        # Discard large rasters before the next site.
        for key in ("features","keep","coverage","y_new","y_old","y_cover"):d.pop(key,None)
    return cache
ALL30={};support30=[]
for group,models in active30.items():
    for site in SITES30:
        d=cache_site_group30(site,group,models)
        for scale,v in d.items():
            ALL30[(group,site,scale)]=v
            support30.append(dict(group=group,site=site,scale_m=scale,
                     supported_cells=len(v["Y"]),families=";".join(models)))
    print("GROUP READY:",group,flush=True)
support30=pd.DataFrame(support30)
support30.to_csv(OUT30/"30_matched_support.csv",index=False)
display(support30.pivot_table(index=["site","group"],columns="scale_m",values="supported_cells"))
if RUN_MODELS30:
    scores=[];pair_differences=[]
    targets=["M1_mean_probability","historical_mean_probability"]
    if INCLUDE_OLD_COVER30:targets.append("historical_cover_fraction_p05")
    for scale in SCALES30:
        for group,models in active30.items():
            for family in models:
                for held in SITES30:
                    train_sites=[s for s in SITES30 if s!=held and len(ALL30[(group,s,scale)]["Y"])>=MIN_CELLS30]
                    test=ALL30[(group,held,scale)]
                    if len(train_sites)<3 or len(test["Y"])<MIN_CELLS30:
                        print("SKIP sparse:",scale,group,family,held,flush=True);continue
                    Xtr_parts=[];Ytr_parts=[]
                    for s in train_sites:
                        d=ALL30[(group,s,scale)]
                        nn=len(d["Y"]);cap=min(TRAIN_CAP30,nn)
                        if nn>cap:
                            rng=np.random.default_rng(20261014+scale+SITES30.index(s))
                            ix=np.sort(rng.choice(nn,cap,replace=False))
                        else:ix=np.arange(nn)
                        Xtr_parts.append(d["X"][family][ix])
                        Ytr_parts.append(d["Y"][ix])
                    Xtr=np.vstack(Xtr_parts);Ytr=np.vstack(Ytr_parts)
                    Xte=test["X"][family];Yte=test["Y"]
                    for target_name in targets:
                        ti=targets.index(target_name)
                        reg=HistGradientBoostingRegressor(learning_rate=.06,max_iter=120,
                            max_leaf_nodes=15,min_samples_leaf=35,
                            l2_regularization=1.,random_state=20261014)
                        reg.fit(Xtr,Ytr[:,ti])
                        pred=np.clip(reg.predict(Xte),0,1)
                        obs=Yte[:,ti];base=np.full_like(obs,Ytr[:,ti].mean())
                        rmse=float(np.sqrt(mean_squared_error(obs,pred)))
                        base_rmse=float(np.sqrt(mean_squared_error(obs,base)))
                        rho=float(spearmanr(obs,pred).statistic) if np.std(pred)>1e-10 and np.std(obs)>1e-10 else np.nan
                        scores.append(dict(scale_m=scale,group=group,family=family,
                            site=held,target=target_name,model="HGB",n=len(obs),n_train=len(Ytr),
                            train_sites="+".join(train_sites),rmse=rmse,baseline_rmse=base_rmse,
                            skill=1-rmse/base_rmse if base_rmse>0 else np.nan,
                            mae=float(mean_absolute_error(obs,pred)),
                            bias=float(np.mean(pred-obs)),r2=float(r2_score(obs,pred)),
                            spearman_rho=rho,observed_mean=float(obs.mean()),
                            predicted_mean=float(pred.mean())))
                    if len(scores)>=2:
                        a,b=scores[-2:]
                        if a["target"]=="M1_mean_probability" and b["target"]=="historical_mean_probability":
                            pair_differences.append(dict(scale_m=scale,group=group,family=family,
                                site=held,M1_RMSE=a["rmse"],historical_RMSE=b["rmse"],
                                M1_skill=a["skill"],historical_skill=b["skill"],
                                M1_R2=a["r2"],historical_R2=b["r2"]))
                    print(f" {scale}m {family} held {held}: {[(x['target'],round(x['rmse'],3)) for x in scores[-len(targets):]]}",flush=True)
        pd.DataFrame(scores).to_csv(OUT30/"30_paired_LOSO_metrics.csv",index=False)
        pd.DataFrame(pair_differences).to_csv(OUT30/"30_M1_vs_historical_paired.csv",index=False)
        print("COMPLETED SCALE",scale,"m",flush=True)
    result30=pd.DataFrame(scores)
    display(result30[result30.target=="M1_mean_probability"].pivot_table(
        index=["scale_m","site"],columns="family",values="rmse").round(3))
    display(pd.DataFrame(pair_differences).round(3))
    print("Saved historical vs M1 paired metrics:",OUT30)
    print("NOTE: old KVO_08 CSV scores use their original models/site sets/targets;")
    print("for a genuinely matched reference comparison use 30_M1_vs_historical_paired.csv.")


## 31 — Exhaustive Yangambi ROI: historical U-Net versus illumination-robust M1

**Independent, fully labelled circle validation (CPU only).** Inputs: `data/Geo_Ref/Annotations/Circles.shp` (select the circle containing the new exhaustive inventory) and `data/Geo_Ref/Annotations/Circle_250_Yangambi.shp` (Musanga polygons). **Every valid pixel inside the chosen circle not contained in a Musanga polygon is an explicit non-Musanga negative.** Outside the circle is ignored. No model training or inference.

Checks shapefile CRS, ROI containment and U-Net raster alignment, then uses exactly the **same valid pixels** for the historical and M1 Yangambi probability maps. Evaluates ROC-AUC, average precision, class prevalence, Brier score, discrimination, calibration, precision/recall/F1/IoU/specificity, and false-positive/false-negative **canopy area** across thresholds. Outputs PR/ROC and reliability plots and an observed-versus-predicted canopy-cover plot. The area-based confusion matrices are an exact census of the evaluated 6–7 cm grid; expensive AUC/calibration calculations use one shared reproducible subsample.

**Yangambi M1 raster warning:** Yangambi's earlier M1 GeoTIFF has ~18% unreadable compressed blocks. Section 31 never marks unreadable pixels as negatives; it logs read failures and refuses to report paired validation if more than 1% of ROI pixels are unavailable. If the circle intersects those damaged blocks, first regenerate the affected Yangambi raster instead of trusting incomplete metrics.

**Critical ROI assumption:** the polygons must be exhaustive and their non-Musanga complement must be confidently annotated within the selected circle. Occluded, uncertain or unlabeled areas violate that assumption; edit the inventory/mask before interpreting these scores. Pixel statistics describe this circle, not forest-wide generalization.


In [ ]:
# Section 31 — exhaustive Yangambi ROI truth / pairwise U-Net probability validation
# CPU, standalone; NO model training, NO satellite data, NO imagery regeneration.
from pathlib import Path
import json, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window, from_bounds
from rasterio.features import geometry_mask
from shapely.geometry import mapping
from shapely.ops import unary_union
from sklearn.metrics import roc_auc_score,average_precision_score,brier_score_loss,roc_curve,precision_recall_curve
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT31=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
ANN31=ROOT31/"data/Geo_Ref/Annotations"
CIRCLES31=ANN31/"Circles.shp"
POSITIVES31=ANN31/"Circle_250_Yangambi.shp"
OLD31=ROOT31/"KVO_work/KVO_04_final_multisite/probability_maps/Yangambi_musanga_probability_uint16_FULL.tif"
M131=ROOT31/"KVO_work/KVO_04_final_multisite/probability_maps_M1_feathered_v2/Yangambi_M1_feathered_probability_uint16_NATIVE.tif"
OUT31=ROOT31/"KVO_work/KVO_04_final_multisite/validation_exhaustive_Yangambi_250m"
OUT31.mkdir(parents=True,exist_ok=True)
THRESHOLDS31=[.10,.20,.30,.40,.50,.60,.70,.80,.90]
SAMPLE_MAX31=160000
PIXEL_SAMPLE_RATE31=.06  # unbiased shared per-pixel inclusion on the valid paired mask
ALLOW_PARTIAL31=False  # never silently accept damaged Yangambi M1 raster
for p in [CIRCLES31,POSITIVES31,OLD31,M131]:
    assert p.is_file(),f"Missing required input: {p}"
circles31=gpd.read_file(CIRCLES31)
pos31=gpd.read_file(POSITIVES31)
assert len(circles31)>0 and len(pos31)>0
assert circles31.crs is not None and pos31.crs is not None
assert circles31.geometry.notna().all() and pos31.geometry.notna().all()
assert circles31.geometry.is_valid.all() and pos31.geometry.is_valid.all(),"Invalid polygon geometry"
assert circles31.geom_type.isin(["Polygon","MultiPolygon"]).all()
assert pos31.geom_type.isin(["Polygon","MultiPolygon"]).all()
print("Annotation polygons:",len(pos31),"| candidate circles:",len(circles31),flush=True)
with rasterio.open(OLD31) as older,rasterio.open(M131) as m1:
    assert older.count==m1.count==1 and older.crs is not None
    assert older.width==m1.width and older.height==m1.height and older.crs==m1.crs
    assert older.transform.almost_equals(m1.transform),"Historical and M1 georeferencing differs"
    assert older.dtypes[0]==m1.dtypes[0]=="uint16"
    oldnodata=older.nodata if older.nodata is not None else 65535
    newnodata=m1.nodata if m1.nodata is not None else 65535
    circ=circles31.to_crs(older.crs)
    labelled=pos31.to_crs(older.crs)
    truthgeom=unary_union(list(labelled.geometry))
    assert truthgeom.area>0
    candidates=[]
    for k,geom in enumerate(circ.geometry):
        overlap=truthgeom.intersection(geom).area
        candidates.append((k,float(overlap/truthgeom.area),float(overlap),float(geom.area)))
    candidates=sorted(candidates,key=lambda x:x[1],reverse=True)
    print("Top candidate circles (row, fraction of annotated Musanga area inside, overlap m2, circle m2):")
    display(pd.DataFrame(candidates[:8],columns=["row","positive_area_containment","intersection_m2","circle_m2"]))
    picked31,contain31,_,_=candidates[0]
    assert contain31>=.98,("Could not locate a circle containing >=98% of labelled Musanga crowns. "
                            "Check files, projections, or select circle manually.")
    if len(candidates)>1 and candidates[1][1]>.95:
        raise RuntimeError("More than one circle contains almost all annotation: select ROI manually.")
    roi31=circ.geometry.iloc[picked31]
    inside31=truthgeom.intersection(roi31)
    # Do not allow wrong/larger circle to be selected if annotations are far from its edge.
    print(f"Selected Circles.shp row={picked31}; Musanga area inside={contain31:.2%}; "
          f"circle area={roi31.area:.1f} m2",flush=True)
    assert roi31.intersects(older.bounds and __import__('shapely').geometry.box(*older.bounds)),         "ROI circle does not intersect Yangambi probability raster"
    # Bound raster windows at full native grid: no loading of the enormous Yangambi map.
    win0=from_bounds(*roi31.bounds,transform=older.transform).round_offsets().round_lengths()
    r0=max(0,int(win0.row_off));c0=max(0,int(win0.col_off))
    r1=min(older.height,int(math.ceil(win0.row_off+win0.height)))
    c1=min(older.width,int(math.ceil(win0.col_off+win0.width)))
    assert r1>r0 and c1>c0
    print("Bounding native pixels:",(r1-r0)*(c1-c0),"| assessment windows:",flush=True)
    totals={"ROI_pixels":0,"paired_valid":0,"invalid_old":0,"invalid_M1":0,"unreadable_windows":0,
            "unreadable_roi_pixels":0,"positive":0}
    confusion={model:{t:np.zeros(4,dtype=np.int64) for t in THRESHOLDS31}
               for model in ("historical","M1")} # TP,FP,FN,TN
    moment={name:{"sum":0.,"sum2":0.,"brier_sum":0.} for name in ("historical","M1")}
    rng=np.random.default_rng(20261014)
    sample_labels=[];sample_old=[];sample_m1=[]
    preview=None
    for row in range(r0,r1,512):
        for col in range(c0,c1,512):
            w=Window(col,row,min(512,c1-col),min(512,r1-row))
            tr=older.window_transform(w)
            shape=(int(w.height),int(w.width))
            roi_mask=geometry_mask([mapping(roi31)],out_shape=shape,transform=tr,invert=True)
            count_roi=int(roi_mask.sum())
            if not count_roi:continue
            truth=geometry_mask([mapping(inside31)],out_shape=shape,transform=tr,invert=True)
            totals["ROI_pixels"]+=count_roi
            try:
                a=older.read(1,window=w)
                b=m1.read(1,window=w)
            except (Exception) as exc:
                totals["unreadable_windows"]+=1
                totals["unreadable_roi_pixels"]+=count_roi
                print(f"UNREADABLE source block at row={row}, col={col}: {str(exc)[:190]}",flush=True)
                continue
            va=(a!=oldnodata)&(a<=10000)
            vb=(b!=newnodata)&(b<=10000)
            totals["invalid_old"]+=int((roi_mask&~va).sum())
            totals["invalid_M1"]+=int((roi_mask&~vb).sum())
            keep=roi_mask&va&vb
            n=int(keep.sum())
            if n==0:continue
            truth=truth[keep]
            pa=a[keep].astype(np.float32)/10000.
            pb=b[keep].astype(np.float32)/10000.
            totals["paired_valid"]+=n
            totals["positive"]+=int(truth.sum())
            for name,p in (("historical",pa),("M1",pb)):
                moment[name]["sum"]+=float(p.sum(dtype=np.float64))
                moment[name]["sum2"]+=float(np.square(p).sum(dtype=np.float64))
                moment[name]["brier_sum"]+=float(np.square(p-truth).sum(dtype=np.float64))
                for th in THRESHOLDS31:
                    pred=p>=th
                    confusion[name][th]+=np.array([
                        int((pred&truth).sum()),int((pred&~truth).sum()),
                        int((~pred&truth).sum()),int((~pred&~truth).sum())],dtype=np.int64)
            take=rng.random(n)<PIXEL_SAMPLE_RATE31
            # Use identical sampled pixels for each model, without class balancing.
            if take.any():
                sample_labels.append(truth[take].astype(np.uint8))
                sample_old.append(pa[take]);sample_m1.append(pb[take])
            if totals["paired_valid"] and totals["paired_valid"]%3000000<n:
                print("Processed jointly valid pixels:",totals["paired_valid"],flush=True)
    frac31=totals["paired_valid"]/max(totals["ROI_pixels"],1)
    print("Pixel-support audit:",totals,"| paired coverage:",f"{frac31:.2%}",flush=True)
    assert totals["paired_valid"]>100 and 0<totals["positive"]<totals["paired_valid"],\
        "ROI insufficient positive/negative valid pixels"
    if frac31<.99 and not ALLOW_PARTIAL31:
        raise RuntimeError(f"Only {frac31:.2%} of ROI is jointly readable and valid. "
          "Yangambi M1 is known damaged; regenerate affected raster before fair validation.")
    pixelarea31=abs(older.transform.a*older.transform.e-older.transform.b*older.transform.d)
# Save exact census and diagnostic curves; metrics refer ONLY to same valid ROI pixels.
n31=totals["paired_valid"];positive31=totals["positive"]
pix31=pd.DataFrame([dict(**totals,paired_fraction=frac31,pixel_area_m2=pixelarea31,
                        musanga_prevalence=positive31/n31,
                        observed_cover_fraction=positive31/n31,
                        observed_Musanga_area_m2=positive31*pixelarea31)])
pix31.to_csv(OUT31/"31_coverage_and_reference.csv",index=False)
display(pix31)
if sample_labels:
    yy=np.concatenate(sample_labels)
    aa=np.concatenate(sample_old);bb=np.concatenate(sample_m1)
    if len(yy)>SAMPLE_MAX31:
        indices=np.random.default_rng(731).choice(len(yy),SAMPLE_MAX31,replace=False)
        yy=yy[indices];aa=aa[indices];bb=bb[indices]
    print("Shared evaluation subsample:",len(yy),"positive prevalence:",yy.mean(),flush=True)
else:raise RuntimeError("No probability sampling; increase PIXEL_SAMPLE_RATE31")
assert np.unique(yy).size==2,"Sample lacks one class; increase PIXEL_SAMPLE_RATE31"
rows=[];curves=[]
for name,p in [("historical",aa),("M1",bb)]:
    auc=roc_auc_score(yy,p)
    ap=average_precision_score(yy,p)
    mu=moment[name]["sum"]/n31
    brier=moment[name]["brier_sum"]/n31
    for th in THRESHOLDS31:
        tp,fp,fn,tn=confusion[name][th]
        precision=tp/(tp+fp) if tp+fp else np.nan
        recall=tp/(tp+fn) if tp+fn else np.nan
        specificity=tn/(tn+fp) if tn+fp else np.nan
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
        iou=tp/(tp+fp+fn) if tp+fp+fn else np.nan
        rows.append(dict(model=name,threshold=th,n_valid=n31,positive_pixels=positive31,
            prevalence=positive31/n31,roc_auc_sample=auc,average_precision_sample=ap,
            brier_exact=brier,mean_probability_exact=mu,precision=precision,
            recall=recall,specificity=specificity,f1=f1,iou=iou,
            predicted_cover=(tp+fp)/n31,observed_cover=positive31/n31,
            cover_bias_fraction=(fp-fn)/n31,
            predicted_Musanga_area_m2=(tp+fp)*pixelarea31,
            false_positive_area_m2=fp*pixelarea31,
            false_negative_area_m2=fn*pixelarea31,
            TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn)))
    fpr,tpr,_=roc_curve(yy,p)
    pp,rr,_=precision_recall_curve(yy,p)
    curves.append((name,fpr,tpr,rr,pp))
metrics31=pd.DataFrame(rows)
metrics31.to_csv(OUT31/"31_threshold_and_discrimination_metrics.csv",index=False)
display(metrics31[metrics31.threshold==.5][["model","roc_auc_sample","average_precision_sample",
    "brier_exact","precision","recall","specificity","f1","iou",
    "observed_cover","predicted_cover","false_positive_area_m2",
    "false_negative_area_m2","cover_bias_fraction"]].round(4))
display(metrics31.pivot(index="threshold",columns="model",
    values=["precision","recall","f1","predicted_cover"]).round(3))
fig,axes=plt.subplots(1,3,figsize=(15,4.4),constrained_layout=True)
for name,fpr,tpr,rr,pp in curves:
    axes[0].plot(fpr,tpr,label=f"{name} AUC={roc_auc_score(yy,aa if name=='historical' else bb):.3f}")
    axes[1].plot(rr,pp,label=name)
axes[0].plot([0,1],[0,1],"k--",linewidth=.8)
axes[1].axhline(float(yy.mean()),color="gray",linestyle="--",label="Prevalence")
axes[0].set(xlabel="False positive rate",ylabel="True positive rate",title="ROC")
axes[1].set(xlabel="Recall",ylabel="Precision",title="Precision–recall")
edges=np.linspace(0,1,11)
for name,p in (("historical",aa),("M1",bb)):
    ix=np.minimum(np.searchsorted(edges,p,side="right")-1,9)
    cnt=np.bincount(ix,minlength=10)
    ymean=np.bincount(ix,weights=yy,minlength=10)/np.maximum(cnt,1)
    pmean=np.bincount(ix,weights=p,minlength=10)/np.maximum(cnt,1)
    axes[2].plot(pmean[cnt>20],ymean[cnt>20],"-o",label=name)
axes[2].plot([0,1],[0,1],"k--",linewidth=.8)
axes[2].set(xlim=(0,1),ylim=(0,1),xlabel="Mean predicted probability",
            ylabel="Observed Musanga frequency",title="Probability calibration")
for ax in axes:ax.legend(fontsize=8);ax.grid(alpha=.15)
fig.savefig(OUT31/"31_ROC_PR_calibration.png",dpi=170,bbox_inches="tight")
display(fig);plt.close(fig)
# Threshold selection here is descriptive: do NOT claim a tuned threshold
# estimated on this same validation circle generalizes to new forests.
print("Saved ROI validation:",OUT31,flush=True)
print("NOTE: Pixels are spatially correlated. Treat AUC/AP as descriptive for this ROI,")
print("not as independent spatial-replicate confidence intervals.")
